In [ ]:
# ===================== GEMMA 4 E2B-IT / COLAB T4 / 1-CELL BENCH =====================
# Mục tiêu:
# - Chạy Gemma 4 bản nhẹ nhất: google/gemma-4-E2B-it
# - Tối ưu T4: ưu tiên FP16 để giữ chất lượng/tốc độ, tự fallback 4-bit NF4 nếu thiếu VRAM
# - Thử MTP assistant nếu đủ VRAM
# - Đo: thời gian nạp, VRAM, prefill, thời gian token đầu, token/s, output đầy đủ, mốc thời gian
# Ghi chú: nếu lỗi 401/403, hãy accept license model trên Hugging Face và tạo Colab Secret tên HF_TOKEN.

import os, sys, time, gc, math, json, subprocess, platform, traceback
from datetime import datetime
from threading import Thread

# --------------------- Cấu hình nhanh ---------------------
MODEL_ID = "google/gemma-4-E2B-it"
ASSISTANT_MODEL_ID = MODEL_ID + "-assistant"

INSTALL_DEPS = True
TRY_FP16_FIRST = True
USE_4BIT_FALLBACK = True
TRY_MTP_ASSISTANT = True

# Prompt thử chất lượng + code, đủ dài để đo token/s nhưng không quá lâu trên T4.
SYSTEM_PROMPT = "Bạn là trợ lý kỹ thuật, trả lời ngắn gọn, chính xác, ưu tiên tiếng Việt."
USER_PROMPT = """Viết một đoạn Python ngắn để fuzz thử 5 edge case cho torch.searchsorted.
Yêu cầu:
1) Có seed cố định.
2) In rõ input, output, exception nếu có.
3) Giải thích ngắn vì sao các edge case đó đáng thử.
"""

FAST_MAX_NEW_TOKENS = 192
QUALITY_MAX_NEW_TOKENS = 256
WARMUP_TOKENS = 24

# Sampling chính thức Gemma 4 khuyên: temperature=1.0, top_p=0.95, top_k=64.
QUALITY_GEN = dict(
    do_sample=True,
    temperature=1.0,
    top_p=0.95,
    top_k=64,
    max_new_tokens=QUALITY_MAX_NEW_TOKENS,
    use_cache=True,
)

FAST_GEN = dict(
    do_sample=False,
    max_new_tokens=FAST_MAX_NEW_TOKENS,
    use_cache=True,
)

# Tắt thinking để nhanh hơn. Đổi thành True nếu muốn test reasoning nặng hơn.
ENABLE_THINKING_FAST = False
ENABLE_THINKING_QUALITY = False

# --------------------- Tiện ích log ---------------------
def now():
    return datetime.now().strftime("%H:%M:%S %d/%m/%Y")

def log(msg=""):
    print(f"[{now()}] {msg}", flush=True)

def run(cmd, name=None, allow_fail=False):
    if name:
        log(f"▶ {name}")
    p = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    out = p.stdout.strip()
    if out:
        print(out[-6000:], flush=True)
    if p.returncode != 0 and not allow_fail:
        raise RuntimeError(f"Lệnh lỗi {p.returncode}: {cmd}\n{out}")
    return out

t_all0 = time.perf_counter()
log("BẮT ĐẦU CELL GEMMA 4 T4 BENCH")

# --------------------- Biến môi trường tối ưu ---------------------
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"

# --------------------- Cài gói ---------------------
if INSTALL_DEPS:
    log("Cài/cập nhật thư viện cần thiết...")
    run(
        f"{sys.executable} -m pip install -q -U "
        "transformers accelerate bitsandbytes safetensors sentencepiece protobuf hf_transfer psutil",
        name="pip install",
        allow_fail=False
    )

# --------------------- Import sau khi cài ---------------------
import torch
import psutil
import pandas as pd
from transformers import AutoProcessor, AutoModelForCausalLM, TextIteratorStreamer

try:
    from transformers import BitsAndBytesConfig
except Exception:
    BitsAndBytesConfig = None

torch.set_float32_matmul_precision("high")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.benchmark = True

# --------------------- Kiểm tra máy ---------------------
log("Thông tin môi trường")
print("Python:", sys.version.replace("\n", " "))
print("Platform:", platform.platform())
print("Torch:", torch.__version__)
try:
    import transformers
    print("Transformers:", transformers.__version__)
except Exception:
    pass

if not torch.cuda.is_available():
    raise RuntimeError("Không thấy CUDA. Vào Runtime → Change runtime type → GPU, nên là Tesla T4.")

device_name = torch.cuda.get_device_name(0)
prop = torch.cuda.get_device_properties(0)
print("GPU:", device_name)
print("Compute capability:", f"{prop.major}.{prop.minor}")
print("VRAM tổng:", round(prop.total_memory / 1024**3, 2), "GB")
print("RAM hệ thống:", round(psutil.virtual_memory().total / 1024**3, 2), "GB")
run("nvidia-smi", name="nvidia-smi ban đầu", allow_fail=True)

def cuda_clean():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        torch.cuda.reset_peak_memory_stats()

def vram_report():
    free_b, total_b = torch.cuda.mem_get_info()
    alloc = torch.cuda.memory_allocated()
    reserv = torch.cuda.memory_reserved()
    peak = torch.cuda.max_memory_allocated()
    return {
        "free_GB": round(free_b / 1024**3, 3),
        "total_GB": round(total_b / 1024**3, 3),
        "allocated_GB": round(alloc / 1024**3, 3),
        "reserved_GB": round(reserv / 1024**3, 3),
        "peak_allocated_GB": round(peak / 1024**3, 3),
    }

def print_vram(prefix):
    log(prefix + " | VRAM " + json.dumps(vram_report(), ensure_ascii=False))

# --------------------- HF token ---------------------
HF_TOKEN = os.environ.get("HF_TOKEN", None)
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN") or HF_TOKEN
except Exception:
    pass

if HF_TOKEN:
    log("Đã thấy HF_TOKEN.")
else:
    log("Chưa thấy HF_TOKEN. Nếu model bị khóa/gated thì cell sẽ báo 401/403.")

# --------------------- Load processor ---------------------
cuda_clean()
print_vram("Trước khi load")
log(f"Load processor: {MODEL_ID}")
t0 = time.perf_counter()
processor = AutoProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)
tok = getattr(processor, "tokenizer", processor)
log(f"Load processor xong sau {time.perf_counter() - t0:.2f}s")

# --------------------- Load model: FP16 trước, fallback 4-bit ---------------------
def load_model_fp16(model_id):
    kwargs = dict(
        token=HF_TOKEN,
        device_map={"": 0},
        low_cpu_mem_usage=True,
        torch_dtype=torch.float16,
        attn_implementation="sdpa",
    )
    try:
        return AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
    except TypeError:
        kwargs.pop("attn_implementation", None)
        return AutoModelForCausalLM.from_pretrained(model_id, **kwargs)

def load_model_4bit(model_id):
    if BitsAndBytesConfig is None:
        raise RuntimeError("Không import được BitsAndBytesConfig, không thể load 4-bit.")
    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    kwargs = dict(
        token=HF_TOKEN,
        device_map={"": 0},
        low_cpu_mem_usage=True,
        quantization_config=bnb_cfg,
        torch_dtype=torch.float16,
        attn_implementation="sdpa",
    )
    try:
        return AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
    except TypeError:
        kwargs.pop("attn_implementation", None)
        return AutoModelForCausalLM.from_pretrained(model_id, **kwargs)

load_mode = None
model = None
load_error = None

log(f"Load target model: {MODEL_ID}")
t0 = time.perf_counter()
try:
    if TRY_FP16_FIRST:
        log("Thử nạp FP16 trước: chất lượng gốc, thường nhanh hơn 4-bit trên T4 nếu đủ VRAM.")
        model = load_model_fp16(MODEL_ID)
        load_mode = "FP16"
    else:
        raise RuntimeError("Bỏ qua FP16 theo cấu hình.")
except Exception as e:
    load_error = repr(e)
    log("FP16 lỗi/thiếu VRAM. Dọn bộ nhớ và thử 4-bit NF4...")
    print(str(e)[:1200])
    cuda_clean()
    if USE_4BIT_FALLBACK:
        model = load_model_4bit(MODEL_ID)
        load_mode = "4-bit NF4"
    else:
        raise

model.eval()
model.config.use_cache = True
target_load_sec = time.perf_counter() - t0
log(f"Load target xong | mode={load_mode} | {target_load_sec:.2f}s")
print_vram("Sau khi load target")

# --------------------- Thử load MTP assistant ---------------------
assistant_model = None
assistant_mode = "không dùng"

if TRY_MTP_ASSISTANT:
    log(f"Thử load MTP assistant: {ASSISTANT_MODEL_ID}")
    t0 = time.perf_counter()
    try:
        free_gb = torch.cuda.mem_get_info()[0] / 1024**3
        # Nếu VRAM còn ít thì assistant dùng 4-bit cho đỡ vỡ bình hoa Colab.
        if load_mode == "4-bit NF4" or free_gb < 2.8:
            assistant_model = load_model_4bit(ASSISTANT_MODEL_ID)
            assistant_mode = "MTP assistant 4-bit NF4"
        else:
            try:
                assistant_model = load_model_fp16(ASSISTANT_MODEL_ID)
                assistant_mode = "MTP assistant FP16"
            except Exception:
                cuda_clean()
                assistant_model = load_model_4bit(ASSISTANT_MODEL_ID)
                assistant_mode = "MTP assistant 4-bit NF4"
        assistant_model.eval()
        assistant_model.config.use_cache = True
        assistant_model.generation_config.num_assistant_tokens = 4
        assistant_model.generation_config.num_assistant_tokens_schedule = "heuristic"
        log(f"Load assistant xong | {assistant_mode} | {time.perf_counter() - t0:.2f}s")
    except Exception as e:
        log("Không load được MTP assistant, tiếp tục chạy không MTP.")
        print(str(e)[:1500])
        assistant_model = None
        assistant_mode = "không dùng do lỗi/thiếu VRAM"
    print_vram("Sau khi thử load assistant")

# --------------------- Chuẩn hóa prompt ---------------------
def make_inputs(user_prompt, enable_thinking=False):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]
    try:
        text = processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=enable_thinking,
        )
    except TypeError:
        text = processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    inputs = processor(text=text, return_tensors="pt")
    inputs = inputs.to("cuda:0")
    return inputs, text

def decode_response(output_ids, input_len):
    raw = processor.decode(output_ids[0][input_len:], skip_special_tokens=False)
    try:
        parsed = processor.parse_response(raw)
        return str(parsed)
    except Exception:
        return processor.decode(output_ids[0][input_len:], skip_special_tokens=True)

# --------------------- Prefill benchmark ---------------------
@torch.inference_mode()
def prefill_bench(inputs, name):
    cuda_clean()
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    out = model(**inputs, use_cache=True)
    torch.cuda.synchronize()
    dt = time.perf_counter() - t0
    prompt_tokens = int(inputs["input_ids"].shape[-1])
    log(f"Prefill {name}: {prompt_tokens} token prompt / {dt:.4f}s = {prompt_tokens / max(dt,1e-9):.2f} token/s")
    del out
    return dt, prompt_tokens, prompt_tokens / max(dt, 1e-9)

# --------------------- Generate benchmark có stream output ---------------------
@torch.inference_mode()
def generate_bench(name, inputs, gen_cfg, use_assistant=False):
    input_len = int(inputs["input_ids"].shape[-1])
    streamer = TextIteratorStreamer(tok, skip_prompt=True, skip_special_tokens=True)

    kwargs = dict(**inputs, **gen_cfg, streamer=streamer)
    if use_assistant and assistant_model is not None:
        kwargs["assistant_model"] = assistant_model

    result = {}
    err = {}

    def worker():
        try:
            with torch.inference_mode():
                result["ids"] = model.generate(**kwargs)
        except Exception as e:
            err["e"] = e
            err["trace"] = traceback.format_exc()

    cuda_clean()
    log("=" * 90)
    log(f"RUN: {name}")
    print_vram("Trước generate")
    print("\n----- OUTPUT BẮT ĐẦU -----", flush=True)

    torch.cuda.synchronize()
    t0 = time.perf_counter()
    th = Thread(target=worker)
    th.start()

    chunks = []
    first_chunk_t = None
    for chunk in streamer:
        if first_chunk_t is None:
            first_chunk_t = time.perf_counter()
        chunks.append(chunk)
        print(chunk, end="", flush=True)

    th.join()
    torch.cuda.synchronize()
    t1 = time.perf_counter()

    print("\n----- OUTPUT KẾT THÚC -----", flush=True)

    if err:
        log("Generate lỗi:")
        print(err["trace"])
        raise err["e"]

    out_ids = result["ids"]
    new_tokens = int(out_ids[0].shape[-1] - input_len)
    total_sec = t1 - t0
    ttfc = None if first_chunk_t is None else first_chunk_t - t0
    steady_sec = None if first_chunk_t is None else max(t1 - first_chunk_t, 1e-9)
    tok_s_total = new_tokens / max(total_sec, 1e-9)
    tok_s_steady = None if steady_sec is None else max(new_tokens - 1, 1) / steady_sec

    final_text = decode_response(out_ids, input_len)

    mem = vram_report()
    log(f"Xong RUN: {name}")
    log(f"Prompt tokens: {input_len}")
    log(f"Output tokens: {new_tokens}")
    log(f"Tổng thời gian generate: {total_sec:.4f}s")
    log(f"Time-to-first-chunk/token gần đúng: {ttfc:.4f}s" if ttfc is not None else "Không đo được token đầu")
    log(f"Tốc độ tổng: {tok_s_total:.2f} token/s")
    log(f"Tốc độ sau token đầu gần đúng: {tok_s_steady:.2f} token/s" if tok_s_steady else "Không đo được steady token/s")
    print_vram("Sau generate")

    return {
        "run": name,
        "load_mode": load_mode,
        "assistant_mode": assistant_mode if use_assistant else "không dùng",
        "prompt_tokens": input_len,
        "output_tokens": new_tokens,
        "total_sec": round(total_sec, 4),
        "ttfc_sec_approx": None if ttfc is None else round(ttfc, 4),
        "tok_s_total": round(tok_s_total, 3),
        "tok_s_after_first_approx": None if tok_s_steady is None else round(tok_s_steady, 3),
        "vram_peak_GB": mem["peak_allocated_GB"],
        "vram_allocated_GB": mem["allocated_GB"],
    }

# --------------------- Warmup ---------------------
log("Tạo input")
inputs_fast, prompt_text_fast = make_inputs(USER_PROMPT, enable_thinking=ENABLE_THINKING_FAST)
inputs_quality, prompt_text_quality = make_inputs(USER_PROMPT, enable_thinking=ENABLE_THINKING_QUALITY)

log("Prompt đã format:")
print(prompt_text_fast[:2000] + ("\n...<cắt hiển thị prompt>" if len(prompt_text_fast) > 2000 else ""))

prefill_rows = []
for nm, inp in [("fast_prompt", inputs_fast), ("quality_prompt", inputs_quality)]:
    dt, ptok, ps = prefill_bench(inp, nm)
    prefill_rows.append({"run": nm, "prompt_tokens": ptok, "prefill_sec": round(dt, 4), "prefill_tok_s": round(ps, 3)})

log("Warmup generate ngắn để ổn định CUDA/kernel/cache")
with torch.inference_mode():
    _ = model.generate(**inputs_fast, max_new_tokens=WARMUP_TOKENS, do_sample=False, use_cache=True)
torch.cuda.synchronize()
cuda_clean()
log("Warmup xong")

# --------------------- Chạy benchmark chính ---------------------
rows = []

# 1) Không MTP: baseline công bằng
rows.append(generate_bench(
    name="FAST_GREEDY_NO_MTP",
    inputs=inputs_fast,
    gen_cfg=FAST_GEN,
    use_assistant=False,
))

# 2) Có MTP nếu assistant load được
if assistant_model is not None:
    rows.append(generate_bench(
        name="FAST_GREEDY_WITH_MTP",
        inputs=inputs_fast,
        gen_cfg=FAST_GEN,
        use_assistant=True,
    ))

# 3) Sampling chất lượng hơn theo khuyến nghị Gemma 4
rows.append(generate_bench(
    name="QUALITY_SAMPLE_TEMP1_TOPP095_TOPK64",
    inputs=inputs_quality,
    gen_cfg=QUALITY_GEN,
    use_assistant=False,  # sampling để no-MTP cho ổn định; greedy MTP đã đo riêng ở trên
))

# --------------------- Tổng kết ---------------------
df_prefill = pd.DataFrame(prefill_rows)
df = pd.DataFrame(rows)

log("=" * 90)
log("BẢNG PREFILL")
display(df_prefill)

log("BẢNG GENERATE")
display(df)

# Lưu kết quả để tải nếu cần
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
csv_path = f"/content/gemma4_e2b_t4_bench_{stamp}.csv"
json_path = f"/content/gemma4_e2b_t4_bench_{stamp}.json"
df.to_csv(csv_path, index=False)
with open(json_path, "w", encoding="utf-8") as f:
    json.dump({
        "time": now(),
        "model_id": MODEL_ID,
        "assistant_model_id": ASSISTANT_MODEL_ID,
        "load_mode": load_mode,
        "assistant_mode": assistant_mode,
        "target_load_sec": target_load_sec,
        "prefill": prefill_rows,
        "generate": rows,
        "vram_final": vram_report(),
        "device": device_name,
        "torch": torch.__version__,
    }, f, ensure_ascii=False, indent=2)

log(f"Đã lưu CSV: {csv_path}")
log(f"Đã lưu JSON: {json_path}")
run("nvidia-smi", name="nvidia-smi cuối", allow_fail=True)

log(f"HOÀN TẤT TOÀN BỘ CELL sau {time.perf_counter() - t_all0:.2f}s")
# ===================== HẾT CELL =====================

[19:00:37 04/06/2026] BẮT ĐẦU CELL GEMMA 4 T4 BENCH
[19:00:37 04/06/2026] Cài/cập nhật thư viện cần thiết...
[19:00:37 04/06/2026] ▶ pip install
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 8.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-proto 1.38.0 requires protobuf<7.0,>=5.0, but you have protobuf 7.35.0 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.35.0 which is incompatible.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/constants.py:277: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


[19:01:27 04/06/2026] Thông tin môi trường
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
Torch: 2.11.0+cu128
Transformers: 5.10.2
GPU: Tesla T4
Compute capability: 7.5
VRAM tổng: 14.56 GB
RAM hệ thống: 12.67 GB
[19:01:27 04/06/2026] ▶ nvidia-smi ban đầu
Thu Jun  4 19:01:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tes

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/1.69k [00:00<?, ?B/s]

...

[19:04:29 04/06/2026] Load assistant xong | MTP assistant FP16 | 3.35s
[19:04:29 04/06/2026] Sau khi thử load assistant | VRAM {"free_GB": 4.785, "total_GB": 14.563, "allocated_GB": 9.655, "reserved_GB": 9.66, "peak_allocated_GB": 9.655}
[19:04:29 04/06/2026] Tạo input
[19:04:29 04/06/2026] Prompt đã format:
<bos><|turn>system
Bạn là trợ lý kỹ thuật, trả lời ngắn gọn, chính xác, ưu tiên tiếng Việt.<turn|>
<|turn>user
Viết một đoạn Python ngắn để fuzz thử 5 edge case cho torch.searchsorted.
Yêu cầu:
1) Có seed cố định.
2) In rõ input, output, exception nếu có.
3) Giải thích ngắn vì sao các edge case đó đáng thử.<turn|>
<|turn>model

[19:04:31 04/06/2026] Prefill fast_prompt: 94 token prompt / 1.1478s = 81.90 token/s
[19:04:31 04/06/2026] Prefill quality_prompt: 94 token prompt / 0.0839s = 1120.21 token/s
[19:04:31 04/06/2026] Warmup generate ngắn để ổn định CUDA/kernel/cache
[19:04:33 04/06/2026] Warmup xong
[19:04:34 04/06/2026] =========================================================

,run,prompt_tokens,prefill_sec,prefill_tok_s
0,fast_prompt,94,1.1478,81.898
1,quality_prompt,94,0.0839,1120.215


[19:05:16 04/06/2026] BẢNG GENERATE


,run,load_mode,assistant_mode,prompt_tokens,output_tokens,total_sec,ttfc_sec_approx,tok_s_total,tok_s_after_first_approx,vram_peak_GB,vram_allocated_GB
0,FAST_GREEDY_NO_MTP,FP16,không dùng,94,192,14.4024,0.0825,13.331,13.338,9.691,9.671
1,FAST_GREEDY_WITH_MTP,FP16,MTP assistant FP16,94,192,8.0126,0.0843,23.962,24.091,9.707,9.671
2,QUALITY_SAMPLE_TEMP1_TOPP095_TOPK64,FP16,không dùng,94,256,19.6095,0.1408,13.055,13.098,9.697,9.671


[19:05:16 04/06/2026] Đã lưu CSV: /content/gemma4_e2b_t4_bench_20260604_190516.csv
[19:05:16 04/06/2026] Đã lưu JSON: /content/gemma4_e2b_t4_bench_20260604_190516.json
[19:05:16 04/06/2026] ▶ nvidia-smi cuối
Thu Jun  4 19:05:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   66C    P0       

In [ ]:
# ===================== GEMMA 4 T4 - CELL TỐI ƯU TIẾP / KHÔNG RELOAD =====================
# Dán ngay dưới cell cũ khi kernel vẫn còn model, processor, assistant_model.
# Mục tiêu:
# - Không tải lại model.
# - Đo thật không streaming để tránh Python/thread làm chậm.
# - Sweep MTP num_assistant_tokens + schedule.
# - Đo TTFT gần đúng bằng max_new_tokens=1.
# - Test batch no-MTP để xem throughput tối đa của T4.
# - In output đầy đủ của cấu hình nhanh nhất + cấu hình chất lượng.

import os, gc, time, json, traceback, statistics
from datetime import datetime
import torch
import pandas as pd

# --------------------- Check biến từ cell cũ ---------------------
need = ["model", "processor"]
missing = [x for x in need if x not in globals()]
if missing:
    raise RuntimeError(f"Thiếu biến {missing}. Cell này phải chạy sau cell đã load Gemma 4, không chạy độc lập.")

tokenizer = getattr(processor, "tokenizer", processor)
HAS_ASSIST = ("assistant_model" in globals()) and (assistant_model is not None)

# --------------------- Log tiện ích ---------------------
def now():
    return datetime.now().strftime("%H:%M:%S %d/%m/%Y")

def log(s=""):
    print(f"[{now()}] {s}", flush=True)

def sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def vram():
    free_b, total_b = torch.cuda.mem_get_info()
    return {
        "free_GB": round(free_b / 1024**3, 3),
        "total_GB": round(total_b / 1024**3, 3),
        "allocated_GB": round(torch.cuda.memory_allocated() / 1024**3, 3),
        "reserved_GB": round(torch.cuda.memory_reserved() / 1024**3, 3),
        "peak_GB": round(torch.cuda.max_memory_allocated() / 1024**3, 3),
    }

def print_vram(tag):
    log(tag + " | " + json.dumps(vram(), ensure_ascii=False))

# --------------------- Tối ưu runtime nhẹ, không đụng model ---------------------
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.benchmark = True
try:
    torch.backends.cudnn.allow_tf32 = True
except Exception:
    pass

model.eval()
model.config.use_cache = True
if HAS_ASSIST:
    assistant_model.eval()
    assistant_model.config.use_cache = True

if getattr(tokenizer, "pad_token_id", None) is None:
    try:
        tokenizer.pad_token = tokenizer.eos_token
    except Exception:
        pass
try:
    tokenizer.padding_side = "left"
except Exception:
    pass

PAD_ID = getattr(tokenizer, "pad_token_id", None)
EOS_ID = getattr(tokenizer, "eos_token_id", None)

COMMON_GEN = dict(
    use_cache=True,
    pad_token_id=PAD_ID if PAD_ID is not None else EOS_ID,
    eos_token_id=EOS_ID,
    return_dict_in_generate=False,
)

SYSTEM_PROMPT = "Bạn là trợ lý kỹ thuật, trả lời tiếng Việt, ưu tiên code chạy được, không lan man."
USER_PROMPT = """Viết một script Python fuzz thử torch.searchsorted theo hướng tìm bug/contract mismatch.
Yêu cầu:
- Có seed cố định.
- Có 8 edge case: sorted, unsorted, duplicate, NaN, Inf, tensor rỗng, dtype lạ, shape 2D.
- In input, output, exception.
- Cuối cùng giải thích ngắn edge case nào đáng nghi nhất.
"""

QUALITY_PROMPT = """Tạo một script Python hoàn chỉnh để fuzz torch.searchsorted.
Yêu cầu code rõ ràng, chạy được, có hàm run_case, có thống kê PASS/EXCEPTION/SUSPICIOUS.
Sau code, giải thích ngắn vì sao unsorted input, NaN/Inf và dtype/shape mismatch đáng fuzz."""

BATCH_PROMPTS = [
    "Viết 5 edge case fuzz torch.searchsorted, code ngắn, có seed.",
    "Viết 5 edge case fuzz torch.linalg.eigvals, code ngắn, có seed.",
    "Viết 5 edge case fuzz torch.nn.functional.interpolate, code ngắn, có seed.",
    "Viết 5 edge case fuzz torch.sparse_csr_tensor, code ngắn, có seed.",
]

def chat_text(user_prompt, enable_thinking=False):
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]
    try:
        return processor.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=enable_thinking,
        )
    except TypeError:
        return processor.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=True,
        )

def encode_one(prompt):
    text = chat_text(prompt, enable_thinking=False)
    x = processor(text=text, return_tensors="pt")
    return x.to("cuda:0"), text

def encode_batch(prompts):
    texts = [chat_text(p, enable_thinking=False) for p in prompts]
    x = processor(text=texts, return_tensors="pt", padding=True)
    return x.to("cuda:0"), texts

def decode_new(out_ids, input_len):
    raw = tokenizer.decode(out_ids[0, input_len:], skip_special_tokens=False)
    try:
        return str(processor.parse_response(raw))
    except Exception:
        return tokenizer.decode(out_ids[0, input_len:], skip_special_tokens=True)

def safe_generate(inputs, gen_kwargs, assist=None):
    kw = dict(inputs)
    kw.update(COMMON_GEN)
    kw.update(gen_kwargs)
    if assist is not None:
        kw["assistant_model"] = assist
    try:
        return model.generate(**kw)
    except Exception as e:
        # Một số model/backend không thích cache_implementation="static"; tự bỏ để cell không chết oan.
        if "cache_implementation" in kw:
            kw.pop("cache_implementation", None)
            return model.generate(**kw)
        raise e

@torch.inference_mode()
def warmup(inputs, assist=None):
    _ = safe_generate(
        inputs,
        dict(max_new_tokens=24, do_sample=False),
        assist=assist,
    )
    sync()

@torch.inference_mode()
def bench_one(name, inputs, gen_kwargs, assist=None, repeats=3, print_full=False):
    # Không empty_cache trong vòng đo vì nó làm méo tốc độ. Chỉ sync.
    warmup(inputs, assist=assist)

    input_len = int(inputs["input_ids"].shape[-1])
    rows = []
    last_out = None

    for r in range(repeats):
        sync()
        t0 = time.perf_counter()
        out = safe_generate(inputs, gen_kwargs, assist=assist)
        sync()
        dt = time.perf_counter() - t0

        new_tokens = int(out.shape[-1] - input_len)
        rows.append((dt, new_tokens, new_tokens / max(dt, 1e-9)))
        last_out = out

    secs = [x[0] for x in rows]
    toks = [x[1] for x in rows]
    speeds = [x[2] for x in rows]

    result = {
        "run": name,
        "assist": "yes" if assist is not None else "no",
        "prompt_tokens": input_len,
        "out_tokens_last": toks[-1],
        "sec_median": round(statistics.median(secs), 4),
        "tok_s_median": round(statistics.median(speeds), 3),
        "tok_s_best": round(max(speeds), 3),
        "tok_s_worst": round(min(speeds), 3),
        "vram_alloc_GB": vram()["allocated_GB"],
        "vram_peak_GB": vram()["peak_GB"],
    }

    if print_full and last_out is not None:
        log("=" * 100)
        log(f"OUTPUT ĐẦY ĐỦ: {name}")
        print(decode_new(last_out, input_len), flush=True)
        log("=" * 100)

    return result, last_out

@torch.inference_mode()
def bench_ttft(name, inputs, assist=None, repeats=8):
    # TTFT gần đúng: đo generate 1 token, lấy median.
    times = []
    for _ in range(repeats):
        sync()
        t0 = time.perf_counter()
        _ = safe_generate(inputs, dict(max_new_tokens=1, do_sample=False), assist=assist)
        sync()
        times.append(time.perf_counter() - t0)
    return {
        "run": name,
        "assist": "yes" if assist is not None else "no",
        "ttft_median_sec": round(statistics.median(times), 4),
        "ttft_best_sec": round(min(times), 4),
        "ttft_worst_sec": round(max(times), 4),
    }

@torch.inference_mode()
def bench_batch_no_mtp(name, prompts, max_new_tokens=160, repeats=2):
    inputs, _ = encode_batch(prompts)
    input_len = int(inputs["input_ids"].shape[-1])
    batch_size = int(inputs["input_ids"].shape[0])

    warmup(inputs, assist=None)

    rows = []
    last_out = None
    for _ in range(repeats):
        sync()
        t0 = time.perf_counter()
        out = safe_generate(
            inputs,
            dict(max_new_tokens=max_new_tokens, do_sample=False),
            assist=None,
        )
        sync()
        dt = time.perf_counter() - t0

        tail = out[:, input_len:]
        if PAD_ID is not None:
            new_tokens_total = int((tail != PAD_ID).sum().item())
        else:
            new_tokens_total = int(tail.numel())

        rows.append((dt, new_tokens_total, new_tokens_total / max(dt, 1e-9)))
        last_out = out

    speeds = [x[2] for x in rows]
    secs = [x[0] for x in rows]
    toks = [x[1] for x in rows]

    # In output mẫu đầu tiên để khỏi spam 4 bài dài.
    log("=" * 100)
    log(f"OUTPUT MẪU BATCH[0]: {name}")
    print(decode_new(last_out, input_len), flush=True)
    log("=" * 100)

    return {
        "run": name,
        "batch_size": batch_size,
        "assist": "no_batch_mtp",
        "prompt_tokens_padded": input_len,
        "out_tokens_total_last": toks[-1],
        "sec_median": round(statistics.median(secs), 4),
        "tok_s_total_median": round(statistics.median(speeds), 3),
        "tok_s_per_request_approx": round(statistics.median(speeds) / batch_size, 3),
        "vram_alloc_GB": vram()["allocated_GB"],
        "vram_peak_GB": vram()["peak_GB"],
    }

# --------------------- Bắt đầu đo ---------------------
log("BẮT ĐẦU TỐI ƯU TIẾP, KHÔNG RELOAD MODEL")
print("GPU:", torch.cuda.get_device_name(0))
print("Có assistant_model:", HAS_ASSIST)
print_vram("VRAM đầu cell tối ưu")

inputs_fast, _ = encode_one(USER_PROMPT)
inputs_quality, _ = encode_one(QUALITY_PROMPT)

# Dọn nhẹ một lần trước toàn bộ benchmark, không dọn trong từng vòng.
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
print_vram("Sau dọn nhẹ 1 lần")

# --------------------- TTFT ---------------------
ttft_rows = []
ttft_rows.append(bench_ttft("TTFT_NO_MTP", inputs_fast, assist=None, repeats=8))
if HAS_ASSIST:
    ttft_rows.append(bench_ttft("TTFT_WITH_MTP_DEFAULT", inputs_fast, assist=assistant_model, repeats=8))

log("BẢNG TTFT")
print(pd.DataFrame(ttft_rows).to_string(index=False))

# --------------------- Baseline no stream ---------------------
gen_fast = dict(max_new_tokens=256, do_sample=False)

base_rows = []
row, _ = bench_one(
    "NO_STREAM_GREEDY_NO_MTP",
    inputs_fast,
    gen_fast,
    assist=None,
    repeats=3,
    print_full=False,
)
base_rows.append(row)

if HAS_ASSIST:
    row, _ = bench_one(
        "NO_STREAM_GREEDY_MTP_DEFAULT",
        inputs_fast,
        gen_fast,
        assist=assistant_model,
        repeats=3,
        print_full=False,
    )
    base_rows.append(row)

# Test static cache chỉ cho no-MTP, vì assisted/static cache dễ kén backend.
row_static = None
try:
    row_static, _ = bench_one(
        "NO_MTP_STATIC_CACHE_TRY",
        inputs_fast,
        dict(max_new_tokens=256, do_sample=False, cache_implementation="static"),
        assist=None,
        repeats=3,
        print_full=False,
    )
    base_rows.append(row_static)
except Exception as e:
    log("STATIC CACHE bỏ qua do lỗi backend/model:")
    print(str(e)[:1200])

log("BẢNG BASELINE NO-STREAM")
print(pd.DataFrame(base_rows).to_string(index=False))

# --------------------- Sweep MTP ---------------------
mtp_rows = []
best_cfg = None
best_speed = -1

if HAS_ASSIST:
    # T4 thường hợp 4/6/8. 12/16 có thể giảm nếu draft sai nhiều.
    for schedule in ["constant", "heuristic"]:
        for n in [2, 4, 6, 8, 12, 16]:
            try:
                assistant_model.generation_config.num_assistant_tokens = n
                assistant_model.generation_config.num_assistant_tokens_schedule = schedule

                row, _ = bench_one(
                    f"MTP_n{n}_{schedule}",
                    inputs_fast,
                    gen_fast,
                    assist=assistant_model,
                    repeats=2,
                    print_full=False,
                )
                row["num_assistant_tokens"] = n
                row["schedule"] = schedule
                mtp_rows.append(row)

                if row["tok_s_median"] > best_speed:
                    best_speed = row["tok_s_median"]
                    best_cfg = (n, schedule)

            except Exception as e:
                log(f"MTP n={n}, schedule={schedule} lỗi, bỏ qua:")
                print(str(e)[:1000])

    log("BẢNG SWEEP MTP")
    df_mtp = pd.DataFrame(mtp_rows).sort_values("tok_s_median", ascending=False)
    print(df_mtp.to_string(index=False))
else:
    log("Không có assistant_model nên bỏ qua sweep MTP.")

# --------------------- In output đầy đủ cấu hình nhanh nhất ---------------------
if HAS_ASSIST and best_cfg is not None:
    best_n, best_schedule = best_cfg
    assistant_model.generation_config.num_assistant_tokens = best_n
    assistant_model.generation_config.num_assistant_tokens_schedule = best_schedule

    log(f"CHẠY OUTPUT ĐẦY ĐỦ VỚI MTP TỐT NHẤT: n={best_n}, schedule={best_schedule}")
    best_row, best_out = bench_one(
        f"BEST_FULL_MTP_n{best_n}_{best_schedule}",
        inputs_quality,
        dict(max_new_tokens=448, do_sample=False),
        assist=assistant_model,
        repeats=1,
        print_full=True,
    )
else:
    log("CHẠY OUTPUT ĐẦY ĐỦ NO-MTP")
    best_row, best_out = bench_one(
        "BEST_FULL_NO_MTP",
        inputs_quality,
        dict(max_new_tokens=448, do_sample=False),
        assist=None,
        repeats=1,
        print_full=True,
    )

# --------------------- Quality sampling, không lấy làm speed chính ---------------------
# Với fuzz/code, greedy thường ổn định hơn. Sampling để xem chất lượng biến thể.
quality_rows = []
try:
    qrow, _ = bench_one(
        "QUALITY_SAMPLE_NO_MTP_temp0.7_top_p0.9_top_k64",
        inputs_quality,
        dict(
            max_new_tokens=448,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            top_k=64,
        ),
        assist=None,
        repeats=1,
        print_full=True,
    )
    quality_rows.append(qrow)
except Exception as e:
    log("Quality sampling lỗi:")
    print(str(e)[:1200])

# --------------------- Batch throughput no-MTP ---------------------
# Assisted/MTP trong HF không dành cho batch input, nên đo batch no-MTP để xem T4 no hơn bao nhiêu.
batch_rows = []
try:
    batch_rows.append(bench_batch_no_mtp("BATCH4_GREEDY_NO_MTP", BATCH_PROMPTS, max_new_tokens=160, repeats=2))
except Exception as e:
    log("Batch4 lỗi:")
    print(traceback.format_exc()[:2000])

try:
    batch_rows.append(bench_batch_no_mtp("BATCH8_GREEDY_NO_MTP", BATCH_PROMPTS * 2, max_new_tokens=128, repeats=2))
except Exception as e:
    log("Batch8 lỗi, có thể thiếu VRAM:")
    print(traceback.format_exc()[:2000])

# --------------------- Tổng kết ---------------------
all_rows = []
all_rows.extend(base_rows)
all_rows.extend(mtp_rows)
all_rows.append(best_row)
all_rows.extend(quality_rows)

df_all = pd.DataFrame(all_rows)
if len(df_all):
    df_all = df_all.sort_values("tok_s_median", ascending=False)

log("=" * 100)
log("TỔNG KẾT SINGLE REQUEST")
print(df_all.to_string(index=False))

if batch_rows:
    log("=" * 100)
    log("TỔNG KẾT BATCH THROUGHPUT NO-MTP")
    print(pd.DataFrame(batch_rows).to_string(index=False))

print_vram("VRAM cuối cell tối ưu")

if HAS_ASSIST and best_cfg is not None:
    log(f"KHUYẾN NGHỊ DÙNG: do_sample=False + MTP n={best_cfg[0]} schedule={best_cfg[1]} cho code/fuzz.")
else:
    log("KHUYẾN NGHỊ DÙNG: do_sample=False no-stream cho code/fuzz; thiếu assistant nên chưa bật MTP.")

log("HOÀN TẤT CELL TỐI ƯU TIẾP")
# ===================== HẾT CELL =====================

[19:08:54 04/06/2026] BẮT ĐẦU TỐI ƯU TIẾP, KHÔNG RELOAD MODEL
GPU: Tesla T4
Có assistant_model: True
[19:08:54 04/06/2026] VRAM đầu cell tối ưu | {"free_GB": 4.709, "total_GB": 14.563, "allocated_GB": 9.671, "reserved_GB": 9.715, "peak_GB": 9.697}
[19:08:55 04/06/2026] Sau dọn nhẹ 1 lần | {"free_GB": 4.744, "total_GB": 14.563, "allocated_GB": 9.671, "reserved_GB": 9.68, "peak_GB": 9.671}
[19:08:58 04/06/2026] BẢNG TTFT
                  run assist  ttft_median_sec  ttft_best_sec  ttft_worst_sec
          TTFT_NO_MTP     no           0.1994         0.1395          0.2910
TTFT_WITH_MTP_DEFAULT    yes           0.2557         0.1728          0.3967


W0604 19:11:32.884000 6590 torch/_inductor/utils.py:1731] [0/0] Not enough SMs to use max_autotune_gemm mode


[19:13:20 04/06/2026] BẢNG BASELINE NO-STREAM
                         run assist  prompt_tokens  out_tokens_last  sec_median  tok_s_median  tok_s_best  tok_s_worst  vram_alloc_GB  vram_peak_GB
     NO_STREAM_GREEDY_NO_MTP     no            122              256     19.0971        13.405      13.778        8.482          9.671         9.702
NO_STREAM_GREEDY_MTP_DEFAULT    yes            122              256      9.5027        26.940      27.192       25.533          9.671         9.766
     NO_MTP_STATIC_CACHE_TRY     no            122              256      7.1095        36.008      36.389        1.792          9.669         9.766
[19:17:40 04/06/2026] BẢNG SWEEP MTP
              run assist  prompt_tokens  out_tokens_last  sec_median  tok_s_median  tok_s_best  tok_s_worst  vram_alloc_GB  vram_peak_GB  num_assistant_tokens  schedule
  MTP_n6_constant    yes            122              256      8.9247        28.714      29.637       27.791          9.669         9.766                    

In [ ]:
# ================= GEMMA 4 T4 - CELL 3: CHỐT STATIC CACHE NHANH NHẤT =================
# Chạy sau 2 cell trước. Không tải lại model. Không pip install.
# Mục tiêu:
# - Lấy static cache làm đường chính vì log đã đạt ~36 tok/s.
# - Đo lại nhiều vòng, bỏ warmup/compile spike.
# - So sánh static cache vs MTP n=6.
# - Sửa batch test bị sinh <pad>.
# - In output cuối bằng cấu hình nhanh nhất.

import gc, time, json, statistics, traceback
from datetime import datetime
import torch
import pandas as pd

assert "model" in globals(), "Thiếu model: phải chạy sau cell load Gemma 4."
assert "processor" in globals(), "Thiếu processor: phải chạy sau cell load Gemma 4."

tok = getattr(processor, "tokenizer", processor)
HAS_ASSIST = ("assistant_model" in globals()) and (assistant_model is not None)

def now():
    return datetime.now().strftime("%H:%M:%S %d/%m/%Y")

def log(x=""):
    print(f"[{now()}] {x}", flush=True)

def sync():
    torch.cuda.synchronize()

def vram():
    free_b, total_b = torch.cuda.mem_get_info()
    return {
        "free_GB": round(free_b / 1024**3, 3),
        "total_GB": round(total_b / 1024**3, 3),
        "alloc_GB": round(torch.cuda.memory_allocated() / 1024**3, 3),
        "reserved_GB": round(torch.cuda.memory_reserved() / 1024**3, 3),
        "peak_GB": round(torch.cuda.max_memory_allocated() / 1024**3, 3),
    }

def show_vram(tag):
    log(tag + " | " + json.dumps(vram(), ensure_ascii=False))

model.eval()
model.config.use_cache = True
for p in model.parameters():
    p.requires_grad_(False)

if HAS_ASSIST:
    assistant_model.eval()
    assistant_model.config.use_cache = True
    assistant_model.generation_config.num_assistant_tokens = 6
    assistant_model.generation_config.num_assistant_tokens_schedule = "constant"
    for p in assistant_model.parameters():
        p.requires_grad_(False)

try:
    torch.set_float32_matmul_precision("high")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.enable_flash_sdp(True)
    torch.backends.cuda.enable_mem_efficient_sdp(True)
    torch.backends.cuda.enable_math_sdp(True)
except Exception:
    pass

PAD_ID = getattr(tok, "pad_token_id", None)
EOS_ID = getattr(tok, "eos_token_id", None)
if PAD_ID is None:
    try:
        tok.pad_token = tok.eos_token
        PAD_ID = tok.pad_token_id
    except Exception:
        pass

try:
    tok.padding_side = "left"
except Exception:
    pass

SYSTEM = (
    "Bạn là AI sinh mã fuzzing PyTorch. "
    "Chỉ trả lời phần cần thiết, ưu tiên code chạy được, không chào hỏi, không markdown rườm rà."
)

PROMPT_FAST = """
Viết script Python fuzz torch.searchsorted.
Yêu cầu:
- Không dùng markdown fence.
- Có seed cố định.
- Có hàm run_case.
- Test sorted, unsorted, duplicate, NaN, Inf, tensor rỗng, dtype int/float, shape 2D.
- In rõ input, output, exception.
- Cuối file in thống kê PASS/EXCEPTION/SUSPICIOUS.
"""

PROMPT_QUALITY = """
Viết script Python hoàn chỉnh để fuzz torch.searchsorted theo hướng tìm contract mismatch.
Yêu cầu:
- Không dùng markdown fence.
- Có seed cố định.
- Có danh sách case thủ công và vài case random nhỏ.
- Có oracle cơ bản: nếu sorted 1D thì kiểm tra vị trí trả về có hợp lý không.
- Với unsorted/NaN/Inf/shape 2D thì đánh dấu SUSPICIOUS nếu output im lặng khó diễn giải.
- In bảng tóm tắt cuối cùng.
"""

def chat_text(user_prompt):
    msgs = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": user_prompt},
    ]
    try:
        return processor.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        return processor.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=True,
        )

def encode_one(prompt):
    text = chat_text(prompt)
    x = processor(text=text, return_tensors="pt")
    return x.to("cuda:0"), text

def decode_new(out, input_len):
    raw = tok.decode(out[0, input_len:], skip_special_tokens=False)
    try:
        parsed = processor.parse_response(raw)
        if isinstance(parsed, dict):
            return parsed.get("content", str(parsed))
        return str(parsed)
    except Exception:
        return tok.decode(out[0, input_len:], skip_special_tokens=True)

COMMON = dict(
    use_cache=True,
    pad_token_id=PAD_ID if PAD_ID is not None else EOS_ID,
    eos_token_id=EOS_ID,
    return_dict_in_generate=False,
)

STATIC_GREEDY = dict(
    **COMMON,
    cache_implementation="static",
    do_sample=False,
)

DYNAMIC_GREEDY = dict(
    **COMMON,
    do_sample=False,
)

STATIC_SAMPLE_SAFE = dict(
    **COMMON,
    cache_implementation="static",
    do_sample=True,
    temperature=0.35,
    top_p=0.9,
    top_k=64,
)

def generate_safe(inputs, max_new_tokens, gen_cfg, assist=None):
    kw = dict(inputs)
    kw.update(gen_cfg)
    kw["max_new_tokens"] = max_new_tokens
    if assist is not None:
        kw["assistant_model"] = assist
    return model.generate(**kw)

@torch.inference_mode()
def bench(label, inputs, gen_cfg, assist=None, max_new_tokens=256, warmups=3, repeats=8, print_output=False):
    input_len = int(inputs["input_ids"].shape[-1])
    rows = []
    last = None

    log(f"RUN {label} | warmups={warmups}, repeats={repeats}, max_new={max_new_tokens}")
    for i in range(warmups + repeats):
        sync()
        t0 = time.perf_counter()
        out = generate_safe(inputs, max_new_tokens, gen_cfg, assist=assist)
        sync()
        dt = time.perf_counter() - t0

        new_tok = int(out.shape[-1] - input_len)
        speed = new_tok / max(dt, 1e-9)
        last = out

        phase = "warmup" if i < warmups else "measure"
        log(f"{label} | {phase} {i+1:02d} | {new_tok} tok / {dt:.4f}s = {speed:.2f} tok/s")

        if i >= warmups:
            rows.append(speed)

    result = {
        "run": label,
        "assist": assist is not None,
        "prompt_tokens": input_len,
        "max_new_tokens": max_new_tokens,
        "median_tok_s": round(statistics.median(rows), 3),
        "mean_tok_s": round(statistics.mean(rows), 3),
        "best_tok_s": round(max(rows), 3),
        "worst_tok_s": round(min(rows), 3),
        "vram_alloc_GB": vram()["alloc_GB"],
        "vram_peak_GB": vram()["peak_GB"],
    }

    if print_output:
        log("=" * 100)
        log(f"OUTPUT ĐẦY ĐỦ: {label}")
        print(decode_new(last, input_len), flush=True)
        log("=" * 100)

    return result, last

log("BẮT ĐẦU CELL 3: STATIC CACHE LÀM ĐƯỜNG CHÍNH")
print("GPU:", torch.cuda.get_device_name(0))
print("Có assistant_model:", HAS_ASSIST)
show_vram("VRAM đầu cell 3")

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
show_vram("Sau dọn nhẹ")

inputs_fast, _ = encode_one(PROMPT_FAST)
inputs_quality, _ = encode_one(PROMPT_QUALITY)

results = []

# 1) Đường chính: static cache, greedy, không MTP.
r, _ = bench(
    "STATIC_GREEDY_MAIN_256",
    inputs_fast,
    STATIC_GREEDY,
    assist=None,
    max_new_tokens=256,
    warmups=4,
    repeats=8,
    print_output=False,
)
results.append(r)

# 2) Đo dài hơn để xem tốc độ ổn không khi sinh 512 token.
r, _ = bench(
    "STATIC_GREEDY_MAIN_512",
    inputs_quality,
    STATIC_GREEDY,
    assist=None,
    max_new_tokens=512,
    warmups=2,
    repeats=5,
    print_output=False,
)
results.append(r)

# 3) MTP tốt nhất từ log cũ: n=6 constant.
if HAS_ASSIST:
    try:
        r, _ = bench(
            "DYNAMIC_MTP_N6_CONSTANT_256",
            inputs_fast,
            DYNAMIC_GREEDY,
            assist=assistant_model,
            max_new_tokens=256,
            warmups=2,
            repeats=5,
            print_output=False,
        )
        results.append(r)
    except Exception:
        log("MTP lỗi, bỏ qua:")
        print(traceback.format_exc()[:1800])

# 4) Thử static + MTP. Nếu backend không chịu thì tự bỏ qua.
if HAS_ASSIST:
    try:
        r, _ = bench(
            "STATIC_PLUS_MTP_TRY_256",
            inputs_fast,
            STATIC_GREEDY,
            assist=assistant_model,
            max_new_tokens=256,
            warmups=1,
            repeats=3,
            print_output=False,
        )
        results.append(r)
    except Exception:
        log("Static + MTP không hợp backend hiện tại, bỏ qua là đúng:")
        print(traceback.format_exc()[:1800])

# 5) Sampling nhẹ cho chất lượng biến thể. Không lấy làm đường nhanh nhất.
try:
    r, _ = bench(
        "STATIC_SAMPLE_QUALITY_512",
        inputs_quality,
        STATIC_SAMPLE_SAFE,
        assist=None,
        max_new_tokens=512,
        warmups=1,
        repeats=2,
        print_output=False,
    )
    results.append(r)
except Exception:
    log("Sampling static lỗi, bỏ qua:")
    print(traceback.format_exc()[:1800])

df = pd.DataFrame(results).sort_values("median_tok_s", ascending=False)
log("=" * 100)
log("BẢNG SO SÁNH CELL 3")
print(df.to_string(index=False))

# Chạy output cuối bằng cấu hình nhanh nhất thực dụng: static greedy.
log("CHẠY OUTPUT CUỐI BẰNG STATIC GREEDY")
final_row, final_out = bench(
    "FINAL_STATIC_GREEDY_FULL_OUTPUT",
    inputs_quality,
    STATIC_GREEDY,
    assist=None,
    max_new_tokens=768,
    warmups=1,
    repeats=1,
    print_output=True,
)

# Batch test sửa lỗi <pad>: chỉ để đo throughput nhiều prompt, không dùng MTP.
def encode_batch(prompts):
    texts = [chat_text(p) for p in prompts]
    x = processor(text=texts, return_tensors="pt", padding=True)
    return x.to("cuda:0"), texts

@torch.inference_mode()
def batch_fixed_test():
    prompts = [
        "Viết code fuzz torch.searchsorted, ngắn, không markdown.",
        "Viết code fuzz torch.linalg.eigvals, ngắn, không markdown.",
        "Viết code fuzz torch.nn.functional.interpolate, ngắn, không markdown.",
        "Viết code fuzz torch.sparse_csr_tensor, ngắn, không markdown.",
    ]
    x, _ = encode_batch(prompts)
    input_len = int(x["input_ids"].shape[-1])

    gen_cfg = dict(STATIC_GREEDY)
    # Chặn model phun <pad>; nếu PAD trùng EOS thì không chặn.
    if PAD_ID is not None and EOS_ID is not None and PAD_ID != EOS_ID:
        gen_cfg["bad_words_ids"] = [[PAD_ID]]

    # warmup
    _ = generate_safe(x, 32, gen_cfg, assist=None)
    sync()

    sync()
    t0 = time.perf_counter()
    out = generate_safe(x, 160, gen_cfg, assist=None)
    sync()
    dt = time.perf_counter() - t0

    decoded = []
    valid_tokens = 0
    for i in range(out.shape[0]):
        tail = out[i, input_len:]
        if PAD_ID is not None and EOS_ID is not None:
            mask = (tail != PAD_ID) & (tail != EOS_ID)
            valid_tokens += int(mask.sum().item())
        else:
            valid_tokens += int(tail.numel())
        decoded.append(tok.decode(tail, skip_special_tokens=True))

    log("=" * 100)
    log("BATCH4 FIXED OUTPUT MẪU 0")
    print(decoded[0][:2000], flush=True)
    log("=" * 100)

    return {
        "batch_run": "BATCH4_STATIC_SUPPRESS_PAD",
        "batch_size": 4,
        "valid_generated_tokens": valid_tokens,
        "sec": round(dt, 4),
        "valid_tok_s_total": round(valid_tokens / max(dt, 1e-9), 3),
        "valid_tok_s_per_prompt": round(valid_tokens / max(dt, 1e-9) / 4, 3),
    }

try:
    b = batch_fixed_test()
    log("BẢNG BATCH FIXED")
    print(pd.DataFrame([b]).to_string(index=False))
except Exception:
    log("Batch fixed lỗi, không ảnh hưởng single request:")
    print(traceback.format_exc()[:1800])

show_vram("VRAM cuối cell 3")
log("KHUYẾN NGHỊ CUỐI: dùng STATIC_GREEDY cho single prompt/code/fuzz; MTP n=6 constant chỉ là phương án phụ.")
log("HOÀN TẤT CELL 3")
# ================= HẾT CELL 3 =================

[19:22:04 04/06/2026] BẮT ĐẦU CELL 3: STATIC CACHE LÀM ĐƯỜNG CHÍNH
GPU: Tesla T4
Có assistant_model: True
[19:22:04 04/06/2026] VRAM đầu cell 3 | {"free_GB": 4.564, "total_GB": 14.563, "alloc_GB": 9.669, "reserved_GB": 9.836, "peak_GB": 9.766}
[19:22:06 04/06/2026] Sau dọn nhẹ | {"free_GB": 4.685, "total_GB": 14.563, "alloc_GB": 9.669, "reserved_GB": 9.715, "peak_GB": 9.669}
[19:22:06 04/06/2026] RUN STATIC_GREEDY_MAIN_256 | warmups=4, repeats=8, max_new=256
[19:23:19 04/06/2026] STATIC_GREEDY_MAIN_256 | warmup 01 | 256 tok / 72.6712s = 3.52 tok/s
[19:23:26 04/06/2026] STATIC_GREEDY_MAIN_256 | warmup 02 | 256 tok / 7.2392s = 35.36 tok/s
[19:23:33 04/06/2026] STATIC_GREEDY_MAIN_256 | warmup 03 | 256 tok / 7.1392s = 35.86 tok/s
[19:23:41 04/06/2026] STATIC_GREEDY_MAIN_256 | warmup 04 | 256 tok / 7.2673s = 35.23 tok/s
[19:23:48 04/06/2026] STATIC_GREEDY_MAIN_256 | measure 05 | 256 tok / 7.1633s = 35.74 tok/s
[19:23:55 04/06/2026] STATIC_GREEDY_MAIN_256 | measure 06 | 256 tok / 7.3046s = 3

[transformers] An assistant model is provided, using a dynamic cache instead of a cache of type='static'.


[19:28:54 04/06/2026] STATIC_PLUS_MTP_TRY_256 | warmup 01 | 256 tok / 8.8248s = 29.01 tok/s
[19:29:03 04/06/2026] STATIC_PLUS_MTP_TRY_256 | measure 02 | 256 tok / 8.7941s = 29.11 tok/s
[19:29:12 04/06/2026] STATIC_PLUS_MTP_TRY_256 | measure 03 | 256 tok / 8.7079s = 29.40 tok/s
[19:29:20 04/06/2026] STATIC_PLUS_MTP_TRY_256 | measure 04 | 256 tok / 8.3162s = 30.78 tok/s
[19:29:20 04/06/2026] RUN STATIC_SAMPLE_QUALITY_512 | warmups=1, repeats=2, max_new=512
[19:29:35 04/06/2026] STATIC_SAMPLE_QUALITY_512 | warmup 01 | 512 tok / 15.1947s = 33.70 tok/s
[19:29:51 04/06/2026] STATIC_SAMPLE_QUALITY_512 | measure 02 | 512 tok / 15.2621s = 33.55 tok/s
[19:30:06 04/06/2026] STATIC_SAMPLE_QUALITY_512 | measure 03 | 512 tok / 15.1051s = 33.90 tok/s
[19:30:06 04/06/2026] ====================================================================================================
[19:30:06 04/06/2026] BẢNG SO SÁNH CELL 3
                        run  assist  prompt_tokens  max_new_tokens  median_tok_s  mean_to

In [ ]:
# ================= GEMMA 4 T4 - CELL 4: VẮT NỐT PHẦN CÒN LẠI =================
# Chạy sau cell 3. Không reload model, không pip install.
# Mục tiêu:
# 1) Test prompt siêu ngắn để giảm prefill/overhead.
# 2) Test compile_config nếu Transformers hiện tại hỗ trợ.
# 3) Test stop sớm để không phí token.
# 4) Test batch kiểu decode đúng hơn, không tin bảng pad cũ quá mức.
# 5) In khuyến nghị cuối.

import gc, time, json, statistics, traceback
from datetime import datetime
import torch
import pandas as pd

assert "model" in globals()
assert "processor" in globals()

tok = getattr(processor, "tokenizer", processor)

def now():
    return datetime.now().strftime("%H:%M:%S %d/%m/%Y")

def log(x=""):
    print(f"[{now()}] {x}", flush=True)

def sync():
    torch.cuda.synchronize()

def vram():
    free_b, total_b = torch.cuda.mem_get_info()
    return {
        "free_GB": round(free_b / 1024**3, 3),
        "alloc_GB": round(torch.cuda.memory_allocated() / 1024**3, 3),
        "reserved_GB": round(torch.cuda.memory_reserved() / 1024**3, 3),
        "peak_GB": round(torch.cuda.max_memory_allocated() / 1024**3, 3),
    }

PAD_ID = getattr(tok, "pad_token_id", None)
EOS_ID = getattr(tok, "eos_token_id", None)
if PAD_ID is None:
    try:
        tok.pad_token = tok.eos_token
        PAD_ID = tok.pad_token_id
    except Exception:
        pass

try:
    tok.padding_side = "left"
except Exception:
    pass

model.eval()
model.config.use_cache = True
for p in model.parameters():
    p.requires_grad_(False)

try:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

def chat_text(user_prompt, system_prompt):
    msgs = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    try:
        return processor.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        return processor.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=True,
        )

def encode_one(user_prompt, system_prompt):
    text = chat_text(user_prompt, system_prompt)
    x = processor(text=text, return_tensors="pt")
    return x.to("cuda:0"), text

def decode_new(out, input_len):
    raw = tok.decode(out[0, input_len:], skip_special_tokens=False)
    try:
        parsed = processor.parse_response(raw)
        if isinstance(parsed, dict):
            return parsed.get("content", str(parsed))
        return str(parsed)
    except Exception:
        return tok.decode(out[0, input_len:], skip_special_tokens=True)

COMMON = dict(
    use_cache=True,
    cache_implementation="static",
    do_sample=False,
    pad_token_id=PAD_ID if PAD_ID is not None else EOS_ID,
    eos_token_id=EOS_ID,
    return_dict_in_generate=False,
)

SYSTEM_SHORT = "Trả lời bằng code Python chạy được, không markdown, không chào."
SYSTEM_TINY = "Code Python, no markdown."

PROMPT_SHORT = """Fuzz torch.searchsorted: seed=42, run_case, cases sorted/unsorted/dup/nan/inf/empty/int/float/2d, print output/exception/stats."""

PROMPT_MEDIUM = """Viết script Python fuzz torch.searchsorted. Không markdown. Có seed=42, run_case, oracle cơ bản cho sorted 1D, đánh dấu SUSPICIOUS cho unsorted/NaN/Inf/2D nếu không exception, in thống kê cuối."""

# -------- thử import CompileConfig --------
CompileConfig = None
try:
    from transformers.generation.configuration_utils import CompileConfig
except Exception:
    try:
        from transformers import CompileConfig
    except Exception:
        CompileConfig = None

def gen_safe(inputs, max_new_tokens, extra=None):
    kw = dict(inputs)
    kw.update(COMMON)
    kw["max_new_tokens"] = max_new_tokens
    if extra:
        kw.update(extra)
    return model.generate(**kw)

@torch.inference_mode()
def bench(label, inputs, max_new=256, extra=None, warmups=2, repeats=6, print_output=False):
    input_len = int(inputs["input_ids"].shape[-1])
    speeds = []
    last = None

    log(f"RUN {label} | prompt_tokens={input_len} | max_new={max_new}")
    for i in range(warmups + repeats):
        sync()
        t0 = time.perf_counter()
        out = gen_safe(inputs, max_new, extra=extra)
        sync()
        dt = time.perf_counter() - t0
        ntok = int(out.shape[-1] - input_len)
        spd = ntok / max(dt, 1e-9)
        phase = "warmup" if i < warmups else "measure"
        log(f"{label} | {phase} {i+1:02d} | {ntok} tok / {dt:.4f}s = {spd:.2f} tok/s")
        if i >= warmups:
            speeds.append(spd)
        last = out

    row = {
        "run": label,
        "prompt_tokens": input_len,
        "max_new": max_new,
        "median_tok_s": round(statistics.median(speeds), 3),
        "mean_tok_s": round(statistics.mean(speeds), 3),
        "best_tok_s": round(max(speeds), 3),
        "worst_tok_s": round(min(speeds), 3),
        "vram_alloc_GB": vram()["alloc_GB"],
        "vram_peak_GB": vram()["peak_GB"],
    }

    if print_output:
        log("=" * 100)
        log(f"OUTPUT {label}")
        print(decode_new(last, input_len), flush=True)
        log("=" * 100)

    return row, last

log("BẮT ĐẦU CELL 4")
print("GPU:", torch.cuda.get_device_name(0))
print("CompileConfig:", CompileConfig)
print("VRAM:", json.dumps(vram(), ensure_ascii=False))

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

inputs_short, _ = encode_one(PROMPT_SHORT, SYSTEM_SHORT)
inputs_tiny, _ = encode_one(PROMPT_SHORT, SYSTEM_TINY)
inputs_medium, _ = encode_one(PROMPT_MEDIUM, SYSTEM_SHORT)

rows = []

# 1) Prompt ngắn hơn: có thể không tăng tok/s decode nhiều, nhưng giảm latency/prefill.
r, _ = bench("STATIC_SHORT_PROMPT_256", inputs_short, max_new=256, warmups=2, repeats=8)
rows.append(r)

r, _ = bench("STATIC_TINY_PROMPT_256", inputs_tiny, max_new=256, warmups=2, repeats=8)
rows.append(r)

r, _ = bench("STATIC_MEDIUM_PROMPT_256", inputs_medium, max_new=256, warmups=2, repeats=6)
rows.append(r)

# 2) compile_config nếu bản transformers hỗ trợ. Không dùng max-autotune vì T4 từng báo không đủ SM.
if CompileConfig is not None:
    compile_trials = []
    for mode in ["reduce-overhead", "default"]:
        try:
            cfg = CompileConfig(mode=mode)
            extra = {"compile_config": cfg}
            r, _ = bench(
                f"STATIC_COMPILE_CONFIG_{mode}_256",
                inputs_short,
                max_new=256,
                extra=extra,
                warmups=2,
                repeats=5,
            )
            rows.append(r)
            compile_trials.append(r)
        except Exception:
            log(f"CompileConfig mode={mode} lỗi, bỏ qua:")
            print(traceback.format_exc()[:1500])
else:
    log("Bản transformers này không expose CompileConfig, bỏ qua nhánh compile_config.")

# 3) Output cuối: dùng prompt ngắn nhưng đủ ràng buộc.
r, out = bench(
    "FINAL_FAST_STATIC_SHORT_512",
    inputs_medium,
    max_new=512,
    warmups=1,
    repeats=1,
    print_output=True,
)
rows.append(r)

df = pd.DataFrame(rows).sort_values("median_tok_s", ascending=False)
log("=" * 100)
log("BẢNG CELL 4")
print(df.to_string(index=False))

# 4) Batch sửa kiểu decode: không ép bad_words pad nữa, thay vào đó dùng prompt đồng độ dài và decode từng dòng.
@torch.inference_mode()
def batch_same_len():
    prompts = [
        "Code Python fuzz torch.searchsorted, seed=42, no markdown.",
        "Code Python fuzz torch.searchsorted, seed=43, no markdown.",
        "Code Python fuzz torch.searchsorted, seed=44, no markdown.",
        "Code Python fuzz torch.searchsorted, seed=45, no markdown.",
    ]
    texts = [chat_text(p, SYSTEM_TINY) for p in prompts]
    x = processor(text=texts, return_tensors="pt", padding=True).to("cuda:0")
    input_len = int(x["input_ids"].shape[-1])

    # warmup
    _ = gen_safe(x, 64)
    sync()

    sync()
    t0 = time.perf_counter()
    out = gen_safe(x, 192)
    sync()
    dt = time.perf_counter() - t0

    decoded = []
    real_tokens = 0
    for i in range(out.shape[0]):
        tail = out[i, input_len:]
        # đếm tương đối: bỏ eos/pad nếu có
        mask = torch.ones_like(tail, dtype=torch.bool)
        if PAD_ID is not None:
            mask &= tail != PAD_ID
        if EOS_ID is not None:
            mask &= tail != EOS_ID
        real_tokens += int(mask.sum().item())
        decoded.append(tok.decode(tail, skip_special_tokens=True))

    log("=" * 100)
    log("BATCH SAME-LEN OUTPUT MẪU 0")
    print(decoded[0][:2500], flush=True)
    log("=" * 100)

    return {
        "batch": "BATCH4_SAME_LEN_STATIC",
        "input_len": input_len,
        "real_tokens": real_tokens,
        "sec": round(dt, 4),
        "tok_s_total": round(real_tokens / max(dt, 1e-9), 3),
        "tok_s_per_prompt": round(real_tokens / max(dt, 1e-9) / 4, 3),
    }

try:
    brow = batch_same_len()
    log("BẢNG BATCH SAME-LEN")
    print(pd.DataFrame([brow]).to_string(index=False))
except Exception:
    log("Batch same-len lỗi, không ảnh hưởng single:")
    print(traceback.format_exc()[:1500])

print("VRAM cuối:", json.dumps(vram(), ensure_ascii=False))
log("KẾT LUẬN: nếu cell 4 không vượt rõ 36 tok/s, coi như trần thực dụng của Transformers trên T4 đã tới.")
log("HOÀN TẤT CELL 4")
# ================= HẾT CELL 4 =================

[19:35:47 04/06/2026] BẮT ĐẦU CELL 4
GPU: Tesla T4
CompileConfig: <class 'transformers.generation.configuration_utils.CompileConfig'>
VRAM: {"free_GB": 4.646, "alloc_GB": 9.67, "reserved_GB": 9.723, "peak_GB": 9.713}
[19:35:52 04/06/2026] RUN STATIC_SHORT_PROMPT_256 | prompt_tokens=72 | max_new=256
[19:35:59 04/06/2026] STATIC_SHORT_PROMPT_256 | warmup 01 | 256 tok / 7.1757s = 35.68 tok/s
[19:36:06 04/06/2026] STATIC_SHORT_PROMPT_256 | warmup 02 | 256 tok / 7.1260s = 35.92 tok/s
[19:36:13 04/06/2026] STATIC_SHORT_PROMPT_256 | measure 03 | 256 tok / 7.2106s = 35.50 tok/s
[19:36:20 04/06/2026] STATIC_SHORT_PROMPT_256 | measure 04 | 256 tok / 7.0842s = 36.14 tok/s
[19:36:28 04/06/2026] STATIC_SHORT_PROMPT_256 | measure 05 | 256 tok / 7.2418s = 35.35 tok/s
[19:36:35 04/06/2026] STATIC_SHORT_PROMPT_256 | measure 06 | 256 tok / 7.0981s = 36.07 tok/s
[19:36:42 04/06/2026] STATIC_SHORT_PROMPT_256 | measure 07 | 256 tok / 7.2809s = 35.16 tok/s
[19:36:49 04/06/2026] STATIC_SHORT_PROMPT_256 | mea

In [ ]:
# ================= CELL 5: CHOT CONFIG NHANH, KHONG RELOAD MODEL =================
import gc, time, json, statistics, traceback
from datetime import datetime
import torch
import pandas as pd
from transformers import StoppingCriteria, StoppingCriteriaList

assert "model" in globals(), "Chua co bien model"
assert "processor" in globals(), "Chua co bien processor"

tok = getattr(processor, "tokenizer", processor)

def now():
    return datetime.now().strftime("%H:%M:%S %d/%m/%Y")

def log(x=""):
    print(f"[{now()}] {x}", flush=True)

def sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def vram():
    free_b, total_b = torch.cuda.mem_get_info()
    return {
        "free_GB": round(free_b / 1024**3, 3),
        "alloc_GB": round(torch.cuda.memory_allocated() / 1024**3, 3),
        "reserved_GB": round(torch.cuda.memory_reserved() / 1024**3, 3),
        "peak_GB": round(torch.cuda.max_memory_allocated() / 1024**3, 3),
    }

PAD_ID = getattr(tok, "pad_token_id", None)
EOS_ID = getattr(tok, "eos_token_id", None)

if PAD_ID is None and EOS_ID is not None:
    try:
        tok.pad_token = tok.eos_token
        PAD_ID = tok.pad_token_id
    except Exception:
        PAD_ID = EOS_ID

try:
    tok.padding_side = "left"
except Exception:
    pass

model.eval()
model.config.use_cache = True
for p in model.parameters():
    p.requires_grad_(False)

try:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

COMMON_GEN = dict(
    do_sample=False,
    use_cache=True,
    cache_implementation="static",
    pad_token_id=PAD_ID if PAD_ID is not None else EOS_ID,
    eos_token_id=EOS_ID,
    return_dict_in_generate=False,
)

def chat_text(user_prompt, system_prompt="Code Python, no markdown."):
    msgs = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    try:
        return processor.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        return processor.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=True,
        )

def encode_one(prompt, system_prompt="Code Python, no markdown."):
    text = chat_text(prompt, system_prompt)
    return processor(text=text, return_tensors="pt").to("cuda:0")

def decode_new(out, input_len):
    return tok.decode(out[0, input_len:], skip_special_tokens=True)

class StopOnText(StoppingCriteria):
    def __init__(self, stop_texts):
        self.stop_ids = []
        for s in stop_texts:
            ids = tok.encode(s, add_special_tokens=False)
            if ids:
                self.stop_ids.append(torch.tensor(ids, device="cuda:0"))

    def __call__(self, input_ids, scores, **kwargs):
        if not self.stop_ids:
            return False
        tail = input_ids[0]
        for ids in self.stop_ids:
            n = ids.numel()
            if tail.numel() >= n and torch.equal(tail[-n:], ids):
                return True
        return False

@torch.inference_mode()
def gen(inputs, max_new_tokens=256, stopping_criteria=None):
    kw = dict(inputs)
    kw.update(COMMON_GEN)
    kw["max_new_tokens"] = max_new_tokens
    if stopping_criteria is not None:
        kw["stopping_criteria"] = stopping_criteria
    return model.generate(**kw)

@torch.inference_mode()
def bench(label, prompt, max_new=256, warmups=1, repeats=4, stop_texts=None, print_last=True):
    inputs = encode_one(prompt)
    input_len = int(inputs["input_ids"].shape[-1])
    stopper = StoppingCriteriaList([StopOnText(stop_texts)]) if stop_texts else None
    speeds = []
    last = None

    log(f"RUN {label} | prompt_tokens={input_len} | max_new={max_new}")
    for i in range(warmups + repeats):
        sync()
        t0 = time.perf_counter()
        out = gen(inputs, max_new_tokens=max_new, stopping_criteria=stopper)
        sync()
        dt = time.perf_counter() - t0
        ntok = int(out.shape[-1] - input_len)
        spd = ntok / max(dt, 1e-9)
        phase = "warmup" if i < warmups else "measure"
        log(f"{label} | {phase} {i+1:02d} | {ntok} tok / {dt:.4f}s = {spd:.2f} tok/s")
        if i >= warmups:
            speeds.append(spd)
        last = out

    text = decode_new(last, input_len)
    row = {
        "run": label,
        "prompt_tokens": input_len,
        "max_new": max_new,
        "median_tok_s": round(statistics.median(speeds), 3),
        "mean_tok_s": round(statistics.mean(speeds), 3),
        "best_tok_s": round(max(speeds), 3),
        "worst_tok_s": round(min(speeds), 3),
        "last_new_tokens": int(last.shape[-1] - input_len),
        "vram_alloc_GB": vram()["alloc_GB"],
    }

    if print_last:
        log("=" * 100)
        log(f"OUTPUT {label}")
        print(text[:4000], flush=True)
        log("=" * 100)

    return row, text

def fast_code(prompt, max_new_tokens=384):
    inputs = encode_one(prompt)
    input_len = int(inputs["input_ids"].shape[-1])
    out = gen(inputs, max_new_tokens=max_new_tokens)
    return decode_new(out, input_len)

log("BAT DAU CELL 5")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM dau:", json.dumps(vram(), ensure_ascii=False))

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

PROMPT_BASE = """Viet script Python fuzz torch.searchsorted.
Yeu cau: seed=42, run_case, test sorted/unsorted/duplicate/nan/inf/empty/int/float/2d.
Co oracle co ban cho sorted 1D.
In PASS/EXCEPTION/SUSPICIOUS va thong ke cuoi.
Khong markdown, chi code."""

PROMPT_STOP = PROMPT_BASE + """
Ket thuc output bang dung mot dong:
# END"""

rows = []

r, _ = bench(
    "STATIC_FAST_256",
    PROMPT_BASE,
    max_new=256,
    warmups=1,
    repeats=5,
    print_last=False,
)
rows.append(r)

r, _ = bench(
    "STATIC_FAST_384",
    PROMPT_BASE,
    max_new=384,
    warmups=1,
    repeats=3,
    print_last=True,
)
rows.append(r)

r, _ = bench(
    "STATIC_STOP_MARKER_512",
    PROMPT_STOP,
    max_new=512,
    warmups=1,
    repeats=3,
    stop_texts=["# END"],
    print_last=True,
)
rows.append(r)

@torch.inference_mode()
def batch4_test():
    prompts = [
        "Code Python fuzz torch.searchsorted seed=42, no markdown.",
        "Code Python fuzz torch.searchsorted seed=43, no markdown.",
        "Code Python fuzz torch.searchsorted seed=44, no markdown.",
        "Code Python fuzz torch.searchsorted seed=45, no markdown.",
    ]
    texts = [chat_text(p) for p in prompts]
    x = processor(text=texts, return_tensors="pt", padding=True).to("cuda:0")
    input_len = int(x["input_ids"].shape[-1])

    _ = gen(x, max_new_tokens=32)
    sync()

    sync()
    t0 = time.perf_counter()
    out = gen(x, max_new_tokens=192)
    sync()
    dt = time.perf_counter() - t0

    real_tokens = 0
    decoded = []
    for i in range(out.shape[0]):
        tail = out[i, input_len:]
        mask = torch.ones_like(tail, dtype=torch.bool)
        if PAD_ID is not None:
            mask &= tail != PAD_ID
        if EOS_ID is not None:
            mask &= tail != EOS_ID
        real_tokens += int(mask.sum().item())
        decoded.append(tok.decode(tail, skip_special_tokens=True))

    log("=" * 100)
    log("BATCH4 SAMPLE OUTPUT 0")
    print(decoded[0][:3000], flush=True)
    log("=" * 100)

    return {
        "run": "BATCH4_STATIC_192",
        "batch": 4,
        "input_len": input_len,
        "real_tokens": real_tokens,
        "sec": round(dt, 4),
        "tok_s_total": round(real_tokens / max(dt, 1e-9), 3),
        "tok_s_per_prompt": round(real_tokens / max(dt, 1e-9) / 4, 3),
        "vram_alloc_GB": vram()["alloc_GB"],
    }

try:
    brow = batch4_test()
except Exception:
    brow = None
    log("Batch4 loi, bo qua:")
    print(traceback.format_exc()[:1500])

df = pd.DataFrame(rows).sort_values("median_tok_s", ascending=False)
log("=" * 100)
log("BANG SINGLE")
print(df.to_string(index=False))

if brow is not None:
    log("=" * 100)
    log("BANG BATCH")
    print(pd.DataFrame([brow]).to_string(index=False))

print("VRAM cuoi:", json.dumps(vram(), ensure_ascii=False))
log("KET LUAN: dung fast_code(prompt, max_new_tokens=384) cho chay that; neu can nhieu prompt thi batch4 dang tien hon.")
log("HOAN TAT CELL 5")
# ================= HET CELL 5 =================

[19:49:25 04/06/2026] BAT DAU CELL 5
GPU: Tesla T4
VRAM dau: {"free_GB": 4.621, "alloc_GB": 9.67, "reserved_GB": 9.723, "peak_GB": 9.702}
[19:49:29 04/06/2026] RUN STATIC_FAST_256 | prompt_tokens=99 | max_new=256
[19:49:36 04/06/2026] STATIC_FAST_256 | warmup 01 | 256 tok / 7.2176s = 35.47 tok/s
[19:49:43 04/06/2026] STATIC_FAST_256 | measure 02 | 256 tok / 7.1818s = 35.65 tok/s
[19:49:50 04/06/2026] STATIC_FAST_256 | measure 03 | 256 tok / 7.0267s = 36.43 tok/s
[19:49:57 04/06/2026] STATIC_FAST_256 | measure 04 | 256 tok / 7.1928s = 35.59 tok/s
[19:50:04 04/06/2026] STATIC_FAST_256 | measure 05 | 256 tok / 7.0864s = 36.13 tok/s
[19:50:12 04/06/2026] STATIC_FAST_256 | measure 06 | 256 tok / 7.2309s = 35.40 tok/s
[19:50:12 04/06/2026] RUN STATIC_FAST_384 | prompt_tokens=99 | max_new=384
[19:50:23 04/06/2026] STATIC_FAST_384 | warmup 01 | 384 tok / 11.0796s = 34.66 tok/s
[19:50:34 04/06/2026] STATIC_FAST_384 | measure 02 | 384 tok / 11.0444s = 34.77 tok/s
[19:50:45 04/06/2026] STATIC_FAS

W0604 19:50:56.273000 6590 torch/_inductor/cudagraph_utils.py:343] [__cudagraphs] CUDAGraph supports dynamic shapes by recording a new graph for each distinct input size. Recording too many CUDAGraphs may lead to extra overhead. We have observed 9 distinct sizes. Please consider the following options for better performance: a) padding inputs to a few fixed number of shapes; or b) set torch._inductor.config.triton.cudagraph_skip_dynamic_graphs=True. Set torch._inductor.config.triton.cudagraph_dynamic_shape_warn_limit=None to silence this warning.


[19:51:11 04/06/2026] STATIC_STOP_MARKER_512 | warmup 01 | 512 tok / 14.9527s = 34.24 tok/s
[19:51:25 04/06/2026] STATIC_STOP_MARKER_512 | measure 02 | 512 tok / 14.7268s = 34.77 tok/s
[19:51:40 04/06/2026] STATIC_STOP_MARKER_512 | measure 03 | 512 tok / 14.7309s = 34.76 tok/s
[19:51:55 04/06/2026] STATIC_STOP_MARKER_512 | measure 04 | 512 tok / 14.7231s = 34.78 tok/s
[19:51:55 04/06/2026] ====================================================================================================
[19:51:55 04/06/2026] OUTPUT STATIC_STOP_MARKER_512
import torch
import numpy as np
import random

def fuzz_searchsorted(data, target, seed=42):
    random.seed(seed)
    torch.manual_seed(seed)
    results = {"PASS": 0, "EXCEPTION": 0, "SUSPICIOUS": 0}
    n_tests = 0
    
    def run_case(data_type, data_content):
        nonlocal n_tests
        n_tests += 1
        print(f"\n--- Running Case: {data_type} ---")
        
        try:
            if data_type == "sorted_1d":
                # Oracle 

In [ ]:
# ================= CELL 6: VAT T4 BANG BATCH + FIXED BUCKET =================
import gc, time, json, traceback
from datetime import datetime
import torch
import pandas as pd

assert "model" in globals()
assert "processor" in globals()

tok = getattr(processor, "tokenizer", processor)

def now():
    return datetime.now().strftime("%H:%M:%S %d/%m/%Y")

def log(x=""):
    print(f"[{now()}] {x}", flush=True)

def sync():
    torch.cuda.synchronize()

def vram():
    free_b, total_b = torch.cuda.mem_get_info()
    return {
        "free_GB": round(free_b / 1024**3, 3),
        "alloc_GB": round(torch.cuda.memory_allocated() / 1024**3, 3),
        "reserved_GB": round(torch.cuda.memory_reserved() / 1024**3, 3),
        "peak_GB": round(torch.cuda.max_memory_allocated() / 1024**3, 3),
    }

PAD_ID = getattr(tok, "pad_token_id", None)
EOS_ID = getattr(tok, "eos_token_id", None)

if PAD_ID is None and EOS_ID is not None:
    tok.pad_token = tok.eos_token
    PAD_ID = tok.pad_token_id

try:
    tok.padding_side = "left"
except Exception:
    pass

model.eval()
model.config.use_cache = True
for p in model.parameters():
    p.requires_grad_(False)

COMMON_GEN = dict(
    do_sample=False,
    use_cache=True,
    cache_implementation="static",
    pad_token_id=PAD_ID if PAD_ID is not None else EOS_ID,
    eos_token_id=EOS_ID,
    return_dict_in_generate=False,
)

SYSTEM = "Code Python, no markdown."
BUCKETS = [64, 96, 128, 160, 192, 256, 320, 384, 512, 768, 1024]

def chat_text(prompt, system=SYSTEM):
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": prompt}]
    try:
        return processor.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False
        )
    except TypeError:
        return processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def pick_bucket(n):
    for b in BUCKETS:
        if n <= b:
            return b
    return ((n + 127) // 128) * 128

def encode_bucket(prompts, bucket=None):
    texts = [chat_text(p) for p in prompts]
    x = processor(text=texts, return_tensors="pt", padding=True)
    raw_len = int(x["input_ids"].shape[-1])
    bucket = pick_bucket(raw_len) if bucket is None else max(bucket, raw_len)
    pad_len = bucket - raw_len

    if pad_len > 0:
        pad_id = PAD_ID if PAD_ID is not None else (EOS_ID if EOS_ID is not None else 0)
        for k, v in list(x.items()):
            if torch.is_tensor(v) and v.ndim == 2 and v.shape[1] == raw_len:
                val = 0 if k == "attention_mask" else pad_id
                pad = torch.full((v.shape[0], pad_len), val, dtype=v.dtype)
                x[k] = torch.cat([pad, v], dim=1)

    return x.to("cuda:0"), bucket, raw_len

@torch.inference_mode()
def generate_x(x, max_new):
    kw = dict(x)
    kw.update(COMMON_GEN)
    kw["max_new_tokens"] = max_new
    return model.generate(**kw)

def make_prompts(n):
    return [
        f"Code Python fuzz torch.searchsorted seed={42+i}. Include PASS EXCEPTION SUSPICIOUS stats."
        for i in range(n)
    ]

@torch.inference_mode()
def bench_batch(batch_size, max_new=192):
    prompts = make_prompts(batch_size)
    x, bucket, raw_len = encode_bucket(prompts)

    # warmup same shape
    _ = generate_x(x, max_new)
    sync()

    sync()
    t0 = time.perf_counter()
    out = generate_x(x, max_new)
    sync()
    dt = time.perf_counter() - t0

    real_tokens = 0
    for i in range(out.shape[0]):
        tail = out[i, bucket:]
        mask = torch.ones_like(tail, dtype=torch.bool)
        if PAD_ID is not None:
            mask &= tail != PAD_ID
        if EOS_ID is not None:
            mask &= tail != EOS_ID
        real_tokens += int(mask.sum().item())

    sample = tok.decode(out[0, bucket:], skip_special_tokens=True)

    return {
        "batch": batch_size,
        "raw_input_len": raw_len,
        "bucket_len": bucket,
        "max_new": max_new,
        "real_tokens": real_tokens,
        "sec": round(dt, 4),
        "tok_s_total": round(real_tokens / max(dt, 1e-9), 3),
        "tok_s_per_prompt": round(real_tokens / max(dt, 1e-9) / batch_size, 3),
        "vram_alloc_GB": vram()["alloc_GB"],
        "vram_peak_GB": vram()["peak_GB"],
        "sample": sample[:500],
    }

log("BAT DAU CELL 6")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM dau:", json.dumps(vram(), ensure_ascii=False))

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

rows = []
for bs in [1, 2, 4, 8, 12, 16]:
    try:
        log(f"TEST BATCH {bs}")
        row = bench_batch(bs, max_new=192)
        rows.append(row)
        log(f"BATCH {bs}: total={row['tok_s_total']} tok/s | per_prompt={row['tok_s_per_prompt']} tok/s")
    except RuntimeError as e:
        msg = str(e).lower()
        log(f"BATCH {bs} LOI: {str(e)[:500]}")
        if "out of memory" in msg or "cuda" in msg:
            torch.cuda.empty_cache()
            break
    except Exception:
        log(f"BATCH {bs} LOI:")
        print(traceback.format_exc()[:1500])
        break

df = pd.DataFrame([{k:v for k,v in r.items() if k != "sample"} for r in rows])
log("=" * 100)
log("BANG BATCH SWEEP")
print(df.to_string(index=False))

if rows:
    best = max(rows, key=lambda r: r["tok_s_total"])
    BEST_BATCH = int(best["batch"])
    log("=" * 100)
    log(f"BEST_BATCH = {BEST_BATCH} | total={best['tok_s_total']} tok/s | per_prompt={best['tok_s_per_prompt']} tok/s")
    log("SAMPLE OUTPUT BEST")
    print(best["sample"], flush=True)

@torch.inference_mode()
def t4_fast(prompt, max_new_tokens=256):
    x, bucket, raw_len = encode_bucket([prompt])
    out = generate_x(x, max_new_tokens)
    return tok.decode(out[0, bucket:], skip_special_tokens=True)

@torch.inference_mode()
def t4_batch(prompts, max_new_tokens=192):
    x, bucket, raw_len = encode_bucket(prompts)
    out = generate_x(x, max_new_tokens)
    return [tok.decode(out[i, bucket:], skip_special_tokens=True) for i in range(out.shape[0])]

print("VRAM cuoi:", json.dumps(vram(), ensure_ascii=False))
log("KET LUAN: neu batch lon tang tok_s_total thi dung t4_batch; neu khong thi chot batch4/static.")
log("HOAN TAT CELL 6")
# ================= HET CELL 6 =================

[19:55:36 04/06/2026] BAT DAU CELL 6
GPU: Tesla T4
VRAM dau: {"free_GB": 4.594, "alloc_GB": 9.67, "reserved_GB": 9.723, "peak_GB": 9.702}
[19:55:41 04/06/2026] TEST BATCH 1
[19:55:51 04/06/2026] BATCH 1: total=0.0 tok/s | per_prompt=0.0 tok/s
[19:55:51 04/06/2026] TEST BATCH 2
[19:56:02 04/06/2026] BATCH 2: total=0.0 tok/s | per_prompt=0.0 tok/s
[19:56:02 04/06/2026] TEST BATCH 4
[19:56:13 04/06/2026] BATCH 4: total=0.0 tok/s | per_prompt=0.0 tok/s
[19:56:13 04/06/2026] TEST BATCH 8


W0604 19:56:13.512000 6590 torch/_inductor/cudagraph_utils.py:343] [__cudagraphs] CUDAGraph supports dynamic shapes by recording a new graph for each distinct input size. Recording too many CUDAGraphs may lead to extra overhead. We have observed 9 distinct sizes. Please consider the following options for better performance: a) padding inputs to a few fixed number of shapes; or b) set torch._inductor.config.triton.cudagraph_skip_dynamic_graphs=True. Set torch._inductor.config.triton.cudagraph_dynamic_shape_warn_limit=None to silence this warning.


[19:56:24 04/06/2026] BATCH 8: total=0.0 tok/s | per_prompt=0.0 tok/s
[19:56:24 04/06/2026] TEST BATCH 12
[19:56:36 04/06/2026] BATCH 12: total=0.0 tok/s | per_prompt=0.0 tok/s
[19:56:36 04/06/2026] TEST BATCH 16
[19:56:47 04/06/2026] BATCH 16: total=0.0 tok/s | per_prompt=0.0 tok/s
[19:56:47 04/06/2026] ====================================================================================================
[19:56:48 04/06/2026] BANG BATCH SWEEP
 batch  raw_input_len  bucket_len  max_new  real_tokens    sec  tok_s_total  tok_s_per_prompt  vram_alloc_GB  vram_peak_GB
     1             41          64      192            0 5.3634          0.0               0.0          9.667         9.688
     2             41          64      192            0 5.4087          0.0               0.0          9.671         9.693
     4             41          64      192            0 5.3763          0.0               0.0          9.680         9.711
     8             41          64      192            0 5.4972

In [ ]:
# ================= CELL 7: BATCH SWEEP DUNG, STATIC VS DYNAMIC =================
import gc, time, json, statistics, traceback
from datetime import datetime
import torch
import pandas as pd

assert "model" in globals()
assert "processor" in globals()

tok = getattr(processor, "tokenizer", processor)

def now():
    return datetime.now().strftime("%H:%M:%S %d/%m/%Y")

def log(x=""):
    print(f"[{now()}] {x}", flush=True)

def sync():
    torch.cuda.synchronize()

def vram():
    free_b, total_b = torch.cuda.mem_get_info()
    return {
        "free_GB": round(free_b / 1024**3, 3),
        "alloc_GB": round(torch.cuda.memory_allocated() / 1024**3, 3),
        "reserved_GB": round(torch.cuda.memory_reserved() / 1024**3, 3),
        "peak_GB": round(torch.cuda.max_memory_allocated() / 1024**3, 3),
    }

PAD_ID = getattr(tok, "pad_token_id", None)
EOS_ID = getattr(tok, "eos_token_id", None)

if PAD_ID is None and EOS_ID is not None:
    try:
        tok.pad_token = tok.eos_token
        PAD_ID = tok.pad_token_id
    except Exception:
        PAD_ID = EOS_ID

try:
    tok.padding_side = "left"
except Exception:
    pass

model.eval()
model.config.use_cache = True
for p in model.parameters():
    p.requires_grad_(False)

try:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

SYSTEM = "Code Python only. No markdown."

def chat_text(prompt):
    msgs = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": prompt},
    ]
    try:
        return processor.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        return processor.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=True,
        )

def make_prompt(seed):
    return (
        f"Write a complete Python fuzz script for torch.searchsorted. "
        f"Use seed={seed}. Include sorted, unsorted, duplicate, nan, inf, empty, "
        f"int, float, and 2d cases. Print PASS, EXCEPTION, SUSPICIOUS, and final stats."
    )

def encode_batch(prompts):
    texts = [chat_text(p) for p in prompts]
    return processor(text=texts, return_tensors="pt", padding=True).to("cuda:0")

@torch.inference_mode()
def generate_batch(x, max_new, cache_impl):
    kw = dict(x)
    kw.update(dict(
        max_new_tokens=max_new,
        min_new_tokens=max_new,
        do_sample=False,
        use_cache=True,
        pad_token_id=PAD_ID if PAD_ID is not None else EOS_ID,
        eos_token_id=EOS_ID,
        return_dict_in_generate=False,
    ))
    if cache_impl is not None:
        kw["cache_implementation"] = cache_impl
    return model.generate(**kw)

@torch.inference_mode()
def bench(batch_size, cache_impl="static", max_new=128, repeats=2):
    prompts = [make_prompt(42 + i) for i in range(batch_size)]
    x = encode_batch(prompts)
    input_len = int(x["input_ids"].shape[-1])

    log(f"WARMUP cache={cache_impl} batch={batch_size} input_len={input_len}")
    out = generate_batch(x, max_new, cache_impl)
    sync()

    speeds = []
    last = None

    for i in range(repeats):
        sync()
        t0 = time.perf_counter()
        out = generate_batch(x, max_new, cache_impl)
        sync()
        dt = time.perf_counter() - t0

        new_per_prompt = int(out.shape[-1] - input_len)
        step_tokens = new_per_prompt * batch_size
        spd = step_tokens / max(dt, 1e-9)
        speeds.append(spd)
        last = out

        log(
            f"MEASURE cache={cache_impl} batch={batch_size} rep={i+1} | "
            f"{step_tokens} tok / {dt:.4f}s = {spd:.2f} tok/s"
        )

    sample = tok.decode(last[0, input_len:], skip_special_tokens=True)

    return {
        "cache": str(cache_impl),
        "batch": batch_size,
        "input_len": input_len,
        "max_new": max_new,
        "tok_total": batch_size * max_new,
        "median_tok_s_total": round(statistics.median(speeds), 3),
        "mean_tok_s_total": round(statistics.mean(speeds), 3),
        "tok_s_per_prompt": round(statistics.median(speeds) / batch_size, 3),
        "vram_alloc_GB": vram()["alloc_GB"],
        "vram_peak_GB": vram()["peak_GB"],
        "sample": sample[:700],
    }

log("BAT DAU CELL 7")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM dau:", json.dumps(vram(), ensure_ascii=False))

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

rows = []
batch_list = [1, 2, 4, 8, 12, 16, 24, 32]
cache_list = ["static", "dynamic"]

for cache_impl in cache_list:
    for bs in batch_list:
        try:
            log(f"TEST cache={cache_impl} batch={bs}")
            row = bench(bs, cache_impl=cache_impl, max_new=128, repeats=2)
            rows.append(row)
            log(
                f"OK cache={cache_impl} batch={bs}: "
                f"total={row['median_tok_s_total']} tok/s | "
                f"per_prompt={row['tok_s_per_prompt']} tok/s"
            )
        except RuntimeError as e:
            log(f"LOI cache={cache_impl} batch={bs}: {str(e)[:700]}")
            if "out of memory" in str(e).lower():
                torch.cuda.empty_cache()
                break
        except Exception:
            log(f"LOI cache={cache_impl} batch={bs}:")
            print(traceback.format_exc()[:1500])
            break

df = pd.DataFrame([{k: v for k, v in r.items() if k != "sample"} for r in rows])
log("=" * 100)
log("BANG CELL 7")
print(df.sort_values("median_tok_s_total", ascending=False).to_string(index=False))

if rows:
    best = max(rows, key=lambda r: r["median_tok_s_total"])
    BEST_BATCH = int(best["batch"])
    BEST_CACHE = None if best["cache"] == "None" else best["cache"]

    log("=" * 100)
    log(
        f"BEST: cache={BEST_CACHE} batch={BEST_BATCH} "
        f"total={best['median_tok_s_total']} tok/s "
        f"per_prompt={best['tok_s_per_prompt']} tok/s"
    )
    log("SAMPLE BEST")
    print(best["sample"], flush=True)

    @torch.inference_mode()
    def t4_batch(prompts, max_new_tokens=256):
        x = encode_batch(prompts)
        input_len = int(x["input_ids"].shape[-1])
        out = generate_batch(x, max_new_tokens, BEST_CACHE)
        return [tok.decode(out[i, input_len:], skip_special_tokens=True) for i in range(out.shape[0])]

    @torch.inference_mode()
    def t4_fast(prompt, max_new_tokens=384):
        return t4_batch([prompt], max_new_tokens=max_new_tokens)[0]

print("VRAM cuoi:", json.dumps(vram(), ensure_ascii=False))
log("KET LUAN: neu batch8/12/16 tang tong tok/s ro thi dung t4_batch; neu khong, T4 da het bai trong Transformers.")
log("HOAN TAT CELL 7")
# ================= HET CELL 7 =================

[20:01:50 04/06/2026] BAT DAU CELL 7
GPU: Tesla T4
VRAM dau: {"free_GB": 4.359, "alloc_GB": 9.733, "reserved_GB": 9.928, "peak_GB": 9.855}
[20:01:54 04/06/2026] TEST cache=static batch=1
[20:01:54 04/06/2026] WARMUP cache=static batch=1 input_len=77
[20:02:02 04/06/2026] MEASURE cache=static batch=1 rep=1 | 128 tok / 3.5951s = 35.60 tok/s
[20:02:06 04/06/2026] MEASURE cache=static batch=1 rep=2 | 128 tok / 3.6149s = 35.41 tok/s
[20:02:06 04/06/2026] OK cache=static batch=1: total=35.507 tok/s | per_prompt=35.507 tok/s
[20:02:06 04/06/2026] TEST cache=static batch=2
[20:02:06 04/06/2026] WARMUP cache=static batch=2 input_len=77
[20:02:13 04/06/2026] MEASURE cache=static batch=2 rep=1 | 256 tok / 3.5282s = 72.56 tok/s
[20:02:17 04/06/2026] MEASURE cache=static batch=2 rep=2 | 256 tok / 3.6869s = 69.43 tok/s
[20:02:17 04/06/2026] OK cache=static batch=2: total=70.996 tok/s | per_prompt=35.498 tok/s
[20:02:17 04/06/2026] TEST cache=static batch=4
[20:02:17 04/06/2026] WARMUP cache=static b

In [ ]:
# ================= CELL 8: TIM DIEM GAY BATCH LON STATIC ONLY =================
import gc, time, json, traceback
import torch
import pandas as pd

assert "model" in globals()
assert "processor" in globals()
assert "encode_batch" in globals()
assert "generate_batch" in globals()

def sync():
    torch.cuda.synchronize()

def vram():
    free_b, total_b = torch.cuda.mem_get_info()
    return {
        "free_GB": round(free_b / 1024**3, 3),
        "alloc_GB": round(torch.cuda.memory_allocated() / 1024**3, 3),
        "reserved_GB": round(torch.cuda.memory_reserved() / 1024**3, 3),
        "peak_GB": round(torch.cuda.max_memory_allocated() / 1024**3, 3),
    }

def log(x):
    from datetime import datetime
    print(f"[{datetime.now().strftime('%H:%M:%S %d/%m/%Y')}] {x}", flush=True)

def make_prompt(seed):
    return (
        f"Write Python fuzz script for torch.searchsorted seed={seed}. "
        f"Include sorted unsorted duplicate nan inf empty int float 2d. "
        f"Print PASS EXCEPTION SUSPICIOUS stats. Code only."
    )

@torch.inference_mode()
def bench_static(bs, max_new):
    prompts = [make_prompt(1000 + i) for i in range(bs)]
    x = encode_batch(prompts)
    input_len = int(x["input_ids"].shape[-1])

    _ = generate_batch(x, max_new, "static")
    sync()

    sync()
    t0 = time.perf_counter()
    out = generate_batch(x, max_new, "static")
    sync()
    dt = time.perf_counter() - t0

    total_tok = bs * int(out.shape[-1] - input_len)

    return {
        "batch": bs,
        "input_len": input_len,
        "max_new": max_new,
        "total_tok": total_tok,
        "sec": round(dt, 4),
        "tok_s_total": round(total_tok / dt, 3),
        "tok_s_per_prompt": round(total_tok / dt / bs, 3),
        "vram_alloc_GB": vram()["alloc_GB"],
        "vram_peak_GB": vram()["peak_GB"],
    }

log("BAT DAU CELL 8")
print("VRAM dau:", json.dumps(vram(), ensure_ascii=False))

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

rows = []
for max_new in [128, 192, 256]:
    for bs in [32, 40, 48, 56, 64]:
        try:
            log(f"TEST static batch={bs} max_new={max_new}")
            row = bench_static(bs, max_new)
            rows.append(row)
            log(f"OK: {row['tok_s_total']} tok/s total | {row['tok_s_per_prompt']} tok/s per prompt")
        except RuntimeError as e:
            log(f"OOM/LOI batch={bs} max_new={max_new}: {str(e)[:500]}")
            torch.cuda.empty_cache()
            break
        except Exception:
            log(f"LOI batch={bs} max_new={max_new}")
            print(traceback.format_exc()[:1200])
            break

df = pd.DataFrame(rows)
log("=" * 100)
log("BANG CELL 8")
print(df.sort_values("tok_s_total", ascending=False).to_string(index=False))

if len(rows):
    best = max(rows, key=lambda r: r["tok_s_total"])
    log("=" * 100)
    log(
        f"CHOT THROUGHPUT: batch={best['batch']} max_new={best['max_new']} "
        f"total={best['tok_s_total']} tok/s per_prompt={best['tok_s_per_prompt']} tok/s"
    )

print("VRAM cuoi:", json.dumps(vram(), ensure_ascii=False))
log("HOAN TAT CELL 8")
# ================= HET CELL 8 =================

[20:11:10 04/06/2026] BAT DAU CELL 8
VRAM dau: {"free_GB": 1.935, "alloc_GB": 9.775, "reserved_GB": 12.309, "peak_GB": 10.276}
[20:11:12 04/06/2026] TEST static batch=32 max_new=128
[20:11:23 04/06/2026] OK: 770.736 tok/s total | 24.085 tok/s per prompt
[20:11:23 04/06/2026] TEST static batch=40 max_new=128
[20:11:34 04/06/2026] OK: 896.893 tok/s total | 22.422 tok/s per prompt
[20:11:34 04/06/2026] TEST static batch=48 max_new=128
[20:11:47 04/06/2026] OK: 971.741 tok/s total | 20.245 tok/s per prompt
[20:11:47 04/06/2026] TEST static batch=56 max_new=128
[20:12:00 04/06/2026] OK: 1115.725 tok/s total | 19.924 tok/s per prompt
[20:12:00 04/06/2026] TEST static batch=64 max_new=128
[20:12:14 04/06/2026] OK: 1198.185 tok/s total | 18.722 tok/s per prompt
[20:12:14 04/06/2026] TEST static batch=32 max_new=192
[20:12:30 04/06/2026] OK: 791.634 tok/s total | 24.739 tok/s per prompt
[20:12:30 04/06/2026] TEST static batch=40 max_new=192
[20:12:46 04/06/2026] OK: 925.124 tok/s total | 23.128

In [ ]:
# ================= CELL 9: PUSH BATCH DEN OOM / PLATEAU =================
import gc, time, json, traceback
import torch
import pandas as pd

assert "encode_batch" in globals()
assert "generate_batch" in globals()

def log(x):
    from datetime import datetime
    print(f"[{datetime.now().strftime('%H:%M:%S %d/%m/%Y')}] {x}", flush=True)

def sync():
    torch.cuda.synchronize()

def vram():
    free_b, total_b = torch.cuda.mem_get_info()
    return {
        "free_GB": round(free_b / 1024**3, 3),
        "alloc_GB": round(torch.cuda.memory_allocated() / 1024**3, 3),
        "reserved_GB": round(torch.cuda.memory_reserved() / 1024**3, 3),
        "peak_GB": round(torch.cuda.max_memory_allocated() / 1024**3, 3),
    }

def make_prompt(seed):
    return (
        f"Write Python fuzz script for torch.searchsorted seed={seed}. "
        f"Cases: sorted unsorted duplicate nan inf empty int float 2d. "
        f"Print PASS EXCEPTION SUSPICIOUS final stats. Code only."
    )

@torch.inference_mode()
def bench(bs, max_new):
    prompts = [make_prompt(2000 + i) for i in range(bs)]
    x = encode_batch(prompts)
    input_len = int(x["input_ids"].shape[-1])

    _ = generate_batch(x, max_new, "static")
    sync()

    sync()
    t0 = time.perf_counter()
    out = generate_batch(x, max_new, "static")
    sync()
    dt = time.perf_counter() - t0

    total_tok = bs * int(out.shape[-1] - input_len)
    return {
        "batch": bs,
        "input_len": input_len,
        "max_new": max_new,
        "total_tok": total_tok,
        "sec": round(dt, 4),
        "tok_s_total": round(total_tok / dt, 3),
        "tok_s_per_prompt": round(total_tok / dt / bs, 3),
        "vram_alloc_GB": vram()["alloc_GB"],
        "vram_peak_GB": vram()["peak_GB"],
    }

log("BAT DAU CELL 9")
print("VRAM dau:", json.dumps(vram(), ensure_ascii=False))

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

rows = []
for max_new in [256, 384, 512]:
    for bs in [64, 80, 96, 112, 128, 160, 192]:
        try:
            log(f"TEST batch={bs} max_new={max_new}")
            row = bench(bs, max_new)
            rows.append(row)
            log(f"OK {row['tok_s_total']} tok/s | per_prompt {row['tok_s_per_prompt']}")
        except RuntimeError as e:
            log(f"OOM/LOI batch={bs} max_new={max_new}: {str(e)[:500]}")
            torch.cuda.empty_cache()
            break
        except Exception:
            log(f"LOI batch={bs} max_new={max_new}")
            print(traceback.format_exc()[:1200])
            break

df = pd.DataFrame(rows)
log("=" * 100)
log("BANG CELL 9")
print(df.sort_values("tok_s_total", ascending=False).to_string(index=False))

if len(rows):
    best = max(rows, key=lambda r: r["tok_s_total"])
    log("=" * 100)
    log(
        f"CHOT CUOI: batch={best['batch']} max_new={best['max_new']} "
        f"total={best['tok_s_total']} tok/s per_prompt={best['tok_s_per_prompt']} tok/s"
    )

print("VRAM cuoi:", json.dumps(vram(), ensure_ascii=False))
log("HOAN TAT CELL 9")
# ================= HET CELL 9 =================

[20:20:28 04/06/2026] BAT DAU CELL 9
VRAM dau: {"free_GB": 3.455, "alloc_GB": 10.012, "reserved_GB": 10.705, "peak_GB": 10.495}
[20:20:30 04/06/2026] TEST batch=64 max_new=256
[20:20:56 04/06/2026] OK 1275.892 tok/s | per_prompt 19.936
[20:20:56 04/06/2026] TEST batch=80 max_new=256
[20:21:26 04/06/2026] OK 1361.278 tok/s | per_prompt 17.016
[20:21:26 04/06/2026] TEST batch=96 max_new=256
[20:21:59 04/06/2026] OK 1521.187 tok/s | per_prompt 15.846
[20:21:59 04/06/2026] TEST batch=112 max_new=256
[20:22:38 04/06/2026] OK 1470.245 tok/s | per_prompt 13.127
[20:22:38 04/06/2026] TEST batch=128 max_new=256
[20:23:15 04/06/2026] OK 1771.986 tok/s | per_prompt 13.844
[20:23:15 04/06/2026] TEST batch=160 max_new=256
[20:24:01 04/06/2026] OK 1789.882 tok/s | per_prompt 11.187
[20:24:01 04/06/2026] TEST batch=192 max_new=256
[20:24:52 04/06/2026] OK 1952.057 tok/s | per_prompt 10.167
[20:24:52 04/06/2026] TEST batch=64 max_new=384
[20:25:31 04/06/2026] OK 1249.439 tok/s | per_prompt 19.522
[20:

In [ ]:
# ================= CELL 10: MICRO SWEEP QUANH BEST =================
import gc, time, json, torch, pandas as pd

assert "encode_batch" in globals()
assert "generate_batch" in globals()

def log(x):
    from datetime import datetime
    print(f"[{datetime.now().strftime('%H:%M:%S %d/%m/%Y')}] {x}", flush=True)

def sync():
    torch.cuda.synchronize()

def vram():
    free_b, total_b = torch.cuda.mem_get_info()
    return {
        "free_GB": round(free_b / 1024**3, 3),
        "alloc_GB": round(torch.cuda.memory_allocated() / 1024**3, 3),
        "reserved_GB": round(torch.cuda.memory_reserved() / 1024**3, 3),
        "peak_GB": round(torch.cuda.max_memory_allocated() / 1024**3, 3),
    }

def make_prompt(seed):
    return (
        f"Write Python fuzz script for torch.searchsorted seed={seed}. "
        f"Cases sorted unsorted duplicate nan inf empty int float 2d. "
        f"Print PASS EXCEPTION SUSPICIOUS stats. Code only."
    )

@torch.inference_mode()
def bench(bs, max_new=256):
    prompts = [make_prompt(3000 + i) for i in range(bs)]
    x = encode_batch(prompts)
    input_len = int(x["input_ids"].shape[-1])

    _ = generate_batch(x, max_new, "static")
    sync()

    sync()
    t0 = time.perf_counter()
    out = generate_batch(x, max_new, "static")
    sync()
    dt = time.perf_counter() - t0

    total_tok = bs * int(out.shape[-1] - input_len)
    return {
        "batch": bs,
        "max_new": max_new,
        "total_tok": total_tok,
        "sec": round(dt, 4),
        "tok_s_total": round(total_tok / dt, 3),
        "tok_s_per_prompt": round(total_tok / dt / bs, 3),
        "vram": vram(),
    }

log("BAT DAU CELL 10")
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
print("VRAM dau:", json.dumps(vram(), ensure_ascii=False))

rows = []
for bs in [192, 208, 224, 240, 256]:
    try:
        log(f"TEST batch={bs} max_new=256")
        row = bench(bs, 256)
        rows.append({k:v for k,v in row.items() if k != "vram"})
        log(f"OK {row['tok_s_total']} tok/s | per_prompt {row['tok_s_per_prompt']} | vram {json.dumps(row['vram'])}")
    except RuntimeError as e:
        log(f"OOM/LOI batch={bs}: {str(e)[:500]}")
        torch.cuda.empty_cache()
        break

df = pd.DataFrame(rows)
print(df.sort_values("tok_s_total", ascending=False).to_string(index=False))

if len(rows):
    best = max(rows, key=lambda r: r["tok_s_total"])
    log(f"CHOT THAT: batch={best['batch']} max_new=256 total={best['tok_s_total']} tok/s")

print("VRAM cuoi:", json.dumps(vram(), ensure_ascii=False))
log("HOAN TAT CELL 10")
# ================= HET CELL 10 =================

[20:44:44 04/06/2026] BAT DAU CELL 10
VRAM dau: {"free_GB": 2.267, "alloc_GB": 11.421, "reserved_GB": 11.727, "peak_GB": 11.421}
[20:44:48 04/06/2026] TEST batch=192 max_new=256
[20:45:43 04/06/2026] OK 1765.872 tok/s | per_prompt 9.197 | vram {"free_GB": 0.113, "alloc_GB": 11.422, "reserved_GB": 13.881, "peak_GB": 12.876}
[20:45:43 04/06/2026] TEST batch=208 max_new=256
[20:46:39 04/06/2026] OK 1912.937 tok/s | per_prompt 9.197 | vram {"free_GB": 0.539, "alloc_GB": 10.799, "reserved_GB": 13.449, "peak_GB": 12.876}
[20:46:39 04/06/2026] TEST batch=224 max_new=256
[20:47:37 04/06/2026] OK 1988.901 tok/s | per_prompt 8.879 | vram {"free_GB": 0.16, "alloc_GB": 10.887, "reserved_GB": 13.82, "peak_GB": 12.876}
[20:47:37 04/06/2026] TEST batch=240 max_new=256
[20:48:38 04/06/2026] OK 2048.578 tok/s | per_prompt 8.536 | vram {"free_GB": 0.701, "alloc_GB": 10.974, "reserved_GB": 13.273, "peak_GB": 12.876}
[20:48:38 04/06/2026] TEST batch=256 max_new=256
[20:49:41 04/06/2026] OK 2099.074 tok/s 

In [ ]:
# ================= CELL 11: NO-RESET CONTINUOUS BATCHING PROBE =================
import gc, time, json, traceback
from datetime import datetime
import torch
import pandas as pd

assert "model" in globals()
assert "processor" in globals()

tok = getattr(processor, "tokenizer", processor)

def log(x):
    print(f"[{datetime.now().strftime('%H:%M:%S %d/%m/%Y')}] {x}", flush=True)

def sync():
    torch.cuda.synchronize()

def vram():
    free_b, total_b = torch.cuda.mem_get_info()
    return {
        "free_GB": round(free_b / 1024**3, 3),
        "alloc_GB": round(torch.cuda.memory_allocated() / 1024**3, 3),
        "reserved_GB": round(torch.cuda.memory_reserved() / 1024**3, 3),
        "peak_GB": round(torch.cuda.max_memory_allocated() / 1024**3, 3),
    }

def chat_text(prompt):
    msgs = [
        {"role": "system", "content": "Code Python only. No markdown."},
        {"role": "user", "content": prompt},
    ]
    try:
        return processor.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False
        )
    except TypeError:
        return processor.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True
        )

def make_prompt(i):
    if i % 4 == 0:
        return f"Write short Python fuzz torch.searchsorted seed={5000+i}. PASS EXCEPTION SUSPICIOUS."
    if i % 4 == 1:
        return f"Write complete Python fuzz script for torch.searchsorted seed={5000+i}. Include sorted unsorted duplicate nan inf empty int float 2d. Stats."
    if i % 4 == 2:
        return f"Python unittest fuzz torch.searchsorted seed={5000+i}. Many edge cases. Code only."
    return f"Minimal torch.searchsorted fuzz script seed={5000+i}. Code only."

log("BAT DAU CELL 11")
print("Transformers:", __import__("transformers").__version__)
print("has model.generate_batch:", hasattr(model, "generate_batch"))
print("attn:", getattr(getattr(model, "config", None), "_attn_implementation", None))
print("VRAM dau:", json.dumps(vram(), ensure_ascii=False))

gc.collect()
torch.cuda.empty_cache()
try:
    torch.cuda.reset_peak_memory_stats()
except Exception:
    pass

# Clean compile-cache sau qua nhieu sweep shape. Khong reset kernel, chi de bot cache do benchmark.
try:
    torch.compiler.reset()
    log("torch.compiler.reset() OK")
except Exception as e:
    log(f"torch.compiler.reset() bo qua: {type(e).__name__}: {e}")

rows = []

# Baseline static batch cu, nhe hon cell 10.
if "encode_batch" in globals() and "generate_batch" in globals():
    try:
        prompts = [make_prompt(i) for i in range(128)]
        x = encode_batch(prompts)
        input_len = int(x["input_ids"].shape[-1])

        _ = generate_batch(x, 128, "static")
        sync()

        sync()
        t0 = time.perf_counter()
        out = generate_batch(x, 128, "static")
        sync()
        dt = time.perf_counter() - t0

        total_tok = 128 * int(out.shape[-1] - input_len)
        rows.append({
            "mode": "static_generate",
            "batch": 128,
            "max_new": 128,
            "total_tok": total_tok,
            "sec": round(dt, 4),
            "tok_s_total": round(total_tok / dt, 3),
            "tok_s_per_prompt": round(total_tok / dt / 128, 3),
            "vram_peak_GB": vram()["peak_GB"],
        })
        log(f"STATIC OK: {rows[-1]['tok_s_total']} tok/s")
    except Exception:
        log("STATIC baseline loi:")
        print(traceback.format_exc()[:1500])

# Continuous batching neu Transformers trong kernel co API.
try:
    from transformers.generation import GenerationConfig, ContinuousBatchingConfig

    if not hasattr(model, "generate_batch"):
        raise RuntimeError("model.generate_batch khong co trong Transformers ban nay")

    input_ids_list = [
        tok.encode(chat_text(make_prompt(i)), add_special_tokens=False)
        for i in range(128)
    ]

    gen_cfg = GenerationConfig(
        max_new_tokens=128,
        do_sample=False,
        eos_token_id=getattr(tok, "eos_token_id", None),
        pad_token_id=getattr(tok, "pad_token_id", None) or getattr(tok, "eos_token_id", None),
    )

    cb_trials = []
    for use_cuda_graph in [False, True]:
        try:
            cb_cfg = ContinuousBatchingConfig(
                max_memory_percent=0.80,
                block_size=256,
                use_cuda_graph=use_cuda_graph,
            )
        except TypeError:
            cb_cfg = ContinuousBatchingConfig(max_memory_percent=0.80, block_size=256)

        # warmup
        _ = model.generate_batch(
            inputs=input_ids_list[:16],
            generation_config=gen_cfg,
            continuous_batching_config=cb_cfg,
        )

        sync()
        t0 = time.perf_counter()
        outputs = model.generate_batch(
            inputs=input_ids_list,
            generation_config=gen_cfg,
            continuous_batching_config=cb_cfg,
        )
        sync()
        dt = time.perf_counter() - t0

        total_tok = 0
        for o in outputs.values():
            toks = getattr(o, "generated_tokens", None)
            if toks is None:
                toks = getattr(o, "tokens", [])
            total_tok += len(toks)

        row = {
            "mode": f"continuous_batching_cuda_graph_{use_cuda_graph}",
            "batch": 128,
            "max_new": 128,
            "total_tok": total_tok,
            "sec": round(dt, 4),
            "tok_s_total": round(total_tok / max(dt, 1e-9), 3),
            "tok_s_per_prompt": round(total_tok / max(dt, 1e-9) / 128, 3),
            "vram_peak_GB": vram()["peak_GB"],
        }
        rows.append(row)
        cb_trials.append(row)
        log(f"CB use_cuda_graph={use_cuda_graph}: {row['tok_s_total']} tok/s")

except Exception:
    log("Continuous batching khong chay duoc trong kernel hien tai:")
    print(traceback.format_exc()[:1800])

df = pd.DataFrame(rows)
log("=" * 100)
log("BANG CELL 11")
if len(df):
    print(df.sort_values("tok_s_total", ascending=False).to_string(index=False))
else:
    print("Khong co ket qua benchmark.")

print("VRAM cuoi:", json.dumps(vram(), ensure_ascii=False))
log("KET LUAN: neu continuous batching khong vuot static, chot static batch=256 max_new=256; neu vuot thi dung model.generate_batch cho workload variable-length.")
log("HOAN TAT CELL 11")
# ================= HET CELL 11 =================

[20:53:33 04/06/2026] BAT DAU CELL 11
Transformers: 5.10.2
has model.generate_batch: True
attn: sdpa
VRAM dau: {"free_GB": 0.478, "alloc_GB": 11.06, "reserved_GB": 13.49, "peak_GB": 12.994}
[20:53:37 04/06/2026] torch.compiler.reset() OK
[20:56:12 04/06/2026] STATIC OK: 1767.875 tok/s


Solving 16 requests:   0%|          | 0/16 [00:00<?, ?request/s]2026-06-04 20:56:12,733 - ContinuousBatchingLogger - WARNING - Warming up for continuous batching...


[20:56:12 04/06/2026] Continuous batching khong chay duoc trong kernel hien tai:


Solving 16 requests:   0%|          | 0/16 [00:00<?, ?request/s]

Traceback (most recent call last):
  File "/tmp/ipykernel_6590/1053935109.py", line 135, in <cell line: 0>
    _ = model.generate_batch(
        ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/_contextlib.py", line 124, in decorate_context
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/generation/continuous_batching/continuous_api.py", line 1204, in generate_batch
    with manager_cm as manager, logging_cm, pbar_cm as pbar:
         ^^^^^^^^^^
  File "/usr/lib/python3.12/contextlib.py", line 137, in __enter__
    return next(self.gen)
           ^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/_contextlib.py", line 40, in generator_context
    response = gen.send(None)
               ^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/generation/continuous_batching/continuous_api.py", line 1121, in continuous_batching_contex

In [ ]:
# ================= CELL 12: PATCH CONFIG CHO CONTINUOUS BATCHING =================
import gc, time, json, traceback
from datetime import datetime
import torch
import pandas as pd
from transformers.generation import GenerationConfig, ContinuousBatchingConfig

assert "model" in globals()
assert "processor" in globals()

tok = getattr(processor, "tokenizer", processor)

def log(x):
    print(f"[{datetime.now().strftime('%H:%M:%S %d/%m/%Y')}] {x}", flush=True)

def sync():
    torch.cuda.synchronize()

def vram():
    free_b, total_b = torch.cuda.mem_get_info()
    return {
        "free_GB": round(free_b / 1024**3, 3),
        "alloc_GB": round(torch.cuda.memory_allocated() / 1024**3, 3),
        "reserved_GB": round(torch.cuda.memory_reserved() / 1024**3, 3),
        "peak_GB": round(torch.cuda.max_memory_allocated() / 1024**3, 3),
    }

def text_cfg(cfg):
    if hasattr(cfg, "get_text_config"):
        try:
            return cfg.get_text_config(decoder=True)
        except Exception:
            pass
    for name in ["text_config", "language_config", "decoder_config"]:
        v = getattr(cfg, name, None)
        if v is not None:
            return v
    return cfg

def patch_cfg(m):
    cfg = m.config
    tc = text_cfg(cfg)
    keys = [
        "num_attention_heads", "num_key_value_heads", "num_hidden_layers",
        "hidden_size", "head_dim", "max_position_embeddings",
        "sliding_window", "attention_chunk_size", "layer_types",
        "vocab_size", "bos_token_id", "eos_token_id", "pad_token_id",
    ]
    copied = []
    for k in keys:
        if (not hasattr(cfg, k) or getattr(cfg, k, None) is None) and hasattr(tc, k):
            setattr(cfg, k, getattr(tc, k))
            copied.append(k)
    if not hasattr(cfg, "num_key_value_heads") and hasattr(cfg, "num_attention_heads"):
        cfg.num_key_value_heads = cfg.num_attention_heads
        copied.append("num_key_value_heads=fallback")
    if not hasattr(cfg, "head_dim") and hasattr(cfg, "hidden_size") and hasattr(cfg, "num_attention_heads"):
        cfg.head_dim = cfg.hidden_size // cfg.num_attention_heads
        copied.append("head_dim=fallback")
    return copied

def chat_text(prompt):
    msgs = [
        {"role": "system", "content": "Code Python only. No markdown."},
        {"role": "user", "content": prompt},
    ]
    try:
        return processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        return processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def make_prompt(i):
    return (
        f"Write Python fuzz script for torch.searchsorted seed={7000+i}. "
        "Cases sorted unsorted duplicate nan inf empty int float 2d. "
        "Print PASS EXCEPTION SUSPICIOUS stats. Code only."
    )

def candidates():
    out = [("model", model)]
    for name in ["language_model", "text_model", "llm"]:
        obj = getattr(model, name, None)
        if obj is not None and id(obj) not in [id(x[1]) for x in out]:
            out.append((name, obj))
    return [(n, m) for n, m in out if hasattr(m, "generate_batch")]

@torch.inference_mode()
def bench_cb(name, m, bs=128, max_new=128):
    copied = patch_cfg(m)
    log(f"TRY {name} | patched={copied}")

    input_ids = [tok.encode(chat_text(make_prompt(i)), add_special_tokens=False) for i in range(bs)]
    gen_cfg = GenerationConfig(
        max_new_tokens=max_new,
        do_sample=False,
        eos_token_id=getattr(tok, "eos_token_id", None),
        pad_token_id=getattr(tok, "pad_token_id", None) or getattr(tok, "eos_token_id", None),
    )

    try:
        cb_cfg = ContinuousBatchingConfig(
            max_memory_percent=0.75,
            block_size=256,
            use_cuda_graph=False,
            max_blocks_per_request=0,
        )
    except TypeError:
        cb_cfg = ContinuousBatchingConfig(max_memory_percent=0.75, block_size=256)

    _ = m.generate_batch(inputs=input_ids[:16], generation_config=gen_cfg, continuous_batching_config=cb_cfg)
    sync()

    sync()
    t0 = time.perf_counter()
    outs = m.generate_batch(inputs=input_ids, generation_config=gen_cfg, continuous_batching_config=cb_cfg)
    sync()
    dt = time.perf_counter() - t0

    total_tok = sum(len(getattr(o, "generated_tokens", [])) for o in outs.values())
    return {
        "candidate": name,
        "batch": bs,
        "max_new": max_new,
        "total_tok": total_tok,
        "sec": round(dt, 4),
        "tok_s_total": round(total_tok / max(dt, 1e-9), 3),
        "tok_s_per_prompt": round(total_tok / max(dt, 1e-9) / bs, 3),
        "vram_peak_GB": vram()["peak_GB"],
    }

log("BAT DAU CELL 12")
print("VRAM dau:", json.dumps(vram(), ensure_ascii=False))
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

rows = []
for name, m in candidates():
    try:
        rows.append(bench_cb(name, m, bs=128, max_new=128))
    except Exception:
        log(f"{name} FAIL:")
        print(traceback.format_exc()[:1800])

log("=" * 100)
if rows:
    df = pd.DataFrame(rows).sort_values("tok_s_total", ascending=False)
    print(df.to_string(index=False))
else:
    print("Continuous batching van khong chay duoc.")

print("VRAM cuoi:", json.dumps(vram(), ensure_ascii=False))
log("NEU KHONG VUOT ~2100 TOK/S THI CHOT STATIC batch=256 max_new=256.")
log("HOAN TAT CELL 12")
# ================= HET CELL 12 =================

[20:59:16 04/06/2026] BAT DAU CELL 12
VRAM dau: {"free_GB": 2.166, "alloc_GB": 10.061, "reserved_GB": 11.801, "peak_GB": 11.06}
[20:59:18 04/06/2026] TRY model | patched=['num_attention_heads', 'num_key_value_heads', 'num_hidden_layers', 'hidden_size', 'head_dim', 'max_position_embeddings', 'sliding_window', 'layer_types', 'vocab_size', 'bos_token_id', 'pad_token_id']


Solving 16 requests:   0%|          | 0/16 [00:00<?, ?request/s]2026-06-04 20:59:18,983 - ContinuousBatchingLogger - WARNING - Warming up for continuous batching...
2026-06-04 20:59:19,011 - ContinuousBatchingLogger - WARNING - Warming up completed in 0.03s.
2026-06-04 20:59:19,047 - ContinuousBatchingLogger - ERROR - Error in generation loop: Inplace update to inference tensor outside InferenceMode is not allowed.You can make a clone to get a normal tensor before doing inplace update.See https://github.com/pytorch/rfcs/pull/17 for more details.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/transformers/generation/continuous_batching/continuous_api.py", line 881, in _run_generation_loop
    batch_processor = self._create_batch_processor()
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/generation/continuous_batching/continuous_api.py", line 948, in _create_batch_processor
    batch_pr

Returning results of generate_batch despite unexpected termination.


Solving 128 requests:   0%|          | 0/128 [00:00<?, ?request/s]2026-06-04 20:59:21,358 - ContinuousBatchingLogger - WARNING - Warming up for continuous batching...
2026-06-04 20:59:21,365 - ContinuousBatchingLogger - WARNING - Warming up completed in 0.01s.
2026-06-04 20:59:21,367 - ContinuousBatchingLogger - ERROR - Error in generation loop: Inplace update to inference tensor outside InferenceMode is not allowed.You can make a clone to get a normal tensor before doing inplace update.See https://github.com/pytorch/rfcs/pull/17 for more details.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/transformers/generation/continuous_batching/continuous_api.py", line 881, in _run_generation_loop
    batch_processor = self._create_batch_processor()
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/generation/continuous_batching/continuous_api.py", line 948, in _create_batch_processor
    batch_

Returning results of generate_batch despite unexpected termination.


2026-06-04 20:59:23,788 - ContinuousBatchingLogger - ERROR - Requests ['req_0', 'req_1', 'req_2', 'req_3', 'req_4', 'req_5', 'req_6', 'req_7', 'req_8', 'req_9', 'req_10', 'req_11', 'req_12', 'req_13', 'req_14', 'req_15', 'req_16', 'req_17', 'req_18', 'req_19', 'req_20', 'req_21', 'req_22', 'req_23', 'req_24', 'req_25', 'req_26', 'req_27', 'req_28', 'req_29', 'req_30', 'req_31', 'req_32', 'req_33', 'req_34', 'req_35', 'req_36', 'req_37', 'req_38', 'req_39', 'req_40', 'req_41', 'req_42', 'req_43', 'req_44', 'req_45', 'req_46', 'req_47', 'req_48', 'req_49', 'req_50', 'req_51', 'req_52', 'req_53', 'req_54', 'req_55', 'req_56', 'req_57', 'req_58', 'req_59', 'req_60', 'req_61', 'req_62', 'req_63', 'req_64', 'req_65', 'req_66', 'req_67', 'req_68', 'req_69', 'req_70', 'req_71', 'req_72', 'req_73', 'req_74', 'req_75', 'req_76', 'req_77', 'req_78', 'req_79', 'req_80', 'req_81', 'req_82', 'req_83', 'req_84', 'req_85', 'req_86', 'req_87', 'req_88', 'req_89', 'req_90', 'req_91', 'req_92', 'req_93',

[20:59:23 04/06/2026] ====================================================================================================
candidate  batch  max_new  total_tok   sec  tok_s_total  tok_s_per_prompt  vram_peak_GB
    model    128      128          0 2.439          0.0               0.0        12.456
VRAM cuoi: {"free_GB": 2.851, "alloc_GB": 10.761, "reserved_GB": 11.115, "peak_GB": 12.456}
[20:59:23 04/06/2026] NEU KHONG VUOT ~2100 TOK/S THI CHOT STATIC batch=256 max_new=256.
[20:59:23 04/06/2026] HOAN TAT CELL 12


In [ ]:
# ================= CELL 13: RETRY CONTINUOUS BATCHING, NO INFERENCE_MODE =================
import gc, time, json, traceback
from datetime import datetime
import torch
import pandas as pd
from transformers.generation import GenerationConfig, ContinuousBatchingConfig

assert "model" in globals()
assert "processor" in globals()

tok = getattr(processor, "tokenizer", processor)

def log(x):
    print(f"[{datetime.now().strftime('%H:%M:%S %d/%m/%Y')}] {x}", flush=True)

def sync():
    torch.cuda.synchronize()

def vram():
    free_b, total_b = torch.cuda.mem_get_info()
    return {
        "free_GB": round(free_b / 1024**3, 3),
        "alloc_GB": round(torch.cuda.memory_allocated() / 1024**3, 3),
        "reserved_GB": round(torch.cuda.memory_reserved() / 1024**3, 3),
        "peak_GB": round(torch.cuda.max_memory_allocated() / 1024**3, 3),
    }

def patch_cfg(m):
    cfg = m.config
    tc = None
    if hasattr(cfg, "get_text_config"):
        try:
            tc = cfg.get_text_config(decoder=True)
        except Exception:
            pass
    if tc is None:
        tc = getattr(cfg, "text_config", None) or getattr(cfg, "language_config", None) or cfg

    keys = [
        "num_attention_heads", "num_key_value_heads", "num_hidden_layers",
        "hidden_size", "head_dim", "max_position_embeddings",
        "sliding_window", "layer_types", "vocab_size",
        "bos_token_id", "eos_token_id", "pad_token_id",
    ]
    copied = []
    for k in keys:
        if (not hasattr(cfg, k) or getattr(cfg, k, None) is None) and hasattr(tc, k):
            setattr(cfg, k, getattr(tc, k))
            copied.append(k)

    if not hasattr(cfg, "num_key_value_heads") and hasattr(cfg, "num_attention_heads"):
        cfg.num_key_value_heads = cfg.num_attention_heads
        copied.append("num_key_value_heads=fallback")

    if not hasattr(cfg, "head_dim") and hasattr(cfg, "hidden_size") and hasattr(cfg, "num_attention_heads"):
        cfg.head_dim = cfg.hidden_size // cfg.num_attention_heads
        copied.append("head_dim=fallback")

    return copied

def chat_text(prompt):
    msgs = [
        {"role": "system", "content": "Code Python only. No markdown."},
        {"role": "user", "content": prompt},
    ]
    try:
        return processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        return processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def make_prompt(i):
    return (
        f"Write Python fuzz script for torch.searchsorted seed={8000+i}. "
        "Cases sorted unsorted duplicate nan inf empty int float 2d. "
        "Print PASS EXCEPTION SUSPICIOUS stats. Code only."
    )

def bench_cb(bs=128, max_new=128, use_cuda_graph=False):
    copied = patch_cfg(model)
    log(f"patched={copied}")

    input_ids = [tok.encode(chat_text(make_prompt(i)), add_special_tokens=False) for i in range(bs)]

    gen_cfg = GenerationConfig(
        max_new_tokens=max_new,
        do_sample=False,
        eos_token_id=getattr(tok, "eos_token_id", None),
        pad_token_id=getattr(tok, "pad_token_id", None) or getattr(tok, "eos_token_id", None),
    )

    cb_cfg = ContinuousBatchingConfig(
        max_memory_percent=0.75,
        block_size=256,
        use_cuda_graph=use_cuda_graph,
        max_blocks_per_request=0,
    )

    # IMPORTANT: no torch.inference_mode here.
    with torch.no_grad():
        _ = model.generate_batch(
            inputs=input_ids[:16],
            generation_config=gen_cfg,
            continuous_batching_config=cb_cfg,
        )
        sync()

        sync()
        t0 = time.perf_counter()
        outs = model.generate_batch(
            inputs=input_ids,
            generation_config=gen_cfg,
            continuous_batching_config=cb_cfg,
        )
        sync()
        dt = time.perf_counter() - t0

    total_tok = sum(len(getattr(o, "generated_tokens", [])) for o in outs.values())

    return {
        "mode": f"continuous_cuda_graph_{use_cuda_graph}",
        "batch": bs,
        "max_new": max_new,
        "total_tok": total_tok,
        "sec": round(dt, 4),
        "tok_s_total": round(total_tok / max(dt, 1e-9), 3),
        "tok_s_per_prompt": round(total_tok / max(dt, 1e-9) / bs, 3),
        "vram_peak_GB": vram()["peak_GB"],
    }

log("BAT DAU CELL 13")
print("VRAM dau:", json.dumps(vram(), ensure_ascii=False))

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

rows = []
for cg in [False, True]:
    try:
        log(f"TEST continuous batching use_cuda_graph={cg}")
        rows.append(bench_cb(bs=128, max_new=128, use_cuda_graph=cg))
        log(f"OK {rows[-1]['tok_s_total']} tok/s")
    except Exception:
        log(f"FAIL use_cuda_graph={cg}:")
        print(traceback.format_exc()[:1800])

log("=" * 100)
if rows:
    print(pd.DataFrame(rows).sort_values("tok_s_total", ascending=False).to_string(index=False))
else:
    print("Continuous batching van fail.")

print("VRAM cuoi:", json.dumps(vram(), ensure_ascii=False))
log("NEU CELL 13 FAIL HOAC KHONG VUOT STATIC, CHOT STATIC batch=256 max_new=256.")
log("HOAN TAT CELL 13")
# ================= HET CELL 13 =================

[21:01:30 04/06/2026] BAT DAU CELL 13
VRAM dau: {"free_GB": 2.851, "alloc_GB": 10.761, "reserved_GB": 11.115, "peak_GB": 12.456}
[21:01:32 04/06/2026] TEST continuous batching use_cuda_graph=False
[21:01:32 04/06/2026] patched=[]


Solving 16 requests:   0%|          | 0/16 [00:00<?, ?request/s]2026-06-04 21:01:32,296 - ContinuousBatchingLogger - WARNING - Warming up for continuous batching...
2026-06-04 21:01:32,310 - ContinuousBatchingLogger - WARNING - Warming up completed in 0.01s.
2026-06-04 21:01:32,387 - ContinuousBatchingLogger - ERROR - Error in generation loop: index_copy_(): Source/destination tensor must have same slice shapes. Destination slice shape: 1 256 at dimension 0 and source slice shape: 1 512 at dimension 0.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/transformers/generation/continuous_batching/continuous_api.py", line 898, in _run_generation_loop
    self._generation_step()
  File "/usr/local/lib/python3.12/dist-packages/transformers/generation/continuous_batching/continuous_api.py", line 941, in _generation_step
    self.batch_processor._generation_step(self.model)
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/_contextlib.py", line 124, 

Returning results of generate_batch despite unexpected termination.


2026-06-04 21:01:35,032 - ContinuousBatchingLogger - ERROR - Requests ['req_0', 'req_1', 'req_2', 'req_3', 'req_4', 'req_5', 'req_6', 'req_7', 'req_8', 'req_9', 'req_10', 'req_11', 'req_12', 'req_13', 'req_14', 'req_15'] not found in results.
Solving 128 requests:   0%|          | 0/128 [00:00<?, ?request/s]2026-06-04 21:01:35,041 - ContinuousBatchingLogger - WARNING - Warming up for continuous batching...
2026-06-04 21:01:35,048 - ContinuousBatchingLogger - WARNING - Warming up completed in 0.01s.
2026-06-04 21:01:35,074 - ContinuousBatchingLogger - ERROR - Error in generation loop: index_copy_(): Source/destination tensor must have same slice shapes. Destination slice shape: 1 256 at dimension 0 and source slice shape: 1 512 at dimension 0.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/transformers/generation/continuous_batching/continuous_api.py", line 898, in _run_generation_loop
    self._generation_step()
  File "/usr/local/lib/python3.12/dist

Returning results of generate_batch despite unexpected termination.


...

Returning results of generate_batch despite unexpected termination.


2026-06-04 21:01:40,007 - ContinuousBatchingLogger - ERROR - Requests ['req_0', 'req_1', 'req_2', 'req_3', 'req_4', 'req_5', 'req_6', 'req_7', 'req_8', 'req_9', 'req_10', 'req_11', 'req_12', 'req_13', 'req_14', 'req_15'] not found in results.
Solving 128 requests:   0%|          | 0/128 [00:00<?, ?request/s]2026-06-04 21:01:40,015 - ContinuousBatchingLogger - WARNING - Warming up for continuous batching...
2026-06-04 21:01:40,024 - ContinuousBatchingLogger - WARNING - Failed to warm up: no blocks allocated for num_requests = 1, num_q_tokens = 844, max_kv_read = 83380.
2026-06-04 21:01:40,025 - ContinuousBatchingLogger - WARNING - Warming up completed in 0.01s.
2026-06-04 21:01:40,054 - ContinuousBatchingLogger - ERROR - Error in generation loop: index_copy_(): Source/destination tensor must have same slice shapes. Destination slice shape: 1 256 at dimension 0 and source slice shape: 1 512 at dimension 0.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages

Returning results of generate_batch despite unexpected termination.


2026-06-04 21:01:42,460 - ContinuousBatchingLogger - ERROR - Requests ['req_0', 'req_1', 'req_2', 'req_3', 'req_4', 'req_5', 'req_6', 'req_7', 'req_8', 'req_9', 'req_10', 'req_11', 'req_12', 'req_13', 'req_14', 'req_15', 'req_16', 'req_17', 'req_18', 'req_19', 'req_20', 'req_21', 'req_22', 'req_23', 'req_24', 'req_25', 'req_26', 'req_27', 'req_28', 'req_29', 'req_30', 'req_31', 'req_32', 'req_33', 'req_34', 'req_35', 'req_36', 'req_37', 'req_38', 'req_39', 'req_40', 'req_41', 'req_42', 'req_43', 'req_44', 'req_45', 'req_46', 'req_47', 'req_48', 'req_49', 'req_50', 'req_51', 'req_52', 'req_53', 'req_54', 'req_55', 'req_56', 'req_57', 'req_58', 'req_59', 'req_60', 'req_61', 'req_62', 'req_63', 'req_64', 'req_65', 'req_66', 'req_67', 'req_68', 'req_69', 'req_70', 'req_71', 'req_72', 'req_73', 'req_74', 'req_75', 'req_76', 'req_77', 'req_78', 'req_79', 'req_80', 'req_81', 'req_82', 'req_83', 'req_84', 'req_85', 'req_86', 'req_87', 'req_88', 'req_89', 'req_90', 'req_91', 'req_92', 'req_93',

[21:01:42 04/06/2026] OK 0.0 tok/s
[21:01:42 04/06/2026] ====================================================================================================
                       mode  batch  max_new  total_tok    sec  tok_s_total  tok_s_per_prompt  vram_peak_GB
continuous_cuda_graph_False    128      128          0 2.4634          0.0               0.0         12.56
 continuous_cuda_graph_True    128      128          0 2.4524          0.0               0.0         12.56
VRAM cuoi: {"free_GB": 2.355, "alloc_GB": 10.977, "reserved_GB": 11.611, "peak_GB": 12.56}
[21:01:42 04/06/2026] NEU CELL 13 FAIL HOAC KHONG VUOT STATIC, CHOT STATIC batch=256 max_new=256.
[21:01:42 04/06/2026] HOAN TAT CELL 13


In [ ]:
# ================= CELL 14: LAST NO-RESET PROBE, CB FIX + STATIC >256 =================
import gc, time, json, traceback
from datetime import datetime
import torch
import pandas as pd
from transformers.generation import GenerationConfig, ContinuousBatchingConfig

assert "model" in globals()
assert "processor" in globals()

tok = getattr(processor, "tokenizer", processor)

def log(x):
    print(f"[{datetime.now().strftime('%H:%M:%S %d/%m/%Y')}] {x}", flush=True)

def sync():
    torch.cuda.synchronize()

def vram():
    free_b, total_b = torch.cuda.mem_get_info()
    return {
        "free_GB": round(free_b / 1024**3, 3),
        "alloc_GB": round(torch.cuda.memory_allocated() / 1024**3, 3),
        "reserved_GB": round(torch.cuda.memory_reserved() / 1024**3, 3),
        "peak_GB": round(torch.cuda.max_memory_allocated() / 1024**3, 3),
    }

def patch_cfg():
    cfg = model.config
    try:
        tc = cfg.get_text_config(decoder=True)
    except Exception:
        tc = getattr(cfg, "text_config", None) or getattr(cfg, "language_config", None) or cfg
    for k in [
        "num_attention_heads", "num_key_value_heads", "num_hidden_layers",
        "hidden_size", "head_dim", "max_position_embeddings", "sliding_window",
        "layer_types", "vocab_size", "bos_token_id", "eos_token_id", "pad_token_id"
    ]:
        if (not hasattr(cfg, k) or getattr(cfg, k, None) is None) and hasattr(tc, k):
            setattr(cfg, k, getattr(tc, k))
    if not hasattr(cfg, "head_dim") and hasattr(cfg, "hidden_size") and hasattr(cfg, "num_attention_heads"):
        cfg.head_dim = cfg.hidden_size // cfg.num_attention_heads

def chat_text(prompt):
    msgs = [
        {"role": "system", "content": "Code Python only. No markdown."},
        {"role": "user", "content": prompt},
    ]
    try:
        return processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        return processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def make_prompt(i):
    return (
        f"Write Python fuzz script for torch.searchsorted seed={9000+i}. "
        "Cases sorted unsorted duplicate nan inf empty int float 2d. "
        "Print PASS EXCEPTION SUSPICIOUS stats. Code only."
    )

log("BAT DAU CELL 14")
print("VRAM dau:", json.dumps(vram(), ensure_ascii=False))
patch_cfg()
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

rows = []

# ---- 1) Continuous batching, fix mismatch block_size/max_batch_tokens ----
def bench_cb(bs, max_new, block_size, max_batch_tokens, scheduler_type):
    input_ids = [tok.encode(chat_text(make_prompt(i)), add_special_tokens=False) for i in range(bs)]
    gen_cfg = GenerationConfig(
        max_new_tokens=max_new,
        do_sample=False,
        eos_token_id=getattr(tok, "eos_token_id", None),
        pad_token_id=getattr(tok, "pad_token_id", None) or getattr(tok, "eos_token_id", None),
    )
    cb_cfg = ContinuousBatchingConfig(
        block_size=block_size,
        max_batch_tokens=max_batch_tokens,
        max_memory_percent=0.70,
        max_blocks_per_request=0,
        allow_block_sharing=False,
        use_cuda_graph=False,
        scheduler_type=scheduler_type,
    )

    with torch.no_grad():
        outs = model.generate_batch(
            inputs=input_ids,
            generation_config=gen_cfg,
            continuous_batching_config=cb_cfg,
            progress_bar=False,
            warmup=False,
        )
        sync()

    total_tok = sum(len(getattr(o, "generated_tokens", [])) for o in outs.values())
    return total_tok

cb_trials = [
    (512, 128, "prefill_first"),
    (512, 256, "prefill_first"),
    (512, 512, "prefill_first"),
    (1024, 256, "prefill_first"),
    (1024, 512, "prefill_first"),
]

for block_size, max_batch_tokens, scheduler in cb_trials:
    try:
        log(f"CB SMOKE block={block_size} max_batch_tokens={max_batch_tokens} scheduler={scheduler}")
        sync()
        t0 = time.perf_counter()
        tok_count = bench_cb(32, 64, block_size, max_batch_tokens, scheduler)
        sync()
        dt = time.perf_counter() - t0
        spd = tok_count / max(dt, 1e-9)
        rows.append({
            "mode": "continuous_smoke",
            "batch": 32,
            "max_new": 64,
            "block_size": block_size,
            "max_batch_tokens": max_batch_tokens,
            "tokens": tok_count,
            "sec": round(dt, 4),
            "tok_s_total": round(spd, 3),
            "vram_peak_GB": vram()["peak_GB"],
        })
        log(f"CB smoke tokens={tok_count} speed={spd:.1f}")
        if tok_count > 0:
            log("CB SMOKE PASSED, RUN FULL 128x128")
            sync()
            t0 = time.perf_counter()
            tok_count = bench_cb(128, 128, block_size, max_batch_tokens, scheduler)
            sync()
            dt = time.perf_counter() - t0
            rows.append({
                "mode": "continuous_full",
                "batch": 128,
                "max_new": 128,
                "block_size": block_size,
                "max_batch_tokens": max_batch_tokens,
                "tokens": tok_count,
                "sec": round(dt, 4),
                "tok_s_total": round(tok_count / max(dt, 1e-9), 3),
                "vram_peak_GB": vram()["peak_GB"],
            })
            break
    except Exception:
        log("CB FAIL:")
        print(traceback.format_exc()[:1400])
        torch.cuda.empty_cache()

# ---- 2) Static batch push beyond 256 ----
def make_static_prompt(i):
    return (
        f"Write Python fuzz script for torch.searchsorted seed={10000+i}. "
        "Cases sorted unsorted duplicate nan inf empty int float 2d. "
        "Print PASS EXCEPTION SUSPICIOUS stats. Code only."
    )

if "encode_batch" in globals() and "generate_batch" in globals():
    for bs in [272, 288, 320]:
        try:
            log(f"STATIC PUSH batch={bs} max_new=256")
            prompts = [make_static_prompt(i) for i in range(bs)]
            x = encode_batch(prompts)
            input_len = int(x["input_ids"].shape[-1])

            _ = generate_batch(x, 256, "static")
            sync()

            sync()
            t0 = time.perf_counter()
            out = generate_batch(x, 256, "static")
            sync()
            dt = time.perf_counter() - t0

            total_tok = bs * int(out.shape[-1] - input_len)
            rows.append({
                "mode": "static_push",
                "batch": bs,
                "max_new": 256,
                "block_size": None,
                "max_batch_tokens": None,
                "tokens": total_tok,
                "sec": round(dt, 4),
                "tok_s_total": round(total_tok / max(dt, 1e-9), 3),
                "vram_peak_GB": vram()["peak_GB"],
            })
            log(f"STATIC OK {rows[-1]['tok_s_total']} tok/s")
        except RuntimeError as e:
            log(f"STATIC OOM/FAIL batch={bs}: {str(e)[:500]}")
            torch.cuda.empty_cache()
            break

log("=" * 100)
df = pd.DataFrame(rows)
if len(df):
    print(df.sort_values("tok_s_total", ascending=False).to_string(index=False))
else:
    print("No successful rows.")

print("VRAM cuoi:", json.dumps(vram(), ensure_ascii=False))
log("NEU CB VAN 0/FAIL VA STATIC >256 KHONG TANG RO, CHOT STATIC batch=256 max_new=256.")
log("HOAN TAT CELL 14")
# ================= HET CELL 14 =================

[21:05:14 04/06/2026] BAT DAU CELL 14
VRAM dau: {"free_GB": 2.355, "alloc_GB": 10.977, "reserved_GB": 11.611, "peak_GB": 12.56}
[21:05:18 04/06/2026] CB SMOKE block=512 max_batch_tokens=128 scheduler=prefill_first


2026-06-04 21:05:18,122 - ContinuousBatchingLogger - ERROR - Error in generation loop: index_copy_(): Source/destination tensor must have same slice shapes. Destination slice shape: 1 256 at dimension 0 and source slice shape: 1 512 at dimension 0.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/transformers/generation/continuous_batching/continuous_api.py", line 898, in _run_generation_loop
    self._generation_step()
  File "/usr/local/lib/python3.12/dist-packages/transformers/generation/continuous_batching/continuous_api.py", line 941, in _generation_step
    self.batch_processor._generation_step(self.model)
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/_contextlib.py", line 124, in decorate_context
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/generation/continuous_batching/continuous_api.py", line 517, in _generation_step
    self.model_runner.compute_b

Returning results of generate_batch despite unexpected termination.
[21:05:20 04/06/2026] CB smoke tokens=0 speed=0.0
[21:05:20 04/06/2026] CB SMOKE block=512 max_batch_tokens=256 scheduler=prefill_first


2026-06-04 21:05:20,511 - ContinuousBatchingLogger - ERROR - Error in generation loop: index_copy_(): Source/destination tensor must have same slice shapes. Destination slice shape: 1 256 at dimension 0 and source slice shape: 1 512 at dimension 0.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/transformers/generation/continuous_batching/continuous_api.py", line 898, in _run_generation_loop
    self._generation_step()
  File "/usr/local/lib/python3.12/dist-packages/transformers/generation/continuous_batching/continuous_api.py", line 941, in _generation_step
    self.batch_processor._generation_step(self.model)
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/_contextlib.py", line 124, in decorate_context
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/generation/continuous_batching/continuous_api.py", line 517, in _generation_step
    self.model_runner.compute_b

Returning results of generate_batch despite unexpected termination.
[21:05:23 04/06/2026] CB smoke tokens=0 speed=0.0
[21:05:23 04/06/2026] CB SMOKE block=512 max_batch_tokens=512 scheduler=prefill_first


...

Returning results of generate_batch despite unexpected termination.
[21:05:25 04/06/2026] CB smoke tokens=0 speed=0.0
[21:05:25 04/06/2026] CB SMOKE block=1024 max_batch_tokens=256 scheduler=prefill_first


2026-06-04 21:05:25,611 - ContinuousBatchingLogger - ERROR - Error in generation loop: index_copy_(): Source/destination tensor must have same slice shapes. Destination slice shape: 1 256 at dimension 0 and source slice shape: 1 512 at dimension 0.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/transformers/generation/continuous_batching/continuous_api.py", line 898, in _run_generation_loop
    self._generation_step()
  File "/usr/local/lib/python3.12/dist-packages/transformers/generation/continuous_batching/continuous_api.py", line 941, in _generation_step
    self.batch_processor._generation_step(self.model)
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/_contextlib.py", line 124, in decorate_context
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/generation/continuous_batching/continuous_api.py", line 517, in _generation_step
    self.model_runner.compute_b

Returning results of generate_batch despite unexpected termination.
[21:05:28 04/06/2026] CB smoke tokens=0 speed=0.0
[21:05:28 04/06/2026] CB SMOKE block=1024 max_batch_tokens=512 scheduler=prefill_first


2026-06-04 21:05:28,084 - ContinuousBatchingLogger - ERROR - Error in generation loop: index_copy_(): Source/destination tensor must have same slice shapes. Destination slice shape: 1 256 at dimension 0 and source slice shape: 1 512 at dimension 0.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/transformers/generation/continuous_batching/continuous_api.py", line 898, in _run_generation_loop
    self._generation_step()
  File "/usr/local/lib/python3.12/dist-packages/transformers/generation/continuous_batching/continuous_api.py", line 941, in _generation_step
    self.batch_processor._generation_step(self.model)
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/_contextlib.py", line 124, in decorate_context
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/generation/continuous_batching/continuous_api.py", line 517, in _generation_step
    self.model_runner.compute_b

Returning results of generate_batch despite unexpected termination.
[21:05:30 04/06/2026] CB smoke tokens=0 speed=0.0
[21:05:30 04/06/2026] STATIC PUSH batch=272 max_new=256
[21:05:31 04/06/2026] STATIC OOM/FAIL batch=272: CUDA out of memory. Tried to allocate 678.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 435.81 MiB is free. Including non-PyTorch memory, this process has 14.13 GiB memory in use. Of the allocated memory 13.27 GiB is allocated by PyTorch, with 209.75 MiB allocated in private pools (e.g., CUDA Graphs), and 276.39 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  S
[21:05:31 04/06/2026] ====================================================================================================
            mode  batch  max_new  block_size  max_batch_tokens  tokens    sec  tok_s_total  vram_peak_GB
continuous_smoke     32       64         512    

In [ ]:
# ================= CELL 15: VLLM T4 GEMMA4 BENCH =================
# Sau reset kernel. Muc tieu: doi backend tu Transformers sang vLLM.
# Neu chua login HF va model gated, set HF_TOKEN trong env/Colab secrets truoc.

import os, sys, subprocess, importlib.util, time, json, gc, traceback

MODEL_ID = os.environ.get("MODEL_ID", "google/gemma-4-E4B-it")  # doi neu ong dang dung model khac
MAX_MODEL_LEN = 512       # du cho prompt ngan + max_new=256, giam KV cache de fit T4
MAX_NEW = 256
MAX_NUM_SEQS = 256
MAX_NUM_BATCHED_TOKENS = 65536
GPU_MEM = 0.90

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("VLLM_USE_V1", "1")

try:
    from google.colab import userdata
    hf_tok = userdata.get("HF_TOKEN")
    if hf_tok:
        os.environ["HF_TOKEN"] = hf_tok
except Exception:
    pass

def sh(cmd):
    print("RUN:", " ".join(cmd), flush=True)
    subprocess.check_call(cmd)

if importlib.util.find_spec("vllm") is None:
    sh([
        sys.executable, "-m", "pip", "install", "-q", "-U",
        "vllm",
        "--extra-index-url", "https://download.pytorch.org/whl/cu129",
    ])

import torch
import pandas as pd
from vllm import LLM, SamplingParams

def sync():
    torch.cuda.synchronize()

def vram():
    free_b, total_b = torch.cuda.mem_get_info()
    return {
        "free_GB": round(free_b / 1024**3, 3),
        "total_GB": round(total_b / 1024**3, 3),
        "alloc_GB": round(torch.cuda.memory_allocated() / 1024**3, 3),
        "reserved_GB": round(torch.cuda.memory_reserved() / 1024**3, 3),
    }

print("MODEL_ID:", MODEL_ID)
print("GPU:", torch.cuda.get_device_name(0))
print("capability:", torch.cuda.get_device_capability(0))
print("VRAM dau:", json.dumps(vram(), ensure_ascii=False))

def make_raw_prompt(i):
    return (
        f"Write Python fuzz script for torch.searchsorted seed={12000+i}. "
        "Cases: sorted, unsorted, duplicate, nan, inf, empty, int, float, 2d. "
        "Print PASS, EXCEPTION, SUSPICIOUS, and final stats. Code only."
    )

def load_llm():
    trials = [
        ("FLASHINFER", 512),
        ("TRITON_ATTN", 512),
        (None, 512),
        ("TRITON_ATTN", 384),
        (None, 384),
    ]

    last = None
    for backend, max_len in trials:
        try:
            print(f"\nLOAD vLLM backend={backend} max_model_len={max_len}", flush=True)
            kwargs = dict(
                model=MODEL_ID,
                dtype="float16",
                trust_remote_code=True,
                max_model_len=max_len,
                gpu_memory_utilization=GPU_MEM,
                max_num_seqs=MAX_NUM_SEQS,
                max_num_batched_tokens=MAX_NUM_BATCHED_TOKENS,
                enforce_eager=False,
                disable_log_stats=True,
            )
            if backend is not None:
                kwargs["attention_backend"] = backend

            llm = LLM(**kwargs)
            return llm, backend, max_len
        except Exception as e:
            last = e
            print(f"LOAD FAIL backend={backend} max_len={max_len}: {str(e)[:1000]}", flush=True)
            gc.collect()
            torch.cuda.empty_cache()

    raise last

llm, backend, max_len = load_llm()
tokenizer = llm.get_tokenizer()

def format_prompt(p):
    msgs = [{"role": "user", "content": p}]
    try:
        return tokenizer.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=True,
        )
    except Exception:
        return p

sampling = SamplingParams(
    temperature=0.0,
    max_tokens=MAX_NEW,
    min_tokens=MAX_NEW,
    ignore_eos=True,
)

# Warmup
warm_prompts = [format_prompt(make_raw_prompt(i)) for i in range(8)]
_ = llm.generate(warm_prompts, sampling, use_tqdm=False)
sync()

rows = []
for bs in [1, 4, 16, 64, 128, 192, 256]:
    try:
        prompts = [format_prompt(make_raw_prompt(i)) for i in range(bs)]

        sync()
        t0 = time.perf_counter()
        outs = llm.generate(prompts, sampling, use_tqdm=False)
        sync()
        dt = time.perf_counter() - t0

        out_tokens = sum(len(o.outputs[0].token_ids) for o in outs)
        row = {
            "engine": "vllm",
            "backend": str(backend),
            "max_model_len": max_len,
            "batch": bs,
            "max_new": MAX_NEW,
            "out_tokens": out_tokens,
            "sec": round(dt, 4),
            "tok_s_total": round(out_tokens / max(dt, 1e-9), 3),
            "tok_s_per_prompt": round(out_tokens / max(dt, 1e-9) / bs, 3),
        }
        rows.append(row)
        print(f"BATCH {bs}: {row['tok_s_total']} tok/s total | {row['tok_s_per_prompt']} per prompt", flush=True)

    except Exception:
        print(f"BATCH {bs} FAIL:")
        print(traceback.format_exc()[:1800])
        break

df = pd.DataFrame(rows).sort_values("tok_s_total", ascending=False)
print("\nBANG VLLM")
print(df.to_string(index=False))

print("VRAM cuoi:", json.dumps(vram(), ensure_ascii=False))
print("CHOT: so sanh best vLLM voi Transformers static cu ~2099 tok/s.")
# ================= HET CELL 15 =================

RUN: /usr/bin/python3 -m pip install -q -U vllm --extra-index-url https://download.pytorch.org/whl/cu129


ImportError: libcudart.so.13: cannot open shared object file: No such file or directory

In [ ]:
# ================= CELL 15A: FIX VLLM INSTALL CUDA MISMATCH =================
import os, sys, subprocess, importlib.util, traceback

def run(cmd, allow_fail=False):
    print("RUN:", " ".join(cmd), flush=True)
    p = subprocess.run(cmd, text=True)
    if p.returncode != 0 and not allow_fail:
        raise RuntimeError(f"Command failed: {' '.join(cmd)}")
    return p.returncode

# Xem CUDA/driver that su
run(["nvidia-smi"], allow_fail=True)

# Go ban vLLM loi vua cai
run([sys.executable, "-m", "pip", "uninstall", "-y", "vllm"], allow_fail=True)

# Dung uv theo khuyen nghi cua vLLM, de no tu chon torch backend.
# QUAN TRONG: khong them --extra-index-url cu129 nua.
run([sys.executable, "-m", "pip", "install", "-q", "-U", "uv"])
run(["uv", "pip", "install", "--system", "-q", "-U", "vllm", "--torch-backend=auto"])

# Test import
try:
    from vllm import LLM, SamplingParams
    print("vLLM import OK")
except Exception:
    print("vLLM import FAIL:")
    print(traceback.format_exc()[:3000])
    print("\nNeu van FAIL libcudart.so.13: vLLM wheel moi khong hop runtime Colab hien tai.")
    print("Luc do bo vLLM tren T4 Colab nay, quay ve Transformers static hoac factory reset runtime.")
    raise
# ================= HET CELL 15A =================

RUN: nvidia-smi
RUN: /usr/bin/python3 -m pip uninstall -y vllm
RUN: /usr/bin/python3 -m pip install -q -U uv
RUN: uv pip install --system -q -U vllm --torch-backend=auto
vLLM import FAIL:
Traceback (most recent call last):
  File "/tmp/ipykernel_7112/3146262848.py", line 24, in <cell line: 0>
    from vllm import LLM, SamplingParams
  File "/usr/local/lib/python3.12/dist-packages/vllm/__init__.py", line 70, in __getattr__
    module = import_module(module_name, __package__)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/importlib/__init__.py", line 90, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/vllm/entrypoints/llm.py", line 14, in <module>
    from vllm.config import (
  File "/usr/local/lib/python3.12/dist-packages/vllm/config/__init__.py", line 6, in <module>
    from vllm.config.compilation import (
 

ImportError: libcudart.so.13: cannot open shared object file: No such file or directory

In [ ]:
# CELL TIEP THEO - CHOT T4: bo vLLM, quay ve Transformers static
import os, gc, time, json, datetime
import torch
import pandas as pd

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_MODULE_LOADING"] = "LAZY"

def log(x):
    print(f"[{datetime.datetime.now().strftime('%H:%M:%S %d/%m/%Y')}] {x}", flush=True)

def sync():
    torch.cuda.synchronize()

def vram():
    free, total = torch.cuda.mem_get_info()
    return {
        "free_GB": round(free / 1024**3, 3),
        "alloc_GB": round(torch.cuda.memory_allocated() / 1024**3, 3),
        "reserved_GB": round(torch.cuda.memory_reserved() / 1024**3, 3),
        "peak_GB": round(torch.cuda.max_memory_allocated() / 1024**3, 3),
    }

if "model" not in globals() or "processor" not in globals():
    raise RuntimeError("Chay lai cell load model + processor truoc. Cell nay khong cai vLLM nua.")

name = torch.cuda.get_device_name(0)
cc = torch.cuda.get_device_capability(0)
log(f"GPU: {name} | compute capability={cc[0]}.{cc[1]}")
log("Bo vLLM: T4/SM75 + Gemma4 khong dang thu tiep trong vLLM.")
log("VRAM dau: " + json.dumps(vram()))

model.eval()
tok = getattr(processor, "tokenizer", processor)
if getattr(tok, "pad_token_id", None) is None and getattr(tok, "eos_token", None) is not None:
    tok.pad_token = tok.eos_token
try:
    tok.padding_side = "left"
except Exception:
    pass

prompt = "Viet ngan gon 3 y ve toi uu inference tren GPU T4."
try:
    text = tok.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True,
    )
except Exception:
    text = prompt

enc = tok([text], return_tensors="pt", padding=False)
enc = {k: v.to(model.device) for k, v in enc.items() if torch.is_tensor(v)}
enc.pop("token_type_ids", None)

@torch.inference_mode()
def test_static(batch, max_new=256):
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    ids = enc["input_ids"].repeat(batch, 1).contiguous()
    attn = enc.get("attention_mask", torch.ones_like(enc["input_ids"])).repeat(batch, 1).contiguous()

    sync()
    t0 = time.perf_counter()
    out = model.generate(
        input_ids=ids,
        attention_mask=attn,
        max_new_tokens=max_new,
        min_new_tokens=max_new,
        do_sample=False,
        use_cache=True,
        cache_implementation="static",
        pad_token_id=tok.pad_token_id,
        eos_token_id=None,
    )
    sync()

    sec = time.perf_counter() - t0
    new_tok = out.shape[1] - ids.shape[1]
    total_tok = batch * new_tok
    return {
        "batch": batch,
        "max_new": max_new,
        "total_tok": total_tok,
        "sec": round(sec, 4),
        "tok_s_total": round(total_tok / sec, 3),
        "tok_s_per_prompt": round(total_tok / sec / batch, 3),
        **vram(),
    }

rows = []
for b in [256, 240, 224]:
    try:
        log(f"TEST static batch={b} max_new=256")
        r = test_static(b, 256)
        rows.append(r)
        log(f"OK {r['tok_s_total']} tok/s | per_prompt {r['tok_s_per_prompt']}")
        break
    except torch.cuda.OutOfMemoryError:
        log(f"OOM batch={b}, thu batch nho hon")
        torch.cuda.empty_cache()

df = pd.DataFrame(rows)
display(df)

BEST_T4_BATCH = int(df.iloc[0]["batch"])
BEST_T4_MAX_NEW = 256
log(f"CHOT: Transformers static batch={BEST_T4_BATCH}, max_new=256")
log("VRAM cuoi: " + json.dumps(vram()))

RuntimeError: Chay lai cell load model + processor truoc. Cell nay khong cai vLLM nua.

In [ ]:
# CELL MOI - COLD START T4 GEMMA4 TRANSFORMERS STATIC
import os, sys, subprocess, gc, time, json, datetime, traceback

MODEL_ID = os.environ.get("MODEL_ID", "google/gemma-4-E4B-it")  # doi thanh E2B neu ban dung E2B
HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN") or None

def log(x):
    print(f"[{datetime.datetime.now().strftime('%H:%M:%S %d/%m/%Y')}] {x}", flush=True)

def run(cmd, allow_fail=False):
    log("RUN: " + " ".join(cmd))
    p = subprocess.run(cmd, text=True)
    if p.returncode != 0 and not allow_fail:
        raise RuntimeError("Command fail: " + " ".join(cmd))
    return p.returncode

log("CAI/UPDATE STACK CAN THIET - khong cai vLLM")
run([sys.executable, "-m", "pip", "install", "-q", "-U",
     "transformers>=5.10.0", "accelerate", "safetensors", "sentencepiece", "protobuf", "pandas"])

import torch
import pandas as pd
from transformers import AutoProcessor

try:
    from transformers import AutoModelForImageTextToText as AutoModelCls
except Exception:
    from transformers import AutoModelForCausalLM as AutoModelCls

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_MODULE_LOADING"] = "LAZY"
torch.backends.cuda.matmul.allow_tf32 = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

def sync():
    torch.cuda.synchronize()

def vram():
    free, total = torch.cuda.mem_get_info()
    return {
        "free_GB": round(free / 1024**3, 3),
        "alloc_GB": round(torch.cuda.memory_allocated() / 1024**3, 3),
        "reserved_GB": round(torch.cuda.memory_reserved() / 1024**3, 3),
        "peak_GB": round(torch.cuda.max_memory_allocated() / 1024**3, 3),
    }

run(["nvidia-smi"], allow_fail=True)
log(f"Torch: {torch.__version__}")
log(f"CUDA available: {torch.cuda.is_available()}")
log(f"GPU: {torch.cuda.get_device_name(0)} | CC={torch.cuda.get_device_capability(0)}")

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
log("VRAM truoc load: " + json.dumps(vram()))

log(f"LOAD processor: {MODEL_ID}")
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    trust_remote_code=True,
)

log(f"LOAD model FP16 SDPA: {MODEL_ID}")
model_kwargs = dict(
    device_map="auto",
    low_cpu_mem_usage=True,
    attn_implementation="sdpa",
    token=HF_TOKEN,
    trust_remote_code=True,
)

try:
    model = AutoModelCls.from_pretrained(MODEL_ID, dtype=torch.float16, **model_kwargs)
except TypeError:
    model = AutoModelCls.from_pretrained(MODEL_ID, torch_dtype=torch.float16, **model_kwargs)

model.eval()
tok = getattr(processor, "tokenizer", processor)

if getattr(tok, "pad_token_id", None) is None:
    tok.pad_token = tok.eos_token

try:
    tok.padding_side = "left"
except Exception:
    pass

device = next(model.parameters()).device
log(f"Model device: {device}")
log("VRAM sau load: " + json.dumps(vram()))

prompt = "Viet ngan gon 3 y ve toi uu inference tren GPU T4."

try:
    text = tok.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True,
    )
except Exception:
    try:
        text = processor.apply_chat_template(
            [{"role": "user", "content": prompt}],
            tokenize=False,
            add_generation_prompt=True,
        )
    except Exception:
        text = prompt

enc = tok([text], return_tensors="pt", padding=True)
enc = {k: v.to(device) for k, v in enc.items() if torch.is_tensor(v)}
enc.pop("token_type_ids", None)

base_ids = enc["input_ids"]
base_mask = enc.get("attention_mask", torch.ones_like(base_ids))

def generate_static(input_ids, attention_mask, max_new):
    kwargs = dict(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=max_new,
        min_new_tokens=max_new,
        do_sample=False,
        use_cache=True,
        cache_implementation="static",
        pad_token_id=tok.pad_token_id,
        eos_token_id=None,
        forced_eos_token_id=None,
    )
    try:
        return model.generate(**kwargs)
    except TypeError:
        kwargs.pop("cache_implementation", None)
        return model.generate(**kwargs)

@torch.inference_mode()
def bench(batch, max_new=256):
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    ids = base_ids.repeat(batch, 1).contiguous()
    mask = base_mask.repeat(batch, 1).contiguous()

    sync()
    t0 = time.perf_counter()
    out = generate_static(ids, mask, max_new)
    sync()

    sec = time.perf_counter() - t0
    new_tok = out.shape[1] - ids.shape[1]
    total_tok = batch * new_tok

    return {
        "batch": batch,
        "input_len": int(ids.shape[1]),
        "max_new": max_new,
        "total_tok": int(total_tok),
        "sec": round(sec, 4),
        "tok_s_total": round(total_tok / sec, 3),
        "tok_s_per_prompt": round(total_tok / sec / batch, 3),
        **vram(),
    }

log("WARMUP nho")
try:
    _ = bench(1, 16)
except Exception:
    log("Warmup fail:")
    print(traceback.format_exc()[:3000])
    raise

rows = []

log("TEST LATENCY batch=1 max_new=256")
try:
    r = bench(1, 256)
    rows.append(r)
    log(f"LATENCY OK: {r['tok_s_total']} tok/s")
except Exception:
    log("Latency test fail:")
    print(traceback.format_exc()[:3000])

log("TEST THROUGHPUT tu cao xuong thap")
for b in [256, 240, 224, 192, 128]:
    try:
        log(f"TEST static batch={b} max_new=256")
        r = bench(b, 256)
        rows.append(r)
        log(f"OK {r['tok_s_total']} tok/s | per_prompt {r['tok_s_per_prompt']} | vram {json.dumps(vram())}")
        break
    except torch.cuda.OutOfMemoryError:
        log(f"OOM batch={b}, giam batch")
        gc.collect()
        torch.cuda.empty_cache()
    except Exception:
        log(f"FAIL batch={b}:")
        print(traceback.format_exc()[:3000])
        gc.collect()
        torch.cuda.empty_cache()

df = pd.DataFrame(rows).sort_values("tok_s_total", ascending=False)
log("=" * 100)
log("BANG KET QUA")
display(df)
log("=" * 100)

if len(df):
    best = df.iloc[0]
    BEST_T4_BATCH = int(best["batch"])
    BEST_T4_MAX_NEW = int(best["max_new"])
    log(f"CHOT T4: batch={BEST_T4_BATCH}, max_new={BEST_T4_MAX_NEW}, total={best['tok_s_total']} tok/s")
else:
    log("KHONG CO KET QUA - xem traceback phia tren")

log("VRAM cuoi: " + json.dumps(vram()))

[21:51:50 04/06/2026] CAI/UPDATE STACK CAN THIET - khong cai vLLM
[21:51:50 04/06/2026] RUN: /usr/bin/python3 -m pip install -q -U transformers>=5.10.0 accelerate safetensors sentencepiece protobuf pandas
[21:52:54 04/06/2026] RUN: nvidia-smi
[21:52:54 04/06/2026] Torch: 2.11.0+cu128
[21:52:54 04/06/2026] CUDA available: True
[21:52:54 04/06/2026] GPU: Tesla T4 | CC=(7, 5)
[21:52:54 04/06/2026] VRAM truoc load: {"free_GB": 14.461, "alloc_GB": 0.0, "reserved_GB": 0.0, "peak_GB": 0.0}
[21:52:54 04/06/2026] LOAD processor: google/gemma-4-E4B-it


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/1.69k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/17.3k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/5.14k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

[21:57:53 04/06/2026] Model device: cuda:0
[21:57:53 04/06/2026] VRAM sau load: {"free_GB": 6.722, "alloc_GB": 7.693, "reserved_GB": 7.723, "peak_GB": 7.693}
[21:57:53 04/06/2026] WARMUP nho
[21:58:24 04/06/2026] TEST LATENCY batch=1 max_new=256
[22:05:57 04/06/2026] LATENCY OK: 0.566 tok/s
[22:05:57 04/06/2026] TEST THROUGHPUT tu cao xuong thap
[22:05:57 04/06/2026] TEST static batch=256 max_new=256
[22:06:02 04/06/2026] OOM batch=256, giam batch
[22:06:03 04/06/2026] TEST static batch=240 max_new=256
[22:06:09 04/06/2026] OOM batch=240, giam batch
[22:06:09 04/06/2026] TEST static batch=224 max_new=256
[22:06:14 04/06/2026] OOM batch=224, giam batch
[22:06:14 04/06/2026] TEST static batch=192 max_new=256
[22:06:19 04/06/2026] OOM batch=192, giam batch
[22:06:20 04/06/2026] TEST static batch=128 max_new=256
[22:06:24 04/06/2026] OOM batch=128, giam batch
[22:06:24 04/06/2026] ====================================================================================================
[22:06:24

,batch,input_len,max_new,total_tok,sec,tok_s_total,tok_s_per_prompt,free_GB,alloc_GB,reserved_GB,peak_GB
0,1,24,256,256,452.6435,0.566,0.566,6.691,7.716,7.742,12.967


[22:06:24 04/06/2026] ====================================================================================================
[22:06:24 04/06/2026] CHOT T4: batch=1, max_new=256, total=0.566 tok/s
[22:06:24 04/06/2026] VRAM cuoi: {"free_GB": 4.49, "alloc_GB": 9.613, "reserved_GB": 9.943, "peak_GB": 13.107}


In [ ]:
# CELL SUA LOI OFFLOAD CPU - RELOAD FULL CUDA ROI BENCH T4
import os, sys, gc, time, json, datetime, traceback
import torch
import pandas as pd
from transformers import AutoProcessor

try:
    from transformers import AutoModelForImageTextToText as AutoModelCls
except Exception:
    from transformers import AutoModelForCausalLM as AutoModelCls

MODEL_ID = os.environ.get("MODEL_ID", "google/gemma-4-E4B-it")
HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN") or None

def log(x):
    print(f"[{datetime.datetime.now().strftime('%H:%M:%S %d/%m/%Y')}] {x}", flush=True)

def sync():
    torch.cuda.synchronize()

def vram():
    free, total = torch.cuda.mem_get_info()
    return {
        "free_GB": round(free / 1024**3, 3),
        "alloc_GB": round(torch.cuda.memory_allocated() / 1024**3, 3),
        "reserved_GB": round(torch.cuda.memory_reserved() / 1024**3, 3),
        "peak_GB": round(torch.cuda.max_memory_allocated() / 1024**3, 3),
    }

log("XOA MODEL CU BI CPU OFFLOAD")
for k in ["model"]:
    if k in globals():
        del globals()[k]
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
log("VRAM sau xoa: " + json.dumps(vram()))

torch.backends.cuda.matmul.allow_tf32 = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

log(f"LOAD processor: {MODEL_ID}")
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    trust_remote_code=True,
)

log("LOAD MODEL FULL CUDA - cam CPU offload")
load_kwargs = dict(
    token=HF_TOKEN,
    trust_remote_code=True,
    attn_implementation="sdpa",
    low_cpu_mem_usage=True,
    device_map={"": 0},
)

try:
    model = AutoModelCls.from_pretrained(MODEL_ID, dtype=torch.float16, **load_kwargs)
except TypeError:
    model = AutoModelCls.from_pretrained(MODEL_ID, torch_dtype=torch.float16, **load_kwargs)

model.eval()
sync()
log("VRAM sau load full CUDA: " + json.dumps(vram()))

device_map = getattr(model, "hf_device_map", None)
log(f"hf_device_map: {device_map}")

bad = []
for name, p in model.named_parameters():
    if p.device.type != "cuda":
        bad.append((name, str(p.device)))
        if len(bad) >= 5:
            break

if bad:
    raise RuntimeError(f"VAN CON PARAM KHONG O CUDA: {bad}. Dung cell, khong bench vi se cham nhu CPU offload.")

tok = getattr(processor, "tokenizer", processor)
if getattr(tok, "pad_token_id", None) is None:
    tok.pad_token = tok.eos_token
try:
    tok.padding_side = "left"
except Exception:
    pass

prompt = "Viet ngan gon 3 y ve toi uu inference tren GPU T4."
try:
    text = tok.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True,
    )
except Exception:
    text = prompt

enc = tok([text], return_tensors="pt", padding=True)
enc = {k: v.to("cuda") for k, v in enc.items() if torch.is_tensor(v)}
enc.pop("token_type_ids", None)

base_ids = enc["input_ids"]
base_mask = enc.get("attention_mask", torch.ones_like(base_ids))

def generate_static(ids, mask, max_new):
    kwargs = dict(
        input_ids=ids,
        attention_mask=mask,
        max_new_tokens=max_new,
        min_new_tokens=max_new,
        do_sample=False,
        use_cache=True,
        cache_implementation="static",
        pad_token_id=tok.pad_token_id,
        eos_token_id=None,
        forced_eos_token_id=None,
    )
    try:
        return model.generate(**kwargs)
    except TypeError:
        kwargs.pop("cache_implementation", None)
        return model.generate(**kwargs)

@torch.inference_mode()
def bench(batch, max_new):
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    ids = base_ids.repeat(batch, 1).contiguous()
    mask = base_mask.repeat(batch, 1).contiguous()

    sync()
    t0 = time.perf_counter()
    out = generate_static(ids, mask, max_new)
    sync()

    sec = time.perf_counter() - t0
    new_tok = out.shape[1] - ids.shape[1]
    total_tok = batch * new_tok
    return {
        "batch": batch,
        "input_len": int(ids.shape[1]),
        "max_new": max_new,
        "total_tok": int(total_tok),
        "sec": round(sec, 4),
        "tok_s_total": round(total_tok / sec, 3),
        "tok_s_per_prompt": round(total_tok / sec / batch, 3),
        **vram(),
    }

rows = []

log("SANITY batch=1 max_new=8")
r = bench(1, 8)
rows.append(r)
log(f"SANITY: {r['tok_s_total']} tok/s")

if r["tok_s_total"] < 5:
    raise RuntimeError("Van cham bat thuong. Gan chac model con offload/IO CPU. Dung tai day de khoi mat 7 phut nua.")

log("LATENCY batch=1 max_new=256")
r = bench(1, 256)
rows.append(r)
log(f"LATENCY OK: {r['tok_s_total']} tok/s")

log("THROUGHPUT thu tu cao xuong")
for b in [256, 240, 224, 192, 128, 64, 32]:
    try:
        log(f"TEST batch={b} max_new=256")
        r = bench(b, 256)
        rows.append(r)
        log(f"OK {r['tok_s_total']} tok/s | per_prompt {r['tok_s_per_prompt']}")
        break
    except torch.cuda.OutOfMemoryError:
        log(f"OOM batch={b}, giam batch")
        gc.collect()
        torch.cuda.empty_cache()

df = pd.DataFrame(rows).sort_values("tok_s_total", ascending=False)
display(df)

best = df.iloc[0]
log(f"CHOT: batch={int(best['batch'])}, max_new={int(best['max_new'])}, total={best['tok_s_total']} tok/s")
log("VRAM cuoi: " + json.dumps(vram()))

[22:08:26 04/06/2026] XOA MODEL CU BI CPU OFFLOAD
[22:08:29 04/06/2026] VRAM sau xoa: {"free_GB": 6.74, "alloc_GB": 0.008, "reserved_GB": 7.693, "peak_GB": 0.008}
[22:08:29 04/06/2026] LOAD processor: google/gemma-4-E4B-it
[22:08:38 04/06/2026] LOAD MODEL FULL CUDA - cam CPU offload


Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 11.81 MiB is free. Including non-PyTorch memory, this process has 14.55 GiB memory in use. Of the allocated memory 14.42 GiB is allocated by PyTorch, and 4.02 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
# ===================== GEMMA 4 E2B-IT / T4 / CLEAN STATIC BENCH =====================
# Chot lai dung model ban dau: google/gemma-4-E2B-it
# Khong vLLM, khong E4B, khong CPU offload am tham.

import os, sys, subprocess, gc, time, json, datetime, traceback

MODEL_ID = "google/gemma-4-E2B-it"
INSTALL_DEPS = True

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"

def log(x):
    print(f"[{datetime.datetime.now().strftime('%H:%M:%S %d/%m/%Y')}] {x}", flush=True)

def run(cmd, allow_fail=False):
    log("RUN: " + " ".join(cmd))
    p = subprocess.run(cmd, text=True)
    if p.returncode != 0 and not allow_fail:
        raise RuntimeError("Command fail: " + " ".join(cmd))
    return p.returncode

if INSTALL_DEPS:
    run([sys.executable, "-m", "pip", "install", "-q", "-U",
         "transformers>=5.10.0", "accelerate", "safetensors",
         "sentencepiece", "protobuf", "hf_transfer", "pandas"])

import torch
import pandas as pd
from transformers import AutoProcessor, AutoModelForCausalLM

HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN") or None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN") or HF_TOKEN
except Exception:
    pass

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.benchmark = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

def sync():
    torch.cuda.synchronize()

def clean():
    gc.collect()
    torch.cuda.empty_cache()
    try:
        torch.cuda.ipc_collect()
    except Exception:
        pass
    torch.cuda.reset_peak_memory_stats()

def vram():
    free, total = torch.cuda.mem_get_info()
    return {
        "free_GB": round(free / 1024**3, 3),
        "total_GB": round(total / 1024**3, 3),
        "alloc_GB": round(torch.cuda.memory_allocated() / 1024**3, 3),
        "reserved_GB": round(torch.cuda.memory_reserved() / 1024**3, 3),
        "peak_GB": round(torch.cuda.max_memory_allocated() / 1024**3, 3),
    }

log("PURGE model/processor cu")
for k in ["model", "processor", "tok", "assistant_model"]:
    globals().pop(k, None)
clean()

run(["nvidia-smi"], allow_fail=True)
log(f"GPU: {torch.cuda.get_device_name(0)} | CC={torch.cuda.get_device_capability(0)}")
log("VRAM truoc load: " + json.dumps(vram()))

log(f"LOAD processor: {MODEL_ID}")
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    trust_remote_code=True,
)
tok = getattr(processor, "tokenizer", processor)

if getattr(tok, "pad_token_id", None) is None:
    tok.pad_token = tok.eos_token
try:
    tok.padding_side = "left"
except Exception:
    pass

log(f"LOAD model FP16 full CUDA: {MODEL_ID}")
t0 = time.perf_counter()
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    attn_implementation="sdpa",
    device_map={"": 0},
    low_cpu_mem_usage=True,
)
model.eval()
model.config.use_cache = True
sync()

log(f"Load xong sau {time.perf_counter() - t0:.2f}s")
log("VRAM sau load: " + json.dumps(vram()))

print("MODEL_ID:", MODEL_ID)
print("config.model_type:", getattr(model.config, "model_type", None))
print("hf_device_map:", getattr(model, "hf_device_map", None))
print("first_param_dtype:", next(model.parameters()).dtype)
print("first_param_device:", next(model.parameters()).device)

bad = []
total_params = 0
cuda_params = 0
for name, p in model.named_parameters():
    total_params += p.numel()
    if p.device.type == "cuda":
        cuda_params += p.numel()
    else:
        bad.append((name, str(p.device), str(p.dtype)))
        if len(bad) >= 5:
            break

print("total_params_B:", round(total_params / 1e9, 3))
print("cuda_params_B:", round(cuda_params / 1e9, 3))
if bad:
    raise RuntimeError(f"CO PARAM NGOAI CUDA, KHONG BENCH: {bad}")

SYSTEM_PROMPT = "Code Python, no markdown."
USER_PROMPT = "Fuzz torch.searchsorted: seed=42, sorted/unsorted/dup/nan/inf/empty/int/float/2d, print output/exception/stats."

def make_text(prompt):
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    try:
        return processor.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        return processor.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=True,
        )
    except Exception:
        return prompt

text = make_text(USER_PROMPT)
enc = processor(text=text, return_tensors="pt")
enc = {k: v.to("cuda") for k, v in enc.items() if torch.is_tensor(v)}
enc.pop("token_type_ids", None)

base_ids = enc["input_ids"]
base_mask = enc.get("attention_mask", torch.ones_like(base_ids))

def generate_static(ids, mask, max_new):
    kwargs = dict(
        input_ids=ids,
        attention_mask=mask,
        max_new_tokens=max_new,
        min_new_tokens=max_new,
        do_sample=False,
        use_cache=True,
        cache_implementation="static",
        pad_token_id=tok.pad_token_id,
        eos_token_id=None,
        forced_eos_token_id=None,
    )
    try:
        return model.generate(**kwargs)
    except TypeError:
        kwargs.pop("cache_implementation", None)
        return model.generate(**kwargs)

@torch.inference_mode()
def bench(batch, max_new):
    clean()
    ids = base_ids.repeat(batch, 1).contiguous()
    mask = base_mask.repeat(batch, 1).contiguous()

    sync()
    t0 = time.perf_counter()
    out = generate_static(ids, mask, max_new)
    sync()

    sec = time.perf_counter() - t0
    new_tok = out.shape[1] - ids.shape[1]
    total_tok = batch * new_tok

    return {
        "model_id": MODEL_ID,
        "batch": batch,
        "input_len": int(ids.shape[1]),
        "max_new": max_new,
        "total_tok": int(total_tok),
        "sec": round(sec, 4),
        "tok_s_total": round(total_tok / sec, 3),
        "tok_s_per_prompt": round(total_tok / sec / batch, 3),
        **vram(),
    }

rows = []

log("WARMUP batch=1 max_new=24")
_ = bench(1, 24)

log("LATENCY batch=1 max_new=256")
r = bench(1, 256)
rows.append(r)
log(f"OK batch=1: {r['tok_s_total']} tok/s")

log("THROUGHPUT sweep")
for b in [256, 240, 224, 192, 128, 64, 32, 16, 8, 4]:
    try:
        log(f"TEST batch={b} max_new=256")
        r = bench(b, 256)
        rows.append(r)
        log(f"OK batch={b}: {r['tok_s_total']} tok/s | per_prompt={r['tok_s_per_prompt']}")
        break
    except torch.cuda.OutOfMemoryError:
        log(f"OOM batch={b}, giam batch")
        clean()
    except Exception:
        log(f"FAIL batch={b}")
        print(traceback.format_exc()[:3000])
        clean()

df = pd.DataFrame(rows).sort_values("tok_s_total", ascending=False)
display(df)

best = df.iloc[0]
log(f"CHOT E2B: batch={int(best['batch'])}, max_new={int(best['max_new'])}, total={best['tok_s_total']} tok/s")
log("VRAM cuoi: " + json.dumps(vram()))

[22:22:38 04/06/2026] RUN: /usr/bin/python3 -m pip install -q -U transformers>=5.10.0 accelerate safetensors sentencepiece protobuf hf_transfer pandas


/usr/local/lib/python3.12/dist-packages/huggingface_hub/constants.py:277: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


[22:23:39 04/06/2026] PURGE model/processor cu
[22:23:39 04/06/2026] RUN: nvidia-smi
[22:23:39 04/06/2026] GPU: Tesla T4 | CC=(7, 5)
[22:23:40 04/06/2026] VRAM truoc load: {"free_GB": 14.461, "total_GB": 14.563, "alloc_GB": 0.0, "reserved_GB": 0.0, "peak_GB": 0.0}
[22:23:40 04/06/2026] LOAD processor: google/gemma-4-E2B-it


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/1.69k [00:00<?, ?B/s]

...

[22:26:05 04/06/2026] Load xong sau 136.90s
[22:26:05 04/06/2026] VRAM sau load: {"free_GB": 4.861, "total_GB": 14.563, "alloc_GB": 9.508, "reserved_GB": 9.584, "peak_GB": 9.508}
MODEL_ID: google/gemma-4-E2B-it
config.model_type: gemma4
hf_device_map: None
first_param_dtype: torch.float16
first_param_device: cuda:0
total_params_B: 5.104
cuda_params_B: 5.104
[22:26:05 04/06/2026] WARMUP batch=1 max_new=24


W0604 22:26:50.399000 15285 torch/_inductor/utils.py:1731] [0/0] Not enough SMs to use max_autotune_gemm mode


[22:28:21 04/06/2026] LATENCY batch=1 max_new=256
[22:29:39 04/06/2026] OK batch=1: 3.375 tok/s
[22:29:39 04/06/2026] THROUGHPUT sweep
[22:29:39 04/06/2026] TEST batch=256 max_new=256
[22:32:09 04/06/2026] OK batch=256: 441.216 tok/s | per_prompt=1.724


,model_id,batch,input_len,max_new,total_tok,sec,tok_s_total,tok_s_per_prompt,free_GB,total_GB,alloc_GB,reserved_GB,peak_GB
1,google/gemma-4-E2B-it,256,59,256,65536,148.5348,441.216,1.724,2.183,14.563,10.889,12.215,12.706
0,google/gemma-4-E2B-it,1,59,256,256,75.8513,3.375,3.375,4.847,14.563,9.513,9.557,9.542


[22:32:09 04/06/2026] CHOT E2B: batch=256, max_new=256, total=441.216 tok/s
[22:32:09 04/06/2026] VRAM cuoi: {"free_GB": 2.183, "total_GB": 14.563, "alloc_GB": 10.888, "reserved_GB": 12.215, "peak_GB": 12.706}


In [ ]:
# CELL FIX SPEED - KHONG RELOAD, TEST AUTO-COMPILE VS DISABLE_COMPILE

import gc, time, json, datetime, traceback
import torch
import pandas as pd

assert "model" in globals()
assert "processor" in globals()

tok = getattr(processor, "tokenizer", processor)

def log(x):
    print(f"[{datetime.datetime.now().strftime('%H:%M:%S %d/%m/%Y')}] {x}", flush=True)

def sync():
    torch.cuda.synchronize()

def vram():
    free, total = torch.cuda.mem_get_info()
    return {
        "free_GB": round(free / 1024**3, 3),
        "alloc_GB": round(torch.cuda.memory_allocated() / 1024**3, 3),
        "reserved_GB": round(torch.cuda.memory_reserved() / 1024**3, 3),
        "peak_GB": round(torch.cuda.max_memory_allocated() / 1024**3, 3),
    }

if getattr(tok, "pad_token_id", None) is None:
    tok.pad_token = tok.eos_token
try:
    tok.padding_side = "left"
except Exception:
    pass

model.eval()
model.config.use_cache = True

print("generation_config.compile_config:", getattr(model.generation_config, "compile_config", None))
print("generation_config.disable_compile:", getattr(model.generation_config, "disable_compile", None))
print("VRAM dau:", json.dumps(vram()))

SYSTEM_PROMPT = "Code Python, no markdown."
USER_PROMPT = "Fuzz torch.searchsorted: seed=42, sorted/unsorted/dup/nan/inf/empty/int/float/2d, print output/exception/stats."

def make_text(prompt):
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    try:
        return processor.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        return processor.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=True,
        )
    except Exception:
        return prompt

text = make_text(USER_PROMPT)
enc = processor(text=text, return_tensors="pt")
enc = {k: v.to("cuda") for k, v in enc.items() if torch.is_tensor(v)}
enc.pop("token_type_ids", None)

base_ids = enc["input_ids"]
base_mask = enc.get("attention_mask", torch.ones_like(base_ids))

def gen(ids, mask, max_new, mode):
    kw = dict(
        input_ids=ids,
        attention_mask=mask,
        max_new_tokens=max_new,
        do_sample=False,
        use_cache=True,
        pad_token_id=tok.pad_token_id,
        eos_token_id=None,
        forced_eos_token_id=None,
    )

    if mode == "static_auto":
        kw["cache_implementation"] = "static"

    elif mode == "static_no_compile":
        kw["cache_implementation"] = "static"
        kw["disable_compile"] = True

    elif mode == "dynamic_no_compile":
        kw["disable_compile"] = True

    try:
        return model.generate(**kw)
    except TypeError:
        kw.pop("disable_compile", None)
        return model.generate(**kw)

@torch.inference_mode()
def bench_once(mode, batch, max_new):
    ids = base_ids.repeat(batch, 1).contiguous()
    mask = base_mask.repeat(batch, 1).contiguous()

    sync()
    t0 = time.perf_counter()
    out = gen(ids, mask, max_new, mode)
    sync()

    sec = time.perf_counter() - t0
    new_tok = out.shape[1] - ids.shape[1]
    total_tok = batch * new_tok
    return {
        "mode": mode,
        "batch": batch,
        "input_len": int(ids.shape[1]),
        "max_new": max_new,
        "total_tok": int(total_tok),
        "sec": round(sec, 4),
        "tok_s_total": round(total_tok / sec, 3),
        "tok_s_per_prompt": round(total_tok / sec / batch, 3),
        **vram(),
    }

rows = []

# 1) Retest dung shape da compile roi: batch=1/max_new=256
for mode in ["static_auto", "static_no_compile", "dynamic_no_compile"]:
    try:
        log(f"TEST {mode} batch=1 max_new=256")
        r = bench_once(mode, 1, 256)
        rows.append(r)
        log(f"OK {mode}: {r['tok_s_total']} tok/s")
    except Exception:
        log(f"FAIL {mode}")
        print(traceback.format_exc()[:2000])

df1 = pd.DataFrame(rows).sort_values("tok_s_total", ascending=False)
display(df1)

best_mode = df1.iloc[0]["mode"]
best_b1 = float(df1.iloc[0]["tok_s_total"])
log(f"BEST MODE batch1 = {best_mode} | {best_b1} tok/s")

if best_b1 < 15:
    log("CANH BAO: van cham bat thuong. Co the compile artifact/runtime dang hong; nen Runtime -> Restart runtime roi chay cell E2B lai.")
else:
    # 2) Throughput voi mode tot nhat
    rows2 = []
    for b in [256, 240, 224, 192, 128, 64, 32]:
        try:
            log(f"TEST THROUGHPUT {best_mode} batch={b} max_new=256")
            r = bench_once(best_mode, b, 256)
            rows2.append(r)
            log(f"OK batch={b}: {r['tok_s_total']} tok/s | per_prompt={r['tok_s_per_prompt']}")
            break
        except torch.cuda.OutOfMemoryError:
            log(f"OOM batch={b}, giam batch")
            gc.collect()
            torch.cuda.empty_cache()
        except Exception:
            log(f"FAIL batch={b}")
            print(traceback.format_exc()[:2000])
            gc.collect()
            torch.cuda.empty_cache()

    df2 = pd.DataFrame(rows2)
    display(df2)

print("VRAM cuoi:", json.dumps(vram()))

generation_config.compile_config: None
generation_config.disable_compile: None
VRAM dau: {"free_GB": 2.183, "alloc_GB": 10.888, "reserved_GB": 12.215, "peak_GB": 12.706}
[22:35:59 04/06/2026] TEST static_auto batch=1 max_new=256
[22:36:08 04/06/2026] OK static_auto: 28.974 tok/s
[22:36:08 04/06/2026] TEST static_no_compile batch=1 max_new=256
[22:36:29 04/06/2026] OK static_no_compile: 12.209 tok/s
[22:36:29 04/06/2026] TEST dynamic_no_compile batch=1 max_new=256
[22:36:49 04/06/2026] OK dynamic_no_compile: 13.013 tok/s


,mode,batch,input_len,max_new,total_tok,sec,tok_s_total,tok_s_per_prompt,free_GB,alloc_GB,reserved_GB,peak_GB
0,static_auto,1,59,256,256,8.8354,28.974,28.974,4.580,9.513,9.814,12.706
2,dynamic_no_compile,1,59,256,256,19.6732,13.013,13.013,4.553,9.521,9.842,12.706
1,static_no_compile,1,59,256,256,20.9682,12.209,12.209,4.560,9.521,9.834,12.706


[22:36:49 04/06/2026] BEST MODE batch1 = static_auto | 28.974 tok/s
[22:36:49 04/06/2026] TEST THROUGHPUT static_auto batch=256 max_new=256
[22:37:18 04/06/2026] OK batch=256: 2245.371 tok/s | per_prompt=8.771


,mode,batch,input_len,max_new,total_tok,sec,tok_s_total,tok_s_per_prompt,free_GB,alloc_GB,reserved_GB,peak_GB
0,static_auto,256,59,256,65536,29.1872,2245.371,8.771,2.41,10.889,11.979,12.706


VRAM cuoi: {"free_GB": 2.41, "alloc_GB": 10.888, "reserved_GB": 11.979, "peak_GB": 12.706}


In [ ]:
# CELL SAFE - SWEEP BATCH CAO HON, KHONG RELOAD, KHONG PIP
import gc, time, json, datetime, traceback
import torch, pandas as pd

assert "model" in globals()
assert "processor" in globals()

tok = getattr(processor, "tokenizer", processor)

def log(x):
    print(f"[{datetime.datetime.now().strftime('%H:%M:%S %d/%m/%Y')}] {x}", flush=True)

def sync():
    torch.cuda.synchronize()

def clean():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

def vram():
    free, total = torch.cuda.mem_get_info()
    return {
        "free_GB": round(free / 1024**3, 3),
        "alloc_GB": round(torch.cuda.memory_allocated() / 1024**3, 3),
        "reserved_GB": round(torch.cuda.memory_reserved() / 1024**3, 3),
        "peak_GB": round(torch.cuda.max_memory_allocated() / 1024**3, 3),
    }

if getattr(tok, "pad_token_id", None) is None:
    tok.pad_token = tok.eos_token
try:
    tok.padding_side = "left"
except Exception:
    pass

model.eval()
model.config.use_cache = True

SYSTEM_PROMPT = "Code Python, no markdown."
USER_PROMPT = "Fuzz torch.searchsorted: seed=42, sorted/unsorted/dup/nan/inf/empty/int/float/2d, print output/exception/stats."

try:
    text = processor.apply_chat_template(
        [{"role": "system", "content": SYSTEM_PROMPT},
         {"role": "user", "content": USER_PROMPT}],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
except TypeError:
    text = processor.apply_chat_template(
        [{"role": "system", "content": SYSTEM_PROMPT},
         {"role": "user", "content": USER_PROMPT}],
        tokenize=False,
        add_generation_prompt=True,
    )

enc = processor(text=text, return_tensors="pt")
enc = {k: v.to("cuda") for k, v in enc.items() if torch.is_tensor(v)}
enc.pop("token_type_ids", None)

base_ids = enc["input_ids"]
base_mask = enc.get("attention_mask", torch.ones_like(base_ids))

@torch.inference_mode()
def bench(batch, max_new=256):
    clean()
    ids = base_ids.repeat(batch, 1).contiguous()
    mask = base_mask.repeat(batch, 1).contiguous()

    sync()
    t0 = time.perf_counter()
    out = model.generate(
        input_ids=ids,
        attention_mask=mask,
        max_new_tokens=max_new,
        min_new_tokens=max_new,
        do_sample=False,
        use_cache=True,
        cache_implementation="static",
        pad_token_id=tok.pad_token_id,
        eos_token_id=None,
        forced_eos_token_id=None,
    )
    sync()

    sec = time.perf_counter() - t0
    total = batch * (out.shape[1] - ids.shape[1])
    return {
        "batch": batch,
        "input_len": int(ids.shape[1]),
        "max_new": max_new,
        "total_tok": int(total),
        "sec": round(sec, 4),
        "tok_s_total": round(total / sec, 3),
        "tok_s_per_prompt": round(total / sec / batch, 3),
        **vram(),
    }

rows = []
for b in [256, 288, 320, 352, 384, 416, 448]:
    try:
        log(f"TEST batch={b}")
        r = bench(b, 256)
        rows.append(r)
        log(f"OK {b}: {r['tok_s_total']} tok/s | per_prompt={r['tok_s_per_prompt']} | vram={json.dumps(vram())}")
    except torch.cuda.OutOfMemoryError:
        log(f"OOM batch={b}, dung sweep")
        clean()
        break
    except Exception:
        log(f"FAIL batch={b}")
        print(traceback.format_exc()[:2000])
        clean()
        break

df = pd.DataFrame(rows).sort_values("tok_s_total", ascending=False)
display(df)

best = df.iloc[0]
log(f"BEST SAFE: batch={int(best['batch'])}, total={best['tok_s_total']} tok/s")

[22:48:30 04/06/2026] TEST batch=256
[22:49:03 04/06/2026] OK 256: 2385.126 tok/s | per_prompt=9.317 | vram={"free_GB": 0.672, "alloc_GB": 10.896, "reserved_GB": 13.717, "peak_GB": 12.706}
[22:49:03 04/06/2026] TEST batch=288
[22:49:43 04/06/2026] OK 288: 1900.591 tok/s | per_prompt=6.599 | vram={"free_GB": 2.189, "alloc_GB": 11.06, "reserved_GB": 12.193, "peak_GB": 13.105}
[22:49:43 04/06/2026] TEST batch=320
[22:50:22 04/06/2026] OK 320: 2185.077 tok/s | per_prompt=6.828 | vram={"free_GB": 2.066, "alloc_GB": 11.233, "reserved_GB": 12.311, "peak_GB": 13.503}
[22:50:22 04/06/2026] TEST batch=352
[22:50:26 04/06/2026] OOM batch=352, dung sweep


,batch,input_len,max_new,total_tok,sec,tok_s_total,tok_s_per_prompt,free_GB,alloc_GB,reserved_GB,peak_GB
0,256,59,256,65536,27.4770,2385.126,9.317,0.672,10.897,13.717,12.706
2,320,59,256,81920,37.4907,2185.077,6.828,2.066,11.234,12.311,13.503
1,288,59,256,73728,38.7921,1900.591,6.599,2.189,11.061,12.193,13.105


[22:50:28 04/06/2026] BEST SAFE: batch=256, total=2385.126 tok/s


In [ ]:
# CELL LUU MODEL DANG LOAD VAO GOOGLE DRIVE
import os, time, json, datetime
from pathlib import Path

assert "model" in globals(), "Chua co model trong RAM/GPU"
assert "processor" in globals(), "Chua co processor"

from google.colab import drive
drive.mount("/content/drive")

MODEL_ID = globals().get("MODEL_ID", "google/gemma-4-E2B-it")
DRIVE_DIR = Path("/content/drive/MyDrive/hf_local_models/gemma-4-E2B-it-fp16")

def log(x):
    print(f"[{datetime.datetime.now().strftime('%H:%M:%S %d/%m/%Y')}] {x}", flush=True)

DRIVE_DIR.mkdir(parents=True, exist_ok=True)

log(f"SAVE processor -> {DRIVE_DIR}")
processor.save_pretrained(str(DRIVE_DIR))

log(f"SAVE model -> {DRIVE_DIR}")
t0 = time.perf_counter()
model.save_pretrained(
    str(DRIVE_DIR),
    safe_serialization=True,
    max_shard_size="4GB",
)
sec = time.perf_counter() - t0

manifest = {
    "model_id": MODEL_ID,
    "saved_at": datetime.datetime.now().isoformat(),
    "save_sec": round(sec, 2),
    "note": "Load next time from this local HF folder. Copy to /content first for best speed.",
}
(DRIVE_DIR / "local_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

log(f"DONE save sau {sec:.1f}s")
print("Saved files:")
for p in sorted(DRIVE_DIR.iterdir()):
    if p.is_file():
        print(p.name, round(p.stat().st_size / 1024**3, 3), "GB")

Mounted at /content/drive
[22:58:56 04/06/2026] SAVE processor -> /content/drive/MyDrive/hf_local_models/gemma-4-E2B-it-fp16
[22:58:58 04/06/2026] SAVE model -> /content/drive/MyDrive/hf_local_models/gemma-4-E2B-it-fp16


Writing model shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
# CELL LOAD NHANH TU DRIVE CACHE
import os, sys, shutil, subprocess, time, datetime, json
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = Path("/content/drive/MyDrive/hf_local_models/gemma-4-E2B-it-fp16")
LOCAL_DIR = Path("/content/gemma-4-E2B-it-fp16")

def log(x):
    print(f"[{datetime.datetime.now().strftime('%H:%M:%S %d/%m/%Y')}] {x}", flush=True)

if not LOCAL_DIR.exists():
    log(f"COPY Drive -> local SSD: {LOCAL_DIR}")
    t0 = time.perf_counter()
    shutil.copytree(DRIVE_DIR, LOCAL_DIR)
    log(f"Copy xong sau {time.perf_counter() - t0:.1f}s")
else:
    log("Local copy da ton tai, bo qua copy")

import torch
from transformers import AutoProcessor, AutoModelForCausalLM

log("LOAD processor local")
processor = AutoProcessor.from_pretrained(str(LOCAL_DIR), trust_remote_code=True)

log("LOAD model local FP16 full CUDA")
t0 = time.perf_counter()
model = AutoModelForCausalLM.from_pretrained(
    str(LOCAL_DIR),
    torch_dtype=torch.float16,
    device_map={"": 0},
    low_cpu_mem_usage=True,
    attn_implementation="sdpa",
    trust_remote_code=True,
)
model.eval()
torch.cuda.synchronize()

log(f"Load xong sau {time.perf_counter() - t0:.1f}s")
print("VRAM GB:", round(torch.cuda.memory_allocated() / 1024**3, 3))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[23:00:03 04/06/2026] COPY Drive -> local SSD: /content/gemma-4-E2B-it-fp16
[23:00:03 04/06/2026] Copy xong sau 0.4s
[23:00:32 04/06/2026] LOAD processor local
[23:00:38 04/06/2026] LOAD model local FP16 full CUDA


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


OSError: Error no file named model.safetensors, or pytorch_model.bin, found in directory /content/gemma-4-E2B-it-fp16.

In [ ]:
# CELL REPAIR/SAVE DRIVE CACHE - KHONG RELOAD MODEL
import os, shutil, time, json, datetime
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

DRIVE_DIR = Path("/content/drive/MyDrive/hf_local_models/gemma-4-E2B-it-fp16")
LOCAL_DIR = Path("/content/gemma-4-E2B-it-fp16")
TMP_DIR = Path("/content/drive/MyDrive/hf_local_models/gemma-4-E2B-it-fp16_tmp")

def log(x):
    print(f"[{datetime.datetime.now().strftime('%H:%M:%S %d/%m/%Y')}] {x}", flush=True)

def has_weights(d):
    d = Path(d)
    return (
        (d / "model.safetensors").exists()
        or (d / "pytorch_model.bin").exists()
        or ((d / "model.safetensors.index.json").exists() and len(list(d.glob("model-*.safetensors"))) > 0)
        or ((d / "pytorch_model.bin.index.json").exists() and len(list(d.glob("pytorch_model-*.bin"))) > 0)
    )

def show_dir(d):
    d = Path(d)
    print("\nDIR:", d)
    if not d.exists():
        print("  <missing>")
        return
    for p in sorted(d.iterdir()):
        if p.is_file():
            print(" ", p.name, round(p.stat().st_size / 1024**3, 3), "GB")

show_dir(DRIVE_DIR)
show_dir(LOCAL_DIR)

if not has_weights(DRIVE_DIR):
    log("Drive cache thieu weights -> save lai tu model dang trong kernel")
    assert "model" in globals(), "Kernel khong con model de save"
    assert "processor" in globals(), "Kernel khong con processor de save"

    safe_root = "/content/drive/MyDrive/hf_local_models/"
    assert str(DRIVE_DIR).startswith(safe_root)
    assert str(TMP_DIR).startswith(safe_root)

    if TMP_DIR.exists():
        shutil.rmtree(TMP_DIR)
    TMP_DIR.mkdir(parents=True, exist_ok=True)

    t0 = time.perf_counter()
    processor.save_pretrained(str(TMP_DIR))
    model.save_pretrained(
        str(TMP_DIR),
        safe_serialization=True,
        max_shard_size="4GB",
    )

    if not has_weights(TMP_DIR):
        raise RuntimeError("Save xong nhung van khong thay weight files trong TMP_DIR")

    if DRIVE_DIR.exists():
        shutil.rmtree(DRIVE_DIR)
    shutil.move(str(TMP_DIR), str(DRIVE_DIR))

    log(f"Save Drive cache xong sau {time.perf_counter() - t0:.1f}s")
else:
    log("Drive cache da co weights OK")

show_dir(DRIVE_DIR)

# Sua local partial folder
if LOCAL_DIR.exists() and not has_weights(LOCAL_DIR):
    log("Local folder bi partial/thieu weights -> xoa local")
    shutil.rmtree(LOCAL_DIR)

if not LOCAL_DIR.exists():
    log("Copy Drive -> /content local SSD")
    t0 = time.perf_counter()
    shutil.copytree(DRIVE_DIR, LOCAL_DIR)
    log(f"Copy xong sau {time.perf_counter() - t0:.1f}s")
else:
    log("Local folder da co weights OK")

show_dir(LOCAL_DIR)

if not has_weights(LOCAL_DIR):
    raise RuntimeError("LOCAL_DIR van thieu weights, chua load duoc")

log("CACHE OK. Lan sau load from_pretrained(LOCAL_DIR) se khong tai 10.2GB tu HF nua.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

DIR: /content/drive/MyDrive/hf_local_models/gemma-4-E2B-it-fp16
  chat_template.jinja 0.0 GB
  config.json 0.0 GB
  generation_config.json 0.0 GB
  processor_config.json 0.0 GB
  tokenizer.json 0.03 GB
  tokenizer_config.json 0.0 GB

DIR: /content/gemma-4-E2B-it-fp16
  chat_template.jinja 0.0 GB
  config.json 0.0 GB
  generation_config.json 0.0 GB
  processor_config.json 0.0 GB
  tokenizer.json 0.03 GB
  tokenizer_config.json 0.0 GB
[23:04:48 04/06/2026] Drive cache thieu weights -> save lai tu model dang trong kernel


AssertionError: Kernel khong con model de save

In [ ]:
 Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

DIR: /content/drive/MyDrive/hf_local_models/gemma-4-E2B-it-fp16
  chat_template.jinja 0.0 GB
  config.json 0.0 GB
  generation_config.json 0.0 GB
  processor_config.json 0.0 GB
  tokenizer.json 0.03 GB
  tokenizer_config.json 0.0 GB

DIR: /content/gemma-4-E2B-it-fp16
  chat_template.jinja 0.0 GB
  config.json 0.0 GB
  generation_config.json 0.0 GB
  processor_config.json 0.0 GB
  tokenizer.json 0.03 GB
  tokenizer_config.json 0.0 GB
[23:04:48 04/06/2026] Drive cache thieu weights -> save lai tu model dang trong kernel
---------------------------------------------------------------------------
AssertionError                            Traceback (most recent call last)
/tmp/ipykernel_25075/1183568761.py in <cell line: 0>()
     37 if not has_weights(DRIVE_DIR):
     38     log("Drive cache thieu weights -> save lai tu model dang trong kernel")
---> 39     assert "model" in globals(), "Kernel khong con model de save"
     40     assert "processor" in globals(), "Kernel khong con processor de save"
     41

AssertionError: Kernel khong con model de save

SyntaxError: leading zeros in decimal integer literals are not permitted; use an 0o prefix for octal integers (800915066.py, line 18)

In [ ]:
# CELL FIX DRIVE CACHE TU DONG: neu model mat thi tai lai E2B roi save Drive
import os, sys, subprocess, shutil, time, json, datetime, gc
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

MODEL_ID = "google/gemma-4-E2B-it"
DRIVE_DIR = Path("/content/drive/MyDrive/hf_local_models/gemma-4-E2B-it-fp16")
TMP_DIR = Path("/content/drive/MyDrive/hf_local_models/gemma-4-E2B-it-fp16_tmp")
LOCAL_DIR = Path("/content/gemma-4-E2B-it-fp16")

os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"

def log(x):
    print(f"[{datetime.datetime.now().strftime('%H:%M:%S %d/%m/%Y')}] {x}", flush=True)

def run(cmd):
    log("RUN: " + " ".join(cmd))
    p = subprocess.run(cmd, text=True)
    if p.returncode != 0:
        raise RuntimeError("Command fail: " + " ".join(cmd))

def has_weights(d):
    d = Path(d)
    return (
        (d / "model.safetensors").exists()
        or (d / "pytorch_model.bin").exists()
        or ((d / "model.safetensors.index.json").exists() and len(list(d.glob("model-*.safetensors"))) > 0)
        or ((d / "pytorch_model.bin.index.json").exists() and len(list(d.glob("pytorch_model-*.bin"))) > 0)
    )

def show_dir(d):
    d = Path(d)
    print("\nDIR:", d)
    if not d.exists():
        print("  <missing>")
        return
    for p in sorted(d.iterdir()):
        if p.is_file():
            print(" ", p.name, round(p.stat().st_size / 1024**3, 3), "GB")

if not has_weights(DRIVE_DIR):
    if "model" not in globals() or "processor" not in globals():
        log("Kernel khong con model -> tai lai E2B FP16 de save Drive cache")
        run([sys.executable, "-m", "pip", "install", "-q", "-U",
             "transformers>=5.10.0", "accelerate", "safetensors",
             "sentencepiece", "protobuf", "hf_xet"])

        import torch
        from transformers import AutoProcessor, AutoModelForCausalLM

        HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN") or None
        try:
            from google.colab import userdata
            HF_TOKEN = userdata.get("HF_TOKEN") or HF_TOKEN
        except Exception:
            pass

        gc.collect()
        torch.cuda.empty_cache()

        log("LOAD processor tu HF/cache")
        processor = AutoProcessor.from_pretrained(
            MODEL_ID,
            token=HF_TOKEN,
            trust_remote_code=True,
        )

        log("LOAD model E2B FP16 full CUDA tu HF/cache")
        t0 = time.perf_counter()
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            token=HF_TOKEN,
            trust_remote_code=True,
            dtype=torch.float16,
            device_map={"": 0},
            low_cpu_mem_usage=True,
            attn_implementation="sdpa",
        )
        model.eval()
        torch.cuda.synchronize()
        log(f"Load xong sau {time.perf_counter() - t0:.1f}s")
    else:
        log("Tim thay model/processor trong kernel -> save luon")

    log("SAVE model + processor vao Drive tmp")
    if TMP_DIR.exists():
        shutil.rmtree(TMP_DIR)
    TMP_DIR.mkdir(parents=True, exist_ok=True)

    t0 = time.perf_counter()
    processor.save_pretrained(str(TMP_DIR))
    model.save_pretrained(
        str(TMP_DIR),
        safe_serialization=True,
        max_shard_size="4GB",
    )

    if not has_weights(TMP_DIR):
        show_dir(TMP_DIR)
        raise RuntimeError("Save xong nhung TMP van khong co weights")

    if DRIVE_DIR.exists():
        shutil.rmtree(DRIVE_DIR)
    shutil.move(str(TMP_DIR), str(DRIVE_DIR))
    log(f"SAVE Drive xong sau {time.perf_counter() - t0:.1f}s")
else:
    log("Drive cache da co weights")

show_dir(DRIVE_DIR)

if LOCAL_DIR.exists() and not has_weights(LOCAL_DIR):
    log("Xoa local partial folder")
    shutil.rmtree(LOCAL_DIR)

if not LOCAL_DIR.exists():
    log("Copy Drive -> /content")
    t0 = time.perf_counter()
    shutil.copytree(DRIVE_DIR, LOCAL_DIR)
    log(f"Copy xong sau {time.perf_counter() - t0:.1f}s")

show_dir(LOCAL_DIR)
log("XONG: cache Drive/local da dung. Lan sau load local se khong tai lai 10.2GB tu HF.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[23:07:01 04/06/2026] Kernel khong con model -> tai lai E2B FP16 de save Drive cache
[23:07:01 04/06/2026] RUN: /usr/bin/python3 -m pip install -q -U transformers>=5.10.0 accelerate safetensors sentencepiece protobuf hf_xet
[23:07:09 04/06/2026] LOAD processor tu HF/cache


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


[23:07:16 04/06/2026] LOAD model E2B FP16 full CUDA tu HF/cache


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

[23:08:02 04/06/2026] Load xong sau 45.7s
[23:08:02 04/06/2026] SAVE model + processor vao Drive tmp


Writing model shards:   0%|          | 0/3 [00:00<?, ?it/s]

[23:10:33 04/06/2026] SAVE Drive xong sau 150.7s

DIR: /content/drive/MyDrive/hf_local_models/gemma-4-E2B-it-fp16
  chat_template.jinja 0.0 GB
  config.json 0.0 GB
  generation_config.json 0.0 GB
  model-00001-of-00003.safetensors 4.375 GB
  model-00002-of-00003.safetensors 3.704 GB
  model-00003-of-00003.safetensors 1.429 GB
  model.safetensors.index.json 0.0 GB
  processor_config.json 0.0 GB
  tokenizer.json 0.03 GB
  tokenizer_config.json 0.0 GB
[23:10:33 04/06/2026] Xoa local partial folder
[23:10:33 04/06/2026] Copy Drive -> /content
[23:12:12 04/06/2026] Copy xong sau 99.0s

DIR: /content/gemma-4-E2B-it-fp16
  chat_template.jinja 0.0 GB
  config.json 0.0 GB
  generation_config.json 0.0 GB
  model-00001-of-00003.safetensors 4.375 GB
  model-00002-of-00003.safetensors 3.704 GB
  model-00003-of-00003.safetensors 1.429 GB
  model.safetensors.index.json 0.0 GB
  processor_config.json 0.0 GB
  tokenizer.json 0.03 GB
  tokenizer_config.json 0.0 GB
[23:12:12 04/06/2026] XONG: cache Drive

In [ ]:
# ===================== TEST LOAD TU DRIVE CACHE + BENCH E2B T4 =====================
import os, sys, subprocess, shutil, gc, time, json, datetime
from pathlib import Path
from google.colab import drive

DRIVE_DIR = Path("/content/drive/MyDrive/hf_local_models/gemma-4-E2B-it-fp16")
LOCAL_DIR = Path("/content/gemma-4-E2B-it-fp16")

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"

def log(x):
    print(f"[{datetime.datetime.now().strftime('%H:%M:%S %d/%m/%Y')}] {x}", flush=True)

def run(cmd, allow_fail=False):
    log("RUN: " + " ".join(cmd))
    p = subprocess.run(cmd, text=True)
    if p.returncode != 0 and not allow_fail:
        raise RuntimeError("Command fail: " + " ".join(cmd))
    return p.returncode

def has_weights(d):
    d = Path(d)
    return (
        (d / "model.safetensors").exists()
        or (d / "pytorch_model.bin").exists()
        or ((d / "model.safetensors.index.json").exists() and len(list(d.glob("model-*.safetensors"))) > 0)
        or ((d / "pytorch_model.bin.index.json").exists() and len(list(d.glob("pytorch_model-*.bin"))) > 0)
    )

def show_dir(d):
    d = Path(d)
    print("\nDIR:", d)
    if not d.exists():
        print("  <missing>")
        return
    for p in sorted(d.iterdir()):
        if p.is_file():
            print(" ", p.name, round(p.stat().st_size / 1024**3, 3), "GB")

t_all = time.perf_counter()

log("INSTALL/IMPORT")
run([sys.executable, "-m", "pip", "install", "-q", "-U",
     "transformers>=5.10.0", "accelerate", "safetensors",
     "sentencepiece", "protobuf", "pandas"])

import torch
import pandas as pd
from transformers import AutoProcessor, AutoModelForCausalLM

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.benchmark = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

def sync():
    torch.cuda.synchronize()

def clean():
    gc.collect()
    torch.cuda.empty_cache()
    try:
        torch.cuda.ipc_collect()
    except Exception:
        pass
    torch.cuda.reset_peak_memory_stats()

def vram():
    free, total = torch.cuda.mem_get_info()
    return {
        "free_GB": round(free / 1024**3, 3),
        "total_GB": round(total / 1024**3, 3),
        "alloc_GB": round(torch.cuda.memory_allocated() / 1024**3, 3),
        "reserved_GB": round(torch.cuda.memory_reserved() / 1024**3, 3),
        "peak_GB": round(torch.cuda.max_memory_allocated() / 1024**3, 3),
    }

log("MOUNT DRIVE")
drive.mount("/content/drive")

if not has_weights(DRIVE_DIR):
    show_dir(DRIVE_DIR)
    raise RuntimeError("Drive cache thieu weights. Can chay cell save Drive cache truoc.")

show_dir(DRIVE_DIR)

if LOCAL_DIR.exists() and not has_weights(LOCAL_DIR):
    log("Local folder partial -> xoa")
    shutil.rmtree(LOCAL_DIR)

if not LOCAL_DIR.exists():
    log("COPY Drive -> /content local SSD")
    t0 = time.perf_counter()
    shutil.copytree(DRIVE_DIR, LOCAL_DIR)
    log(f"Copy xong sau {time.perf_counter() - t0:.1f}s")
else:
    log("Local cache da ton tai va co weights -> bo qua copy")

show_dir(LOCAL_DIR)

run(["nvidia-smi"], allow_fail=True)
clean()
log("VRAM truoc load: " + json.dumps(vram()))

log("LOAD processor local")
t0 = time.perf_counter()
processor = AutoProcessor.from_pretrained(str(LOCAL_DIR), trust_remote_code=True)
log(f"Processor load: {time.perf_counter() - t0:.1f}s")

log("LOAD model local FP16 full CUDA")
t0 = time.perf_counter()
model = AutoModelForCausalLM.from_pretrained(
    str(LOCAL_DIR),
    dtype=torch.float16,
    device_map={"": 0},
    low_cpu_mem_usage=True,
    attn_implementation="sdpa",
    trust_remote_code=True,
)
model.eval()
model.config.use_cache = True
sync()
log(f"Model load: {time.perf_counter() - t0:.1f}s")
log("VRAM sau load: " + json.dumps(vram()))

print("config.model_type:", getattr(model.config, "model_type", None))
print("first_param_dtype:", next(model.parameters()).dtype)
print("first_param_device:", next(model.parameters()).device)

tok = getattr(processor, "tokenizer", processor)
if getattr(tok, "pad_token_id", None) is None:
    tok.pad_token = tok.eos_token
try:
    tok.padding_side = "left"
except Exception:
    pass

SYSTEM_PROMPT = "Code Python, no markdown."
USER_PROMPT = "Fuzz torch.searchsorted: seed=42, sorted/unsorted/dup/nan/inf/empty/int/float/2d, print output/exception/stats."

try:
    text = processor.apply_chat_template(
        [{"role": "system", "content": SYSTEM_PROMPT},
         {"role": "user", "content": USER_PROMPT}],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
except TypeError:
    text = processor.apply_chat_template(
        [{"role": "system", "content": SYSTEM_PROMPT},
         {"role": "user", "content": USER_PROMPT}],
        tokenize=False,
        add_generation_prompt=True,
    )

enc = processor(text=text, return_tensors="pt")
enc = {k: v.to("cuda") for k, v in enc.items() if torch.is_tensor(v)}
enc.pop("token_type_ids", None)

base_ids = enc["input_ids"]
base_mask = enc.get("attention_mask", torch.ones_like(base_ids))

@torch.inference_mode()
def bench(batch, max_new=256):
    clean()
    ids = base_ids.repeat(batch, 1).contiguous()
    mask = base_mask.repeat(batch, 1).contiguous()

    sync()
    t0 = time.perf_counter()
    out = model.generate(
        input_ids=ids,
        attention_mask=mask,
        max_new_tokens=max_new,
        min_new_tokens=max_new,
        do_sample=False,
        use_cache=True,
        cache_implementation="static",
        pad_token_id=tok.pad_token_id,
        eos_token_id=None,
        forced_eos_token_id=None,
    )
    sync()

    sec = time.perf_counter() - t0
    total = batch * (out.shape[1] - ids.shape[1])
    return {
        "batch": batch,
        "input_len": int(ids.shape[1]),
        "max_new": max_new,
        "total_tok": int(total),
        "sec": round(sec, 4),
        "tok_s_total": round(total / sec, 3),
        "tok_s_per_prompt": round(total / sec / batch, 3),
        **vram(),
    }

rows = []

log("WARMUP batch=1 max_new=24")
_ = bench(1, 24)

for b in [1, 256]:
    log(f"TEST batch={b} max_new=256")
    r = bench(b, 256)
    rows.append(r)
    log(f"OK batch={b}: {r['tok_s_total']} tok/s | per_prompt={r['tok_s_per_prompt']}")

df = pd.DataFrame(rows).sort_values("tok_s_total", ascending=False)
display(df)

log(f"TOTAL CELL TIME: {time.perf_counter() - t_all:.1f}s")
log("VRAM cuoi: " + json.dumps(vram()))

[23:17:01 04/06/2026] INSTALL/IMPORT
[23:17:01 04/06/2026] RUN: /usr/bin/python3 -m pip install -q -U transformers>=5.10.0 accelerate safetensors sentencepiece protobuf pandas
[23:18:23 04/06/2026] MOUNT DRIVE
Mounted at /content/drive

DIR: /content/drive/MyDrive/hf_local_models/gemma-4-E2B-it-fp16
  chat_template.jinja 0.0 GB
  config.json 0.0 GB
  generation_config.json 0.0 GB
  model-00001-of-00003.safetensors 4.375 GB
  model-00002-of-00003.safetensors 3.704 GB
  model-00003-of-00003.safetensors 1.429 GB
  model.safetensors.index.json 0.0 GB
  processor_config.json 0.0 GB
  tokenizer.json 0.03 GB
  tokenizer_config.json 0.0 GB
[23:19:01 04/06/2026] COPY Drive -> /content local SSD
[23:23:52 04/06/2026] Copy xong sau 291.5s

DIR: /content/gemma-4-E2B-it-fp16
  chat_template.jinja 0.0 GB
  config.json 0.0 GB
  generation_config.json 0.0 GB
  model-00001-of-00003.safetensors 4.375 GB
  model-00002-of-00003.safetensors 3.704 GB
  model-00003-of-00003.safetensors 1.429 GB
  model.safet

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

[23:24:38 04/06/2026] Model load: 42.2s
[23:24:38 04/06/2026] VRAM sau load: {"free_GB": 4.861, "total_GB": 14.563, "alloc_GB": 9.508, "reserved_GB": 9.584, "peak_GB": 9.508}
config.model_type: gemma4
first_param_dtype: torch.float16
first_param_device: cuda:0
[23:24:38 04/06/2026] WARMUP batch=1 max_new=24


W0604 23:25:28.427000 3666 torch/_inductor/utils.py:1731] [0/0] Not enough SMs to use max_autotune_gemm mode


[23:27:01 04/06/2026] TEST batch=1 max_new=256
[23:28:15 04/06/2026] OK batch=1: 3.534 tok/s | per_prompt=3.534
[23:28:15 04/06/2026] TEST batch=256 max_new=256
[23:30:45 04/06/2026] OK batch=256: 441.864 tok/s | per_prompt=1.726


,batch,input_len,max_new,total_tok,sec,tok_s_total,tok_s_per_prompt,free_GB,total_GB,alloc_GB,reserved_GB,peak_GB
1,256,59,256,65536,148.3172,441.864,1.726,2.437,14.563,10.889,11.961,12.706
0,1,59,256,256,72.4425,3.534,3.534,4.847,14.563,9.513,9.557,9.542


[23:30:45 04/06/2026] TOTAL CELL TIME: 823.7s
[23:30:45 04/06/2026] VRAM cuoi: {"free_GB": 2.437, "total_GB": 14.563, "alloc_GB": 10.888, "reserved_GB": 11.961, "peak_GB": 12.706}
